In [1]:
def ccr_discado_mr_mes(spark):
    from pyspark.sql import functions as F
    from pyspark.sql import Window
    def csv_spark(spark,nombre_archivo,delimitador,header01):
        filePath = os.path.join(ruta_base, nombre_archivo)
        return spark.read.csv(filePath, sep=delimitador, header=header01,multiLine=True)

    file_list=[file for file in os.listdir(ruta_base)
        if "~$" not in file and "copia" not in file.lower() and "a365_outbound_genesys_cloud_" in file.lower()]

    if file_list:
        for filename in file_list:
            df_ccr=csv_spark(spark,filename,',',True) 
            print(filename)

            from pyspark.sql.functions import col, lit
            from functools import reduce

            if 'abandon_time' not in df_ccr.columns:
                df_ccr = df_ccr.withColumn("abandon_time", lit(None))

            campos_existentes = [campo for campo in [
                "dialing_duration", "alert_time", "acw_time",
                "talk_time", "not_responding_time", 'abandon_time'
            ] if campo in df_ccr.columns]

            df_ccr = reduce(
                lambda df, c: df.withColumn(
                    c,
                    regexp_replace(col(c).cast("string"), r"\.0$", "").cast("int")
                ),
                campos_existentes,
                df_ccr
            )

            df_ccr = df_ccr.withColumn("resource_name",when(col('user_name').isNotNull(),lower(split(col("user_name"), "@")[0])).otherwise(None))

            df_ccr = df_ccr.withColumn(
                'call_result',
                when(lower(col('tipificacion')) == 'inin-wrap-up-timeout', 'ININ-OUTBOUND-ANSWER')
                .when(lower(col('tipificacion')).like('inin-%'), col('tipificacion'))
                .when(col('tipificacion').isNull(),
                    when(col('abandon_time')>=0 ,'ININ-OUTBOUND-ABANDONED')
                    .when((col('talk_time')>=15) &
                            (col('resource_name').isNotNull()),'ININ-OUTBOUND-ANSWER')
                    .when((col('campaign_name')=='CMP_CL_OUT_ECO_A365_PORTA_IVR') & (col('resource_name').isNotNull()),'ININ-OUTBOUND-ANSWER')
                    .when((col('campaign_name')=='CMP_CL_OUT_ECO_A365_PORTA_IVR') & (col('resource_name').isNull()),'ININ-OUTBOUND-LIVE-VOICE')
                    .otherwise('ININ-OUTBOUND-MACHINE')
                    )
                .otherwise('ININ-OUTBOUND-ANSWER')
            )

            df_ccr = df_ccr.withColumn(
                'tipificacion',
                when(lower(col('tipificacion')) == 'inin-wrap-up-timeout', 'NO TIPIFICO A TIEMPO')
                .when(lower(col('tipificacion')).like('inin-%'), 'NO APLICA')
                .when(col('tipificacion').isNull(),
                    when(col('abandon_time')>=0 ,'ABANDONO')
                    .when((col('talk_time')>=15) &
                            (col('resource_name').isNotNull()),'NO TIPIFICO A TIEMPO')
                    .when((col('campaign_name')=='CMP_CL_OUT_ECO_A365_PORTA_IVR') & (col('resource_name').isNotNull()),'NO TIPIFICO A TIEMPO')
                    .when((col('campaign_name')=='CMP_CL_OUT_ECO_A365_PORTA_IVR') & (col('resource_name').isNull()),'ESCUCHO LOCUCION')
                    .otherwise('NO APLICA')
                )
                .otherwise(col('tipificacion'))
            )

            df_ccr = df_ccr.withColumn("inicio", date_format(col("inicio"), "HH:mm:ss"))

            df_ccr=df_ccr.select('outbound_contact_id','campaign_name','tipificacion','tipificacion_dialer','call_result','dia','inicio','resource_name','talk_time','abandon_time')
                
            df_ccr=df_ccr.withColumnRenamed('outbound_contact_id','chain_id')
            df_ccr=df_ccr.withColumnRenamed('talk_time','talk_duration')
            df_ccr=df_ccr.withColumnRenamed('dia','fecha')
            df_ccr=df_ccr.withColumnRenamed('tipificacion','tipificacion_1')
            valores_reemplazo = {
                'talk_duration': '0',
            }

            df_ccr = df_ccr.fillna(valores_reemplazo)
            df_ccr = df_ccr.withColumn("talk_duration", col("talk_duration").cast("int"))
            df_ccr =add_call_result_dia(df_ccr)
            df_ccr =add_peso_dia(spark,df_ccr)
            df_ccr =asignacion_estado(df_ccr)

            window_spec_mr = Window.partitionBy('campaign_name','chain_id').orderBy(F.col("peso").desc(), F.col("inicio").desc())

            df_ccr = df_ccr.withColumns({
                "unico_mr": F.row_number().over(window_spec_mr),
            })

            condiciones = {
                'q_mr': (F.col('unico_mr') == 1),
            }

            for nombre_col, condicion in condiciones.items():
                df_ccr = df_ccr.withColumn(nombre_col, F.when(condicion, 1).otherwise(0))

            df_ccr=df_ccr.filter(col('resource_name').isNotNull())
            
            from pyspark.sql import functions as F

            agg_expr1 = [col('resource_name').alias('usuario_siop')]

            agg_expr2 = [
                # F.sum("q_mr").alias("q_mr"),
                # F.count("fecha").alias("q_d"),
                F.count(F.when(F.col("estado1") == "CONTACTO", F.col("fecha"))).alias("q_d_contacto_ccr"),
                F.count(F.when(F.col("estado1") != "CONTACTO", F.col("fecha"))).alias("q_d_no_contacto_ccr"),
                F.sum(F.when(F.col("estado1") == "CONTACTO", F.col("q_mr")).otherwise(0)).alias("q_mr_contacto_ccr"),
                F.sum(F.when(F.col("estado1") != "CONTACTO", F.col("q_mr")).otherwise(0)).alias("q_mr_no_contacto_ccr"),
            ]

            return df_ccr.groupBy(agg_expr1).agg(*agg_expr2)
            # overwrite_table_SQL(df_ccr_resumen,'DB_temporal','prueba_p_01_01','01')
    

In [ ]:
# def cargar_web_ventas(spark):

#     file_list=[file for file in os.listdir(ruta_base)
#         if "~$" not in file and "copia" not in file.lower() and "reporte_webform_entel_tienda" in file.lower() and ".xlsx" in file.lower()]

#     if file_list:
#         for filename in file_list:
#             file_xlsx = os.path.join(ruta_base, filename)
#             df01 = pd.read_excel(file_xlsx,header=6)
#             df01['CAMPANIA USUARIO REG. LINEA'] = df01['CAMPANIA USUARIO REG. LINEA'].apply(lambda x: re.sub(r'\s+', ' ', quitar_tildes(x).strip()))
            
#             df01 = df01.applymap(replace_newlines)
#             temporal = 'web_ventas.csv'
#             df01.to_csv(f'{ruta_base}\\{temporal}', index=False, sep=';')
#             df=csv_spark(spark,temporal,';',True)
#             df = df.toDF(*[
#                 c.lower()
#                 .replace(" ", "_")
#                 .replace(".", "_")
#                 .replace("__", "_")
#                 for c in df.columns
#             ])
#             df = df.withColumn("d_identidad", regexp_replace(col("num_doc_usuario_reg_linea"), r'^0+', ''))
            
#             agg_expr1 = ['d_identidad']

#             agg_expr2 = [
#                 F.count(F.col("tipo_operacion")).alias("q_ventas_wdv"), 
#                 F.count(F.when(F.col("tipo_operacion") == "HOGAR FIBRA", F.col("tipo_operacion"))).alias("q_fibra_hogar"),
#                 F.count(F.when(F.col("tipo_operacion") == "2DA LINEA", F.col("tipo_operacion"))).alias("q_2da_linea"),
#                 F.count(F.when(F.col("tipo_operacion") == "RENOVACION DE EQUIPO", F.col("tipo_operacion"))).alias("q_renovacion_equipo"),
#                 F.count(F.when(F.col("tipo_operacion") == "NUEVA LINEA", F.col("tipo_operacion"))).alias("q_nueva_linea"),
#                 F.count(F.when(F.col("tipo_operacion") == "PORTA PP-SS", F.col("tipo_operacion"))).alias("q_porta_pp_ss"),
#                 F.count(F.when(F.col("tipo_operacion") == "PORTA SS-SS", F.col("tipo_operacion"))).alias("q_porta_ss_ss"),
#                 F.count(F.when(F.col("tipo_operacion") == "MIGRACION", F.col("tipo_operacion"))).alias("q_migracion"),
#             ]
#             # mover_archivo(ruta_base,temporal)
            
#             return df.groupBy(agg_expr1).agg(*agg_expr2)

def ccr_discado_mr_dia(spark):
    from pyspark.sql import functions as F
    from pyspark.sql import Window
    def csv_spark(spark,nombre_archivo,delimitador,header01):
        filePath = os.path.join(ruta_base, nombre_archivo)
        return spark.read.csv(filePath, sep=delimitador, header=header01,multiLine=True)

    file_list=[file for file in os.listdir(ruta_base)
        if "~$" not in file and "copia" not in file.lower() and "a365_outbound_genesys_cloud_" in file.lower()]

    if file_list:
        for filename in file_list:
            df_ccr=csv_spark(spark,filename,',',True) 
            print(filename)

            from pyspark.sql.functions import col, lit
            from functools import reduce

            if 'abandon_time' not in df_ccr.columns:
                df_ccr = df_ccr.withColumn("abandon_time", lit(None))

            campos_existentes = [campo for campo in [
                "dialing_duration", "alert_time", "acw_time",
                "talk_time", "not_responding_time", 'abandon_time'
            ] if campo in df_ccr.columns]

            df_ccr = reduce(
                lambda df, c: df.withColumn(
                    c,
                    regexp_replace(col(c).cast("string"), r"\.0$", "").cast("int")
                ),
                campos_existentes,
                df_ccr
            )

            df_ccr = df_ccr.withColumn("resource_name",when(col('user_name').isNotNull(),lower(split(col("user_name"), "@")[0])).otherwise(None))

            df_ccr = df_ccr.withColumn(
                'call_result',
                when(lower(col('tipificacion')) == 'inin-wrap-up-timeout', 'ININ-OUTBOUND-ANSWER')
                .when(lower(col('tipificacion')).like('inin-%'), col('tipificacion'))
                .when(col('tipificacion').isNull(),
                    when(col('abandon_time')>=0 ,'ININ-OUTBOUND-ABANDONED')
                    .when((col('talk_time')>=15) &
                            (col('resource_name').isNotNull()),'ININ-OUTBOUND-ANSWER')
                    .when((col('campaign_name')=='CMP_CL_OUT_ECO_A365_PORTA_IVR') & (col('resource_name').isNotNull()),'ININ-OUTBOUND-ANSWER')
                    .when((col('campaign_name')=='CMP_CL_OUT_ECO_A365_PORTA_IVR') & (col('resource_name').isNull()),'ININ-OUTBOUND-LIVE-VOICE')
                    .otherwise('ININ-OUTBOUND-MACHINE')
                    )
                .otherwise('ININ-OUTBOUND-ANSWER')
            )




            df_ccr = df_ccr.withColumn(
                'tipificacion',
                when(lower(col('tipificacion')) == 'inin-wrap-up-timeout', 'NO TIPIFICO A TIEMPO')
                .when(lower(col('tipificacion')).like('inin-%'), 'NO APLICA')
                .when(col('tipificacion').isNull(),
                    when(col('abandon_time')>=0 ,'ABANDONO')
                    .when((col('talk_time')>=15) &
                            (col('resource_name').isNotNull()),'NO TIPIFICO A TIEMPO')
                    .when((col('campaign_name')=='CMP_CL_OUT_ECO_A365_PORTA_IVR') & (col('resource_name').isNotNull()),'NO TIPIFICO A TIEMPO')
                    .when((col('campaign_name')=='CMP_CL_OUT_ECO_A365_PORTA_IVR') & (col('resource_name').isNull()),'ESCUCHO LOCUCION')
                    .otherwise('NO APLICA')
                )
                .otherwise(col('tipificacion'))
            )


            df_ccr = df_ccr.withColumn("inicio", date_format(col("inicio"), "HH:mm:ss"))

            df_ccr=df_ccr.select('outbound_contact_id','campaign_name','tipificacion','tipificacion_dialer','call_result','dia','inicio','resource_name','talk_time','abandon_time')
                
            df_ccr=df_ccr.withColumnRenamed('outbound_contact_id','chain_id')
            df_ccr=df_ccr.withColumnRenamed('talk_time','talk_duration')
            df_ccr=df_ccr.withColumnRenamed('dia','fecha')
            df_ccr=df_ccr.withColumnRenamed('tipificacion','tipificacion_1')
            valores_reemplazo = {
                'talk_duration': '0',
            }

            df_ccr = df_ccr.fillna(valores_reemplazo)
            df_ccr = df_ccr.withColumn("talk_duration", col("talk_duration").cast("int"))
            df_ccr =add_call_result_dia(df_ccr)
            df_ccr =add_peso_dia(spark,df_ccr)
            df_ccr =asignacion_estado(df_ccr)

            window_spec_mr = Window.partitionBy('campaign_name','chain_id').orderBy(F.col("peso").desc(), F.col("inicio").desc())

            df_ccr = df_ccr.withColumns({
                "unico_mr": F.row_number().over(window_spec_mr),
            })

            condiciones = {
                'q_mr': (F.col('unico_mr') == 1),
            }

            for nombre_col, condicion in condiciones.items():
                df_ccr = df_ccr.withColumn(nombre_col, F.when(condicion, 1).otherwise(0))

            df_ccr=df_ccr.filter(col('resource_name').isNotNull())
            
            from pyspark.sql import functions as F

            agg_expr1 = [col('resource_name').alias('usuario_siop')]

            agg_expr2 = [
                # F.sum("q_mr").alias("q_mr"),
                # F.count("fecha").alias("q_d"),
                F.count(F.when(F.col("estado1") == "CONTACTO", F.col("fecha"))).alias("q_d_contacto_ccr"),
                F.count(F.when(F.col("estado1") != "CONTACTO", F.col("fecha"))).alias("q_d_no_contacto_ccr"),
                F.sum(F.when(F.col("estado1") == "CONTACTO", F.col("q_mr")).otherwise(0)).alias("q_mr_contacto_ccr"),
                F.sum(F.when(F.col("estado1") != "CONTACTO", F.col("q_mr")).otherwise(0)).alias("q_mr_no_contacto_ccr"),
            ]

            return df_ccr.groupBy(agg_expr1).agg(*agg_expr2)
            # overwrite_table_SQL(df_ccr_resumen,'DB_temporal','prueba_p_01_01','01')


In [ ]:
def carga_campaing_history():
    from functools import reduce
    from pyspark.sql import functions as F
    from pyspark.sql import Window
    spark = SparkSession.builder \
        .appName("SparkExample") \
        .master("local[*]") \
        .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-8.4.1.jre11.jar') \
        .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-8.4.1.jre11.jar') \
        .config('spark.executor.memory', '10g') \
        .config('spark.driver.memory', '10g') \
        .config('spark.sql.session.timeZone', 'UTC') \
        .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
        .getOrCreate()

    file_list=[file for file in os.listdir(ruta_base)
        if "~$" not in file and "copia" not in file.lower() and "campaign attempt detail" in file.lower()]

    for filename in file_list:
        if file_list:
            filePath = os.path.join(ruta_base, filename)
            df_ccr=spark.read.csv(filePath, sep=';', header=True,multiLine=True)
            df_ccr=df_ccr.distinct()
            df_ccr = df_ccr.toDF(*[
                c.lower()
                .replace(" ", "_")
                .replace(".", "_")
                .replace("__", "_")
                for c in df_ccr.columns
            ])

            df_ccr=df_ccr.filter(col('media_type')=='voice')

            df_ccr=df_ccr.select('media_type','campaign_name','division_name','conversation_id','start','disconnect','dialing','time_to_agent','time_to_flow','time_to_abandon','duration','dnis','caller_id','system_disposition','agent','wrapup_name','wrapup_duration')

            df_ccr = df_ccr.withColumn("inicio", date_format(to_timestamp(col("start"), "M/d/yy hh:mm:ss a"), "HH:mm:ss"))

            df_ccr = df_ccr.withColumn(
                'call_result',
                when(col('time_to_abandon').isNotNull(), 'ININ-OUTBOUND-ABANDONED')
                .when(col('wrapup_name')=='ININ-WRAP-UP-TIMEOUT','ININ-OUTBOUND-ANSWER')
                .when((col('wrapup_name').isNull()) &(col('wrapup_name').isNull()),'ININ-OUTBOUND-GENERAL-ERROR')
                .when((col('wrapup_name').isNull()) &(col('wrapup_name').contains('SIP')),'ININ-OUTBOUND-GENERAL-ERROR')
                .when((col('wrapup_name').isNotNull()) &(col('wrapup_name').contains('SIP')),
                    when((lower(col('wrapup_name')) == 'inin-outbound-transferred-to-flow')|
                        (lower(col('wrapup_name')) == 'paso_por_outbound_flow'),
                        when(col('campaign_name')=='CMP_CL_OUT_ECO_A365_PORTA_IVR','ININ-OUTBOUND-ANSWER')
                        .otherwise('ININ-OUTBOUND-ABANDONED')
                    )      
                    .otherwise(col('wrapup_name')) 
                )
                .when((col('wrapup_name').isNull()) &(col('wrapup_name').isNotNull()),'ININ-OUTBOUND-ANSWER')
                .when((col('wrapup_name').isNotNull()) &(col('wrapup_name').isNull()),
                    when((lower(col('wrapup_name')) == 'inin-outbound-transferred-to-flow')|
                        (lower(col('wrapup_name')) == 'paso_por_outbound_flow'),
                        when(col('campaign_name')=='CMP_CL_OUT_ECO_A365_PORTA_IVR','ININ-OUTBOUND-ANSWER')
                        .otherwise('ININ-OUTBOUND-ABANDONED')
                    )      
                    .otherwise(col('wrapup_name'))
                ) 
                .otherwise('ININ-OUTBOUND-ANSWER') 
            )

            df_ccr = df_ccr.withColumn(
                'wrapup',
                when(col('time_to_abandon').isNotNull(), 'NO APLICA')
                .when(col('wrapup_name') == 'ININ-WRAP-UP-TIMEOUT', 'NO TIPIFICO A TIEMPO')
                .when((col('wrapup_name').isNull()) & (col('wrapup_name').isNull()), 'NO APLICA')
                .when((col('wrapup_name').isNull()) & (col('wrapup_name').contains('SIP')), 'NO APLICA')
                .when((col('wrapup_name').isNotNull()) & (col('wrapup_name').contains('SIP')),
                    when((lower(col('wrapup_name')) == 'inin-outbound-transferred-to-flow') |
                        (lower(col('wrapup_name')) == 'paso_por_outbound_flow'),
                        when(col('campaign_name') == 'CMP_CL_OUT_ECO_A365_PORTA_IVR', 'ESCUCHO LOCUCION')
                        .otherwise('NO APLICA')
                    )
                    .otherwise('NO APLICA')
                )
                .when((col('wrapup_name').isNull()) & (col('wrapup_name').isNotNull()), col('wrapup_name'))
                .when((col('wrapup_name').isNotNull()) & (col('wrapup_name').isNull()),
                    when((lower(col('wrapup_name')) == 'inin-outbound-transferred-to-flow') |
                        (lower(col('wrapup_name')) == 'paso_por_outbound_flow'),
                        when(col('campaign_name') == 'CMP_CL_OUT_ECO_A365_PORTA_IVR', 'ESCUCHO LOCUCION')
                        .otherwise('NO APLICA')
                    )
                    .otherwise('NO APLICA')
                )
                .otherwise(col('wrapup_name'))
            )
            Export_list_base_sql(df_ccr,'DB_CCR','tb_Detailed_Attempt_history_acumulado',filename)
    spark.stop()
carga_campaing_history()

In [3]:
ruta_fibra='C:\\Users\\A365\\Documents\\datos'


In [4]:
import sys 
sys.path.append('C:/Users/A365/Documents/script_01/resumen')
from funciones import *

In [5]:

def add_call_result_dia(df):
    
    call_result_lower = F.lower(F.col("call_result"))
    
    return df.withColumn(
        "call_result",
        F.when(call_result_lower == "answer", "ININ-OUTBOUND-ANSWER")
         .when(call_result_lower == "inin-wrap-up-timeout", "ININ-OUTBOUND-ANSWER")
         .when(call_result_lower == "no answer", "ININ-OUTBOUND-NO-ANSWER")
         .when(call_result_lower == "abandoned", "ININ-OUTBOUND-ABANDONED")
         .when(call_result_lower == "answering machine detected", "ININ-OUTBOUND-MACHINE")
         .when(call_result_lower == "busy", "ININ-OUTBOUND-BUSY")
         .when(call_result_lower == "unknown call result", "No call")
         .when(call_result_lower == "dropped", "ININ-OUTBOUND-DROPPED")
         .when(call_result_lower == "silence", "ININ-OUTBOUND-SILENCE")
         .when(call_result_lower == "fax detected", "ININ-OUTBOUND-FAX")
         .when(F.col("call_result").isNull(), 
               F.when(F.col('campaign_name').like("%IVR%"), "ININ-OUTBOUND-ANSWER")
                .otherwise("ININ-OUTBOUND-ABANDONED")
         )
         .otherwise(F.col("call_result"))
    )

def add_peso_dia(spark,df):
    query1 = f"""
    SELECT
    distinct
    upper(codigo_genesys) as tipificacion_1
    ,codigo_genesys1 as tipificacion
    ,estado
    ,obs1 AS peso
    FROM DB_interaccion.dbo.tb_tipificacion
    """
    # where obs1<>'ok'
    df_tipi = obtener_tabla_sql(spark,'DB_interaccion',query1)
    return df.join(df_tipi, ["tipificacion_1"], "left")

def asignacion_estado(df):
    
    df_ref = df.withColumn(
        'estado1',
        when(col('abandon_time').isNotNull(),'NO CONTACTO')
        .when(
            (col("talk_duration") >= 15) &
            (col('tipificacion_1').isNotNull()) & 
            (col('tipificacion_1') != 'NO APLICA') & 
            (col('tipificacion_1') != 'ESCUCHO LOCUCION') ,
            'CONTACTO'
        ).when(
            (col("talk_duration") >= 15) &
            (col('tipificacion_1') != 'ESCUCHO LOCUCION') &
            (lower(col('tipificacion_dialer')) == 'inin-outbound-transferred-to-flow'), 
            'CONTACTO'
        ).otherwise('NO CONTACTO')
    )
    
            
    df_ref = df_ref.withColumn(
        'estado',
        when(col('abandon_time')>=0,'ABANDONO')
        .when(col('estado1')=='CONTACTO',col('estado'))
        .when(col('estado').isin('CONTACTADO','NO CONTACTADO'),'NC MENOR A 15 SEG')
        .when(col('tipificacion_1').isNull(), 'NC ESCUCHO LOCUCION')
        .when(col('tipificacion_1')=='ESCUCHO LOCUCION', 'NC ESCUCHO LOCUCION')
        .otherwise('MAQUINA')  
    )   

    df_ref = df_ref.withColumn(
        'peso',
        when(col('estado') == 'ABANDONO', 990)
        .when(col('estado') == 'MAQUINA', 991)
        .when(col('estado') == 'NC ESCUCHO LOCUCION',995)
        .when(col('estado1') == 'NO CONTACTO', 999)
        .otherwise(col('peso'))
    )

    df_ref= df_ref.withColumn(
                'tipificacion',
                when(col('estado') == 'ABANDONO', 'ABANDONO')
                .when(col('estado') == 'MAQUINA', 'MAQUINA')
                .when(col('estado') == 'NC ESCUCHO LOCUCION','NO CONTACTO ESCUCHO LOCUCION')
                .otherwise(col('tipificacion'))
            ) 
                
    df_ref=df_ref.drop('tipificacion_dialer')
    return df_ref


def ccr_discado_mr_mes(spark,fecha_fin,indices):

    def obtener_tabla_resumen(spark, campaign_name, db_ccr, db_campana,nombre_skill, fecha_fin):
        from pyspark.sql.functions import col
        from pyspark.sql import functions as F

        query = f"""
            select 
            ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS indice
            ,chain_id
            ,fecha
            ,call_result
            ,inicio
            ,upper(tipificacion) as tipificacion_1
            ,tipificacion_dialer
            ,abandon_time
            ,talk_duration
            ,resource_name
            ,campaign_name
            FROM {db_ccr}
            where fecha between DATEADD(MONTH, DATEDIFF(MONTH, 0, '{fecha_fin}'), 0) and '{fecha_fin}'
            and campaign_name in ('{campaign_name}')
        """
        df_ccr = obtener_tabla_sql(spark,"DB_CCR", query)
        
        df_ccr=add_call_result_dia(df_ccr)
        
        df_ccr = add_peso_dia(spark,df_ccr)
        df_ccr =asignacion_estado(df_ccr)
        window_spec_mr = Window.partitionBy('chain_id').orderBy(F.col("peso").desc(), F.col("inicio").desc())

        df_ccr = df_ccr.withColumns({
            "unico_mr": F.row_number().over(window_spec_mr),
        })

        condiciones = {
            'q_mr': (F.col('unico_mr') == 1),
        }

        for nombre_col, condicion in condiciones.items():
            df_ccr = df_ccr.withColumn(nombre_col, F.when(condicion, 1).otherwise(0))

        df_ccr=df_ccr.filter(col('resource_name').isNotNull())
        
        agg_expr1 = [col('resource_name').alias('usuario_siop')]

        agg_expr2 = [
            F.count(F.when(F.col("estado1") == "CONTACTO", F.col("fecha"))).alias("q_d_contacto_ccr"),
            F.count(F.when(F.col("estado1") != "CONTACTO", F.col("fecha"))).alias("q_d_no_contacto_ccr"),
            F.sum(F.when(F.col("estado1") == "CONTACTO", F.col("q_mr")).otherwise(0)).alias("q_mr_contacto_ccr"),
            F.sum(F.when(F.col("estado1") != "CONTACTO", F.col("q_mr")).otherwise(0)).alias("q_mr_no_contacto_ccr"),
        ]
        return df_ccr.groupBy(agg_expr1).agg(*agg_expr2)

    from pyspark.sql import functions as F
    from pyspark.sql.functions import lit
    from functools import reduce

    spark = SparkSession.builder \
        .appName("SparkExample") \
        .master("local[*]") \
        .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-12.8.1.jre8.jar') \
        .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-12.8.1.jre8.jar') \
        .config('spark.executor.memory', '22g') \
        .config('spark.driver.memory', '22g') \
        .getOrCreate()

    dfs = [
        obtener_tabla_resumen(spark, *campaigns_out[i], fecha_fin) for i in indices
    ]

    df_ccr_base = reduce(lambda df1, df2: df1.unionByName(df2), dfs)
    
    agg_expr1 = ['usuario_siop']

    agg_expr2 = [
        # F.sum('q_d_contacto_ccr').alias("q_d_contacto_ccr"),
        # F.sum(F.when(F.col("estado1") != "CONTACTO", F.col("fecha"))).alias("q_d_no_contacto_ccr"),
        F.sum('q_mr_contacto_ccr').alias("q_mr_contacto_ccr"),
        F.sum('q_mr_no_contacto_ccr').alias("q_mr_no_contacto_ccr"),
    ]
    return df_ccr_base.groupBy(agg_expr1).agg(*agg_expr2)
            
def cloud_discado_mr_dia(spark):
    from functools import reduce
    from pyspark.sql import functions as F
    from pyspark.sql import Window
    dfs = []
    ruta_fibra='C:\\Users\\A365\\Documents\\datos'
    file_list=[file for file in os.listdir(ruta_fibra)
        if "~$" not in file and "copia" not in file.lower() and "campaign attempt detail" in file.lower()]

    for filename in file_list:
        if file_list:
            for filename in file_list:
                filePath = os.path.join(ruta_fibra, filename)
                df_tmp=spark.read.csv(filePath, sep=';', header=True,multiLine=True)
                dfs.append(df_tmp)
            
    if dfs:
        df_ccr = reduce(lambda df1, df2: df1.union(df2), dfs)        
        df_ccr = df_ccr.distinct()
    else:
        print("⚠️ No se encontraron archivos válidos en la carpeta.")
        return None
    
    # df_ccr = reduce(lambda df1, df2: df1.union(df2), dfs)        
    # df_ccr=df_ccr.distinct()
    df_ccr = df_ccr.toDF(*[
        c.lower()
        .replace(" ", "_")
        .replace(".", "_")
        .replace("__", "_")
        for c in df_ccr.columns
    ])

    df_ccr=df_ccr.filter(col('media_type')=='voice')

    df_ccr=df_ccr.select('media_type','campaign_name','division_name','conversation_id','start','disconnect','dialing','time_to_agent','time_to_flow','time_to_abandon','duration','dnis','caller_id','system_disposition','agent','wrapup_name','wrapup_duration')

    df_ccr = df_ccr.withColumn("inicio", date_format(to_timestamp(col("start"), "M/d/yy hh:mm:ss a"), "HH:mm:ss"))

    df_ccr = df_ccr.withColumn(
        'call_result',
        when(col('time_to_abandon').isNotNull(), 'ININ-OUTBOUND-ABANDONED')
        .when(col('wrapup_name')=='ININ-WRAP-UP-TIMEOUT','ININ-OUTBOUND-ANSWER')
        .when((col('wrapup_name').isNull()) &(col('wrapup_name').isNull()),'ININ-OUTBOUND-GENERAL-ERROR')
        .when((col('wrapup_name').isNull()) &(col('wrapup_name').contains('SIP')),'ININ-OUTBOUND-GENERAL-ERROR')
        .when((col('wrapup_name').isNotNull()) &(col('wrapup_name').contains('SIP')),
            when((lower(col('wrapup_name')) == 'inin-outbound-transferred-to-flow')|
                (lower(col('wrapup_name')) == 'paso_por_outbound_flow'),
                when(col('campaign_name')=='CMP_CL_OUT_ECO_A365_PORTA_IVR','ININ-OUTBOUND-ANSWER')
                .otherwise('ININ-OUTBOUND-ABANDONED')
            )      
            .otherwise(col('wrapup_name')) 
        )
        .when((col('wrapup_name').isNull()) &(col('wrapup_name').isNotNull()),'ININ-OUTBOUND-ANSWER')
        .when((col('wrapup_name').isNotNull()) &(col('wrapup_name').isNull()),
            when((lower(col('wrapup_name')) == 'inin-outbound-transferred-to-flow')|
                (lower(col('wrapup_name')) == 'paso_por_outbound_flow'),
                when(col('campaign_name')=='CMP_CL_OUT_ECO_A365_PORTA_IVR','ININ-OUTBOUND-ANSWER')
                .otherwise('ININ-OUTBOUND-ABANDONED')
            )      
            .otherwise(col('wrapup_name'))
        ) 
        .otherwise('ININ-OUTBOUND-ANSWER') 
    )

    df_ccr = df_ccr.withColumn(
        'wrapup',
        when(col('time_to_abandon').isNotNull(), 'NO APLICA')
        .when(col('wrapup_name') == 'ININ-WRAP-UP-TIMEOUT', 'NO TIPIFICO A TIEMPO')
        .when((col('wrapup_name').isNull()) & (col('wrapup_name').isNull()), 'NO APLICA')
        .when((col('wrapup_name').isNull()) & (col('wrapup_name').contains('SIP')), 'NO APLICA')
        .when((col('wrapup_name').isNotNull()) & (col('wrapup_name').contains('SIP')),
            when((lower(col('wrapup_name')) == 'inin-outbound-transferred-to-flow') |
                (lower(col('wrapup_name')) == 'paso_por_outbound_flow'),
                when(col('campaign_name') == 'CMP_CL_OUT_ECO_A365_PORTA_IVR', 'ESCUCHO LOCUCION')
                .otherwise('NO APLICA')
            )
            .otherwise('NO APLICA')
        )
        .when((col('wrapup_name').isNull()) & (col('wrapup_name').isNotNull()), col('wrapup_name'))
        .when((col('wrapup_name').isNotNull()) & (col('wrapup_name').isNull()),
            when((lower(col('wrapup_name')) == 'inin-outbound-transferred-to-flow') |
                (lower(col('wrapup_name')) == 'paso_por_outbound_flow'),
                when(col('campaign_name') == 'CMP_CL_OUT_ECO_A365_PORTA_IVR', 'ESCUCHO LOCUCION')
                .otherwise('NO APLICA')
            )
            .otherwise('NO APLICA')
        )
        .otherwise(col('wrapup_name'))
    )
    df_ccr = df_ccr.withColumn("indice", monotonically_increasing_id())

    query1 = f"""
    select codigo_genesys as Wrapup
    ,upper(codigo_genesys1) as tipificacion, motivo,estado,contacto,interes,calificacion,obs1 from DB_interaccion.dbo.tb_tipificacion
    """
    df_tipis = obtener_tabla_sql(spark,"DB_venta",query1)

    df_ccr=df_ccr.join(df_tipis,['Wrapup'],'left')

    window_spec = Window.partitionBy('indice').orderBy(col("indice").desc())
    df_ccr = df_ccr.withColumn("unico", row_number().over(window_spec))
    df_ccr = df_ccr.filter(col("unico") == 1).drop('indice')

    df_ccr = df_ccr.withColumn(
        "Dialing",
        F.when(
            F.col("Dialing").rlike("^[0-9]*\\.?[0-9]+$"),   
            F.col("Dialing").cast("double")                 
        ).otherwise(None)                                      
    )
    df_ccr = df_ccr.withColumn(
        "Duration",
        F.when(
            F.col("Duration").rlike("^[0-9]*\\.?[0-9]+$"),   
            F.col("Duration").cast("double")                 
        ).otherwise(None)                                    
    )


    df_ccr = df_ccr.withColumn(
        "talk_duration", F.round(F.col("Duration") - F.col("Dialing")).cast("bigint")
    )

    df_ccr = df_ccr.withColumn(
        'estado_01',  
        when(
            (col("talk_duration") >= 15) &
            (col('tipificacion').isNotNull()),
            'CONTACTO'
        ).otherwise('NO CONTACTO')
    )

    df_ccr = df_ccr.withColumn(
        'obs1',
        when(col('call_result')!='ININ-OUTBOUND-ANSWER', 0)
        .when(col('estado_01') == 'no contacto', col('obs1') -300)
        .otherwise(col('obs1'))
    )

    window_spec = Window.partitionBy('campaign_name','dnis').orderBy(col("obs1").desc(),col("inicio").desc())
    df_ccr = df_ccr.withColumn("unico_todo", row_number().over(window_spec))
    df_ccr=df_ccr.withColumn('unico_todo',when(col('unico_todo')==1, 1).otherwise(0))

    df_ccr=df_ccr.withColumn('agent',when(col('call_result')=='ININ-OUTBOUND-ABANDONED','ABANDONO').otherwise(col('agent')))

    df_ccr=df_ccr.withColumn('agent',when((col('call_result')!='ININ-OUTBOUND-ANSWER') & 
                                                (col('agent').isNull()) 
                                                ,'MAQUINA')
                                            .when(col('estado').isNull(),'MAQUINA')
                                            .otherwise(col('agent')))

    df_ccr=df_ccr.withColumn('estado',when(col('agent')=='MAQUINA','MAQUINA')
                            .when(col('estado').isNull(),'MAQUINA')
                            .when(col('agent')=='ABANDONO','ABANDONO')
                            .otherwise(col('estado')))


    df_ccr = (
        df_ccr.filter(~col("agent").isin("ABANDONO", "MAQUINA"))
            .withColumn("agent", lower(col("agent")))
    )
    agg_expr1 = ['agent']


    agg_expr2 = [
        # F.count(F.when(F.col("estado_01") == "CONTACTO", F.col("dnis"))).alias("q_d_contacto_cloud"),
        # F.count(F.when(F.col("estado_01") != "CONTACTO", F.col("dnis"))).alias("q_d_no_contacto_cloud"),
        F.sum(F.when(F.col("estado_01") == "CONTACTO", F.col("unico_todo")).otherwise(0)).alias("q_mr_contacto_cloud"),
        F.sum(F.when(F.col("estado_01") != "CONTACTO", F.col("unico_todo")).otherwise(0)).alias("q_mr_no_contacto_cloud"),
    ]

    return df_ccr.groupBy(agg_expr1).agg(*agg_expr2)

def ventas_fibra_wdv(spark):

    file_list=[file for file in os.listdir(ruta_base)
        if "~$" not in file 
            and "copia" not in file.lower() 
            and "webventas" in file.lower() 
            and ".xlsx" in file.lower()]

    warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

    temporal = 'carga_ventas_fibra.csv'
    if file_list:
        for filename in file_list:
            file_xlsx = os.path.join(ruta_base, filename)
            # df01 = pd.read_excel(file_xlsx,header=0)
            df01 = pd.read_excel(file_xlsx, sheet_name="Hoja1", header=0)
            df01 = df01.applymap(replace_newlines)  

            df01.columns = [c.lower().replace(" ", "_") for c in df01.columns]

    df01.to_csv(f'{ruta_base}\\{temporal}', index=False, sep=';')
    df_ref=csv_spark(spark,temporal,';',True)

    df_ref= df_ref.filter(
        (col("fechaventa") >= "2025-09-01") & 
        (col("fechaventa") <= "2025-09-30")
    )

    df_ref=df_ref.select('dni','producto').filter(col('producto')!='Movil')

    df_ref=df_ref.withColumn('tipo_operacion',when(upper(F.col("producto")) == "FIBRA",'HOGAR FIBRA')
                             .otherwise( F.col("producto")))
    df_ref=df_ref.withColumn("d_identidad", regexp_replace(col("dni"), r'^0+', ''))

    return df_ref.select('d_identidad','tipo_operacion')

def cargar_web_ventas(spark,fecha_fin):
    query1=f"""
    select 
    distinct
    [NUM.DOC USUARIO REG. LINEA] as d_identidad
    ,[TIPO OPERACION] as tipo_operacion
    from DB_venta.dbo.tb_registro_venta
    where cast([FECHA REGISTRO] as date) between DATEADD(MONTH, DATEDIFF(MONTH, 0, '{fecha_fin}'), 0) and '{fecha_fin}'
    and [CAMPANIA USUARIO REG. LINEA] in(
    'EXCLUSIVA RECUPERADOS',
    'MIGRACIONES',
    'PORTA5',
    'SEGUNDAS LINEAS 1',
    'PERFILADA 1',
    'PREPAGO IVR'
    )
    """
    df=obtener_tabla_sql(spark,'DB_tiempo',query1)

    return df.withColumn("d_identidad", regexp_replace(col("d_identidad"), r'^0+', ''))


def cloud_conexiones(spark,fecha_fin):
    from pyspark.sql import functions as F

    query1 = f"""
        select 
        [Interval Start] as fecha
        ,[Agent Name] as agent
        ,user_genesys as usuario_siop
        ,cast([Logged In]as float)/3600 AS logged_in
        ,cast([On Queue] as float)/3600 AS on_queue
        ,cast([idle]as float)/3600 AS idle
        ,cast([Off Queue]as float)/3600 AS off_queue
        ,cast([Interacting]as float)/3600 AS Interacting
        ,(cast([Logged In]as float)-cast( Idle as float)- cast([Off Queue]as float))/3600 AS t_operativo
        ,upper(Skills) as Skills
        from DB_Tiempo.dbo.tb_agent_status
        where [Interval Start] between DATEADD(MONTH, DATEDIFF(MONTH, 0, '{fecha_fin}'), 0) and '{fecha_fin}'
        and Skills in(
        'sk_CL_OUT_ECO_A365_PORTA_PERFILADA'
        ,'sk_CL_OUT_ECO_A365_PORTA_RECUPERADOS'
        ,'SK_CL_OUT_ECO_A365_PORTA_IVR'
        ,'SK_CL_OUT_ECO_A365_SEGUNDAS_LINEAS'
        ,'SK_CL_OUT_ECO_A365_MIGRACIONES'
        ,'SK_CL_OUT_ECO_A365_MOVIL_PORTA5'
        ,'SK_CL_OUT_ECO_A365_HOGAR_FIBRA_CLIENTE'
        ,'SK_CL_OUT_ECO_A365_HOGAR_FIBRA_NOCLIENTE'
        )
        """
    df_tiempo= obtener_tabla_sql(spark,'DB_tiempo',query1)

    query1=f"""
    select
    case
    when nombre_skill= 'PORTABILIDAD_PERFILADA_1' then  'Sk_CL_OUT_ECO_A365_PORTA_PERFILADA'
    when nombre_skill= 'PORTABILIDAD_RECUPERADOS' then  'sk_CL_OUT_ECO_A365_PORTA_RECUPERADOS'
    when nombre_skill= 'PORTABILIDAD_PREPAGO_IVR' then  'SK_CL_OUT_ECO_A365_PORTA_IVR'
    when nombre_skill= 'SEGUNDAS_LINEAS' then  'SK_CL_OUT_ECO_A365_SEGUNDAS_LINEAS'
    when nombre_skill= 'MIGRACIONES_1' then  'SK_CL_OUT_ECO_A365_MIGRACIONES'
    when nombre_skill= 'MOVIL_PORTA5' then  'SK_CL_OUT_ECO_A365_MOVIL_PORTA5'
    when nombre_skill= 'HOGAR_FIBRA_CLIENTE' then  'SK_CL_OUT_ECO_A365_HOGAR_FIBRA_CLIENTE'
    when nombre_skill= 'HOGAR_FIBRA_NOCLIENTE' then  'SK_CL_OUT_ECO_A365_HOGAR_FIBRA_NOCLIENTE'
    end  as Skills
    ,fecha
    ,1 as ref_fecha1
    from DB_tiempo.dbo.fecha_no_aplica
    where nombre_skill<>'MIGRACIONES_2'
    """
    df_fecha_no_aplica=obtener_tabla_sql(spark,'DB_tiempo',query1)

    df_tiempo=df_tiempo.join(df_fecha_no_aplica,['Skills','fecha'],'left')
    
    df_tiempo = df_tiempo.withColumn(
        "ref_fecha1",
        F.when(F.date_format("fecha", "E") == "Sun", F.lit(1))    # domingo
        .when(col('ref_fecha1')==1,F.lit(1))
        .otherwise(F.lit(0))                                     # demás días
    )

    col_01 = ['logged_in','on_queue','off_queue','Interacting','t_operativo']

    for c in col_01:
        df_tiempo = df_tiempo.withColumn(
            c,
            when(col("ref_fecha1") == 1, 0).otherwise(col(c))
        )

    return df_tiempo.drop('ref_fecha1')
                    
# def cloud_conexiones(spark):
#     from pyspark.sql import functions as F


#     campaing_name_01 = [
#         'SK_CL_OUT_ECO_A365_HOGAR_FIBRA_CLIENTE',
#         'SK_CL_OUT_ECO_A365_HOGAR_FIBRA_NOCLIENTE',
#         'SK_CL_OUT_ECO_A365_PORTA_IVR',
#         'SK_CL_OUT_ECO_A365_MOVIL_PORTA5',
#         'SK_CL_OUT_ECO_A365_MIGRACIONES',
#         'SK_CL_OUT_ECO_A365_SEGUNDAS_LINEAS',
#         'SK_CL_OUT_ECO_A365_SEGUNDAS_LINEAS2',
#         'SK_CL_OUT_ECO_A365_PORTA_PERFILADA',
#         'SK_CL_OUT_ECO_A365_PORTA_EXCLUSIVA',
#         ]

#     ref1_name='agent_status'

#     file_list=[file for file in os.listdir(ruta_base)
#         if "~$" not in file 
#             and "copia" not in file.lower() 
#             and f"{ref1_name.lower()}" in file.lower() 
#             and ".csv" in file.lower()]

#     if file_list:
#         for filename in file_list:
#             file_csv = os.path.join(ruta_base, filename)

#             df_ref=csv_spark(spark,file_csv,';',True)
#             df_ref=df_ref.filter(col('Skills').isin(campaing_name_01))
            
#             df_ref = df_ref.withColumn("user_genesys", lower(split(col("Email"), "@")[0]))

#             df_ref = df_ref.withColumn(
#                 "Log In",
#                 date_format(to_timestamp(col("Log In"), "MM/dd/yy hh:mm a"), "yyyy-MM-dd HH:mm:ss")
#             )

#             df_ref = df_ref.withColumn(
#                 "Log Out",
#                 date_format(to_timestamp(col("Log Out"), "MM/dd/yy hh:mm a"), "yyyy-MM-dd HH:mm:ss")
#             )
#             df_ref = df_ref.withColumn("agent", lower(col("Agent Name")))
#             df_ref = df_ref.withColumn("user_genesys", lower(col("user_genesys")))

#             df_ref=df_ref.select('agent','Logged In','On Queue','Idle','Interacting','Off Queue','Log In','Log Out','Skills',col('user_genesys').alias('usuario_siop'))
            
#             valores_reemplazo = {
#                 'On Queue': '0',
#                 'Idle': '0',
#                 'Interacting': '0',
#                 'Off Queue': '0',
#             }

#             df_ref = df_ref.fillna(valores_reemplazo)

#             df_ref = df_ref.toDF(*[
#                 c.lower()
#                 .replace(" ", "_")
#                 .replace(".", "_")
#                 .replace("__", "_")
#                 for c in df_ref.columns
#             ])
            
#             cols_to_divide = ["logged_in", "on_queue", "idle", "interacting",'off_queue']

#             for c in cols_to_divide:
#                 df_ref = df_ref.withColumn(c, col(c).cast("double") / 3600)
                
#             return df_ref
#             # return df_ref.drop('skills')

def consolidado_ult_dota_asistencia(spark,fecha_fin):

    def add_columnas_necesarias(df):
        condiciones = {
            'debe_laborar': F.when(F.col('estado_ref').isin('P','T','DS','FER','D_OJT','FI','FJ','NC','PEND','PF','DE','CR','CAP'), 1).otherwise(0),
            'laborados': F.when(F.col('estado_ref').isin('P','T','DS','FER','PF','DE','CR','CAP'), 1).otherwise(0),
            'faltas': F.when(F.col('estado_ref').isin('D_OJT','FI','FJ','NC','PEND'), 1).otherwise(0),
            'inactivo': F.when(F.col('estado_ref').isin('LP','V','DM','LM','LSG'), 1).otherwise(0),
            'total': F.when(F.col('estado_ref').isin('P','T','DS','FER','D_OJT','FI','FJ','PEND','NC','LP','V','DM','LM','PF','DE','CR','CAP','LSG'), 1).otherwise(0),
            'contratados_dia': F.when(F.col('estado_ref').isin('P','T','DS','D_OJT','FI','FJ','PEND','PEND','NC','LP','V','DM','LM','PF','DE','FER','CR','CAP','LSG'), 1).otherwise(0),  # DEBERIA IR FER 
            'activos_dia': F.when(F.col('estado_ref').isin('P','T','DS','D_OJT','FI','FJ','NC','PEND','PF','DE','CAP'), 1).otherwise(0),
            'conectados_dia': F.when(F.col('estado_ref').isin('P','T','DS','CAP'), 1).otherwise(0),
            'dias_venta': F.when(F.col('estado_ref').isin('P','T','D_OJT','FI','FJ','NC'), 1).otherwise(0),
            't_acu': F.when(F.col('estado_ref').isin('T'), 1).otherwise(0),
            'p_acu': F.when(F.col('estado_ref').isin('P'), 1).otherwise(0),
        }
        for nombre_col, condicion in condiciones.items():
            df = df.withColumn(nombre_col, condicion)

        return  df

    query1 = f"""
        SELECT 
        [DNI] as d_identidad
        ,[NOMBRES] as nombres_siop
        ,case 
        when [CAMPAÑA]='ENTEL - PORTABILIDAD PERFILADA 2' then 'ENTEL - PORTABILIDAD PERFILADA'
        when [CAMPAÑA]='ENTEL - PORTABILIDAD EXCLUSIVA' then 'ENTEL - PORTABILIDAD PERFILADA'
        when [CAMPAÑA]='ENTEL - PORTABILIDAD NO PERFILADA' then 'ENTEL - PORTABILIDAD PERFILADA'
        else [CAMPAÑA]
        end as campana_siop
        ,[SUPERVISOR] as supervisor_siop
        ,[ENTRADA] as entrada
        ,[SALIDA] as salida
        ,[JORNADA] as jornada
        ,[CARGO] as cargo
        ,[ESTADO] as estado_siop
        ,[SUBESTADO] as sub_estado
        ,[F_INICIO] as f_inicio
        ,Tipo_trabajo as tipo_trabajo
        ,[fecha] as fecha
        ,[usuario_temp] as usuario_siop
        FROM [DB_dota].[dbo].[tb_dota_diaria_siop]
        where fecha between DATEADD(MONTH, DATEDIFF(MONTH, 0, '{fecha_fin}'), 0) and '{fecha_fin}'
        and CAMPAÑA in(
            'ENTEL - MIGRACIONES'
            ,'ENTEL - PORTABILIDAD EXCLUSIVA'
            ,'ENTEL - PORTABILIDAD NO PERFILADA'
            ,'ENTEL - PORTABILIDAD PERFILADA'
            ,'ENTEL - PORTABILIDAD PERFILADA 2'
            ,'ENTEL - PORTABILIDAD PREPAGO IVR'
            ,'ENTEL - PORTABILIDAD RECUPERADOS'
            ,'ENTEL - SEGUNDAS LINEAS'
            ,'ENTEL - HOGAR FIBRA OPTICA'
        )
    """
    df_dota = obtener_tabla_sql(spark,"DB_CCR",query1)

    query1 = f"""
        SELECT 
        distinct
        [DNI] as dni_operaciones
        ,[NOMBRES] as supervisor_siop
        ,[CARGO] as cargo_operaciones
        ,F_INICIO as f_inicio_super
        ,case 
        when [CAMPAÑA]='ENTEL - PORTABILIDAD EXCLUSIVA' then 'MOVIL'
        when [CAMPAÑA]='ENTEL - PORTABILIDAD PERFILADA 2' then 'MOVIL'
        when [CAMPAÑA]='ENTEL - PORTABILIDAD NO PERFILADA' then 'MOVIL'
        when [CAMPAÑA]='ENTEL - PORTABILIDAD EMPRESAS' then 'MOVIL'
        when [CAMPAÑA]='ENTEL - PORTABILIDAD RECUPERADOS' then 'MOVIL'
        when [CAMPAÑA]='ENTEL - PORTABILIDAD PREPAGO IVR' then 'MOVIL'
        when [CAMPAÑA]='ENTEL - SEGUNDAS LINEAS' then 'MOVIL'
        when [CAMPAÑA]='ENTEL - MIGRACIONES' then 'MOVIL'	
        when [CAMPAÑA]='ENTEL - HOGAR FIBRA OPTICA' then 'MOVIL'	
        else 'OTROS'
        end as tipo_gestion
        FROM [DB_dota].[dbo].[tb_dota_diaria_siop]
        where cargo not like '%TELEOPERADOR%'
    """
    df_info_super = obtener_tabla_sql(spark,"DB_CCR",query1)

    df_dota = df_dota.withColumn("d_identidad", regexp_replace(col("d_identidad"), r'^0+', ''))

    df_dota = df_dota.join(df_info_super,['supervisor_siop'],'left')

    window_spec_01 = Window.partitionBy('fecha','d_identidad').orderBy(col("fecha").desc())
    df_dota = df_dota.withColumn("unico", row_number().over(window_spec_01))
    df_dota= df_dota.filter(col("unico") == 1).drop('unico')

    query1 = f"""
        select d_identidad,fecha,duracion,estado_ref,1 as ref_asistencia,fecha_descarga from DB_dota.dbo.tb_asistencia_diaria_siop
        where fecha between DATEADD(MONTH, DATEDIFF(MONTH, 0, '{fecha_fin}'), 0) and '{fecha_fin}'
    """
    df_asistencia_tot = obtener_tabla_sql(spark,"DB_CCR",query1)

    # window_spec_01 = Window.partitionBy('fecha','d_identidad').orderBy(col("fecha_descarga").desc())
    window_spec_01 = Window.partitionBy('fecha','d_identidad').orderBy(col("fecha_descarga").desc())
    df_asistencia_tot = df_asistencia_tot.withColumn("unico", row_number().over(window_spec_01))
    df_asistencia_tot= df_asistencia_tot.filter(col("unico") == 1).drop('unico','fecha_descarga')

    df_asistencia_tot = df_asistencia_tot.withColumn("d_identidad", regexp_replace(col("d_identidad"), r'^0+', ''))

    valores_reemplazo = {
        'estado_ref': 'BJ',
    }
    df_asistencia_tot = df_asistencia_tot.fillna(valores_reemplazo)

    df_min_fecha = (
        df_asistencia_tot.filter(F.col("estado_ref") != "BJ")
        .groupBy("d_identidad")
        .agg(F.min("fecha").alias("fecha_ingreso"))
    )

    df_asistencia = (
        df_asistencia_tot.join(df_min_fecha, ["d_identidad"],"inner")
        .filter(F.col("fecha") >= F.col("fecha_ingreso")).drop('fecha_ingreso')
    )

    df_dota = df_dota.withColumn('contrato',when(col('estado_siop')=='Activo','Contratado').otherwise('Baja'))

    df_dota = df_dota.withColumn(
        'estado',
        when(col('estado_siop') == 'Baja', col('sub_estado'))
        .when(
            col('sub_estado').isin('Conectado y trabajando', 'Embarazada', 'En periodo de lactancia'),
            'Activo'
        )
        .otherwise('Inactivo')
    )

    df_asistencia=df_asistencia.withColumn('estado_ref',when(col('estado_ref')=='_','PEND').otherwise(col('estado_ref')))

    df_asistencia=add_columnas_necesarias(df_asistencia)
    df_tot=df_asistencia.join(df_dota,['d_identidad','fecha'],'left')

    window = Window.partitionBy("d_identidad").orderBy("fecha").rowsBetween(Window.unboundedPreceding, 0)
    df_tot=  df_tot.withColumn("nombres_siop", F.last("nombres_siop", True).over(window)) \
                    .withColumn("dni_operaciones", F.last("dni_operaciones", True).over(window)) \
                    .withColumn("cargo_operaciones", F.last("cargo_operaciones", True).over(window)) \
                    .withColumn("campana_siop", F.last("campana_siop", True).over(window)) \
                    .withColumn("supervisor_siop", F.last("supervisor_siop", True).over(window)) \
                    .withColumn("entrada", F.last("entrada", True).over(window)) \
                    .withColumn("salida", F.last("salida", True).over(window)) \
                    .withColumn("usuario_siop", F.last("usuario_siop", True).over(window)) \
                    .withColumn("cargo", F.last("cargo", True).over(window)) \
                    .withColumn("estado_siop", F.last("estado_siop", True).over(window)) \
                    .withColumn("sub_estado", F.last("sub_estado", True).over(window)) \
                    .withColumn("f_inicio", F.last("f_inicio", True).over(window)) \
                    .withColumn("contrato", F.last("contrato", True).over(window)) \
                    .withColumn("estado", F.last("estado", True).over(window))\
                    .withColumn("tipo_gestion", F.last("tipo_gestion", True).over(window))

    window = Window.partitionBy("d_identidad").orderBy(col("fecha").desc()).rowsBetween(Window.unboundedPreceding, 0)
    df_tot=  df_tot.withColumn("nombres_siop", F.last("nombres_siop", True).over(window)) \
                    .withColumn("usuario_siop", F.last("usuario_siop", True).over(window)) \
                    .withColumn("dni_operaciones", F.last("dni_operaciones", True).over(window)) \
                    .withColumn("cargo_operaciones", F.last("cargo_operaciones", True).over(window)) \
                    .withColumn("campana_siop", F.last("campana_siop", True).over(window)) \
                    .withColumn("supervisor_siop", F.last("supervisor_siop", True).over(window)) \
                    .withColumn("entrada", F.last("entrada", True).over(window)) \
                    .withColumn("salida", F.last("salida", True).over(window)) \
                    .withColumn("cargo", F.last("cargo", True).over(window)) \
                    .withColumn("estado_siop", F.last("estado_siop", True).over(window)) \
                    .withColumn("sub_estado", F.last("sub_estado", True).over(window)) \
                    .withColumn("f_inicio", F.last("f_inicio", True).over(window)) \
                    .withColumn("contrato", F.last("contrato", True).over(window)) \
                    .withColumn("estado", F.last("estado", True).over(window))\
                    .withColumn("tipo_gestion", F.last("tipo_gestion", True).over(window))
                    
    valores_reemplazo = {
        'duracion': 0,
        'estado_ref': 'BJ',
        'debe_laborar': 0,
        'laborados': 0,
        'faltas': 0,
        'inactivo': 0,
        'total': 0,
        'contratados_dia': 0,
        'activos_dia': 0,
        'conectados_dia': 0,
        'tipo_gestion': 'OTROS',
    }
    df_tot = df_tot.fillna(valores_reemplazo)

    df_tot = df_tot.withColumn('campana_siop_a',when(col('campana_siop').contains('MIGRA'),'MIGRACIONES')
                            .when(col('campana_siop')=='ENTEL - PORTABILIDAD PERFILADA','PERFILADA')
                            .when(col('campana_siop')=='ENTEL - PORTABILIDAD PERFILADA 2','PERFILADA')
                            .when(col('campana_siop')=='ENTEL - PORTABILIDAD PREPAGO IVR' ,'IVR')
                            .when(col('campana_siop')=='ENTEL - PORTABILIDAD RECUPERADOS','RECUPERADOS')
                            .when(col('campana_siop')=='ENTEL - HOGAR FIBRA OPTICA','HOGAR FIBRA')
                            .when(col('campana_siop')=='ENTEL - SEGUNDAS LINEAS','SEGUNDAS LINEAS')
                            .otherwise('NO_APLICA'))

    df_tot=df_tot.persist()
    categoricas = [
        "fecha",'d_identidad',"estado_ref", "ref_asistencia", "supervisor_siop", "nombres_siop",
        "campana_siop", "entrada", "salida", "jornada", "cargo", "estado_siop",
        "sub_estado", "f_inicio", "tipo_trabajo",
        "usuario_siop", "dni_operaciones", "cargo_operaciones", "f_inicio_super",
        "tipo_gestion", "contrato", "estado", "campana_siop_a",
    ]

    numericas = [
        "debe_laborar", "laborados", "faltas", "inactivo", "total",
        "contratados_dia", "activos_dia", "conectados_dia", "dias_venta",
        "t_acu", "p_acu"
    ]   

    df_categ = df_tot.select(categoricas)

    window_spec_01 = Window.partitionBy('d_identidad').orderBy(col("fecha").desc())
    df_categ = df_categ.withColumn("unico", row_number().over(window_spec_01))
    df_categ= df_categ.filter((col("unico") == 1) &
                             (col("fecha") == fecha_fin) ).drop('unico')

    df_num = df_tot.groupBy("d_identidad").agg(
        *[F.sum(F.col(c)).alias(c) for c in numericas]
    )

    df_tot= df_categ.join(df_num, on="d_identidad", how="inner")


    df_tot_resumen = df_tot.withColumn("dias_antiguedad", datediff(col("fecha"), col("f_inicio")))
    df_tot_resumen = df_tot_resumen.withColumn("dias_antiguedad_super", datediff(col("fecha"), col("f_inicio_super")))
    

    df_tot_resumen = df_tot_resumen.withColumn('antiguedad',when(col('dias_antiguedad')<15,'OJT')
                                            .when(col('dias_antiguedad')<30,'menor a 30 días')
                                            .when(col('dias_antiguedad')<91,'1 a 3 meses')
                                            .when(col('dias_antiguedad')<181,'3 a 6 meses')
                                            .otherwise('mayor a 6 meses'))

    return  df_tot_resumen.withColumn('antiguedad_super',when(col('dias_antiguedad_super')<15,'OJT')
                                                .when(col('dias_antiguedad_super')<30,'menor a 30 días')
                                                .when(col('dias_antiguedad_super')<91,'1 a 3 meses')
                                                .when(col('dias_antiguedad_super')<181,'3 a 6 meses')
                                                .otherwise('mayor a 6 meses'))

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-8.4.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-8.4.1.jre11.jar') \
    .config('spark.executor.memory', '10g') \
    .config('spark.driver.memory', '10g') \
    .config('spark.sql.session.timeZone', 'UTC') \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()


campaigns_out = [
    ('CMP_CL_OUT_ECO_A365_MIGRACIONES', 'TB_outbound_ccrmovil', 'DB_MIGRACIONES','MIGRACIONES_1'),
    ('CMP_CL_OUT_ECO_A365_MIGRACIONES2', 'TB_outbound_ccrmovil', 'DB_MIGRACIONES','MIGRACIONES_2'),
    ('CMP_CL_OUT_ECO_A365_SEGUNDAS_LINEAS', 'TB_outbound_ccrmovil', 'DB_SEGUNDAS','SEGUNDAS_LINEAS'),
    ('CMP_CL_OUT_ECO_A365_PORTA_PERFILADA', 'TB_outbound_ccrmovil', 'DB_PERFILADA','PORTABILIDAD_PERFILADA_1'),
    ('CMP_CL_OUT_ECO_A365_PORTA_EXCLUSIVA', 'TB_outbound_ccrmovil', 'DB_PERFILADA','PORTABILIDAD_PERFILADA_2'),
    ('CMP_CL_OUT_ECO_A365_PORTA_RECUPERADOS', 'TB_outbound_ccrmovil', 'DB_RECUPERADOS','PORTABILIDAD_RECUPERADOS'),
    ('CMP_CL_OUT_ECO_A365_PORTA_IVR', 'TB_outbound_ccrmovilivr', 'DB_IVR','PORTABILIDAD_PREPAGO_IVR'),
    ('CMP_CL_OUT_ECO_A365_PORTA_6', 'TB_outbound_ccrmovilivr', 'DB_IVR','PORTA_IVR_PILOTO'),
    ('CMP_CL_OUT_ECO_A365_MOVIL_PORTA5', 'TB_outbound_ccrmovil', 'DB_PERFILADA','MOVIL_PORTA5'),
    ('CMP_CL_OUT_ECO_A365_SEGUNDAS_LINEAS2', 'TB_outbound_ccrmovil', 'DB_SEGUNDAS','SEGUNDAS_LINEAS2'),
    ('CMP_CL_OUT_ECO_A365_MIGRACIONES3', 'TB_outbound_ccrmovil', 'DB_MIGRACIONES','CONECTA_MAYOR'),
    ('CMP_CL_OUT_ECO_A365_RENOVACION_EQUIPOS3', 'TB_outbound_ccrmovil', 'DB_HOGAR','FIBRA_CROSS_3.0'),
    ('CMP_CL_OUT_ECO_A365_RENOVACION_EQUIPOS2', 'TB_outbound_ccrmovil', 'DB_HOGAR','FIBRA_CROSS_3.0_MOVIL'),
    ('CMP_CL_OUT_ECO_A365_SEGUNDAS_LINEAS2', 'TB_outbound_ccrmovil', 'DB_SEGUNDAS','SEGUNDAS_LINEAS2'),
    ('CMP_CL_OUT_ECO_A365_MIGRACIONES', 'TB_outbound_ccrmovil', 'DB_MIGRACIONES','MIGRACIONES_2'),
    ('CMP_CL_OUT_ECO_A365_HOGAR_FIBRA_CLIENTE', 'TB_outbound_ccrmovil', 'DB_ECO_HOGAR','HOGAR_FIBRA_CLIENTE'),
    ('CMP_CL_OUT_ECO_A365_HOGAR_FIBRA_NOCLIENTE', 'TB_outbound_ccrmovil', 'DB_ECO_HOGAR','HOGAR_FIBRA_NOCLIENTE'),
]
fecha_fin='2025-09-23'
indices = [0,2,3,5,6,8,15,16] 



⚠️ No se encontraron archivos válidos en la carpeta.


In [6]:
df_ccr=ccr_discado_mr_mes(spark,fecha_fin,indices)
df_cloud=cloud_discado_mr_dia(spark)

df_conexiones=cloud_conexiones(spark,fecha_fin).persist()

df_dota=consolidado_ult_dota_asistencia(spark,fecha_fin).persist()


⚠️ No se encontraron archivos válidos en la carpeta.


In [7]:

df_ventas_brutas=cargar_web_ventas(spark,fecha_fin)
df_ventas_fibra_wdv=ventas_fibra_wdv(spark)

df_ventas_brutas=df_ventas_brutas.unionByName(df_ventas_fibra_wdv)

agg_expr1 = ['d_identidad']

agg_expr2 = [
    F.count('tipo_operacion').alias("q_ventas"),
    F.count(F.when(F.col("tipo_operacion") == "HOGAR FIBRA", F.col("tipo_operacion"))).alias("q_fibra_hogar"),
    F.count(F.when(F.col("tipo_operacion") == "2DA LINEA", F.col("tipo_operacion"))).alias("q_2da_linea"),
    F.count(F.when(F.col("tipo_operacion") == "RENOVACION DE EQUIPO", F.col("tipo_operacion"))).alias("q_renovacion_equipo"),
    F.count(F.when(F.col("tipo_operacion") == "NUEVA LINEA", F.col("tipo_operacion"))).alias("q_nueva_linea"),
    F.count(F.when(F.col("tipo_operacion") == "PORTA PP-SS", F.col("tipo_operacion"))).alias("q_porta_pp_ss"),
    F.count(F.when(F.col("tipo_operacion") == "PORTA SS-SS", F.col("tipo_operacion"))).alias("q_porta_ss_ss"),
    F.count(F.when(F.col("tipo_operacion") == "MIGRACION", F.col("tipo_operacion"))).alias("q_migracion"),
]
# mover_archivo(ruta_base,temporal)

df_ventas= df_ventas_brutas.groupBy(agg_expr1).agg(*agg_expr2)


C:\Users\A365\AppData\Local\Temp\ipykernel_39676\3927279639.py:362: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df01 = df01.applymap(replace_newlines)


In [8]:

if df_cloud:
    df=df_conexiones.select('agent','usuario_siop').distinct()
    df_cloud=df_cloud.join(df,['agent'],'left').drop('agent')
    
    agg_expr1 = ['usuario_siop']

    agg_expr2 = [
        F.sum('q_mr_contacto_cloud').alias("q_mr_contacto_ccr"), 
        F.sum('q_mr_no_contacto_cloud').alias("q_mr_no_contacto_ccr"), 
    ]
    # mover_archivo(ruta_base,temporal)

    df_cloud= df_cloud.groupBy(agg_expr1).agg(*agg_expr2)
    df_ccr = df_ccr.unionByName(df_cloud)

agg_expr1 = ['usuario_siop']

agg_expr2 = [
    F.sum('logged_in').alias("logged_in"), 
    F.sum('on_queue').alias("on_queue"), 
    F.sum('idle').alias("idle"), 
    F.sum('off_queue').alias("off_queue"), 
    F.sum('Interacting').alias("Interacting"), 
    F.sum('t_operativo').alias("t_operativo"), 
]

df_conexiones= df_conexiones.groupBy(agg_expr1).agg(*agg_expr2)

from functools import reduce

df_dota_ref=df_dota.select('d_identidad','usuario_siop').distinct()
df_ventas=df_ventas.join(df_dota_ref,['d_identidad'],'left').drop('d_identidad')

dfs1 = [df_ccr, df_conexiones,df_ventas]

df_01 = reduce(
    lambda left, right: left.join(right, 'usuario_siop', "outer"),
    dfs1
)

df_resumen=df_01.join(df_dota,['usuario_siop'],'left')
df_resumen=df_resumen.filter(col('cargo')=='TELEOPERADOR')

window_spec = Window.partitionBy('d_identidad').orderBy(col("d_identidad").desc())
df_resumen = df_resumen.withColumn("unico", row_number().over(window_spec))


df_resumen=df_resumen.filter(col('cargo')=='TELEOPERADOR')


In [9]:
overwrite_table_SQL(df_resumen,'DB_temporal','ala_01','')

In [ ]:

    from pyspark.sql import Row

    filas_nuevas = [
        Row(campana_siop_a="HOGAR FIBRA", fecha="2025-09-18", ref_fecha1=1),
        Row(campana_siop_a="HOGAR FIBRA", fecha="2025-09-19", ref_fecha1=1)
    ]

    df_adicional_campana_a = spark.createDataFrame(filas_nuevas)
    df_fecha_no_aplica = df_fecha_no_aplica.union(df_adicional_campana_a)


    # df_tiempos = df_tiempos.withColumn("d_identidad", regexp_replace(col("d_identidad"), r'^0+', ''))


In [ ]:
=IF(HASONEVALUE(ala_01[d_identidad]);
	RANKX ( 
	ALL( ala_01[d_identidad];
	ala_01[supervisor];
	ala_01[Nombre_completo];
	ala_01[estado_];
	ala_01[antiguedad_];
	ala_01[usuario_]		);
	[Efect];
	;
    	DESC;
    	Dense)
    	;BLANK()
   )


In [ ]:
=VAR TablaBase =
    all(
ala_01[d_identidad];
	ala_01[supervisor];
	ala_01[Nombre_completo];
	ala_01[estado_];
	ala_01[antiguedad_];
	ala_01[usuario_]	
    )
VAR maxPos = MAXX(TablaBase ;[ranking])
VAR P75 = maxPos * 0.75
VAR P50 = maxPos * 0.5
VAR P25 = maxPos * 0.25
RETURN
IF (
    HASONEVALUE ( ala_01[d_identidad] );
    SWITCH (
        TRUE();
        [ranking] <= P25; "Q1";
        [ranking] <= P50; "Q2";
        [ranking] <= P75; "Q3";       
         "Q4"
    )
)


In [ ]:

query1=f"""
select
nombre_skill as campana_siop_a
,fecha
,1 as ref_fecha1
from DB_tiempo.dbo.fecha_no_aplica
where nombre_skill<>'MIGRACIONES_2'
"""
df_fecha_no_aplica=obtener_tabla_sql(spark,'DB_tiempo',query1)

from pyspark.sql import Row

filas_nuevas = [
    Row(campana_siop_a="HOGAR FIBRA", fecha="2025-09-18", ref_fecha1=1),
    Row(campana_siop_a="HOGAR FIBRA", fecha="2025-09-19", ref_fecha1=1)
]

df_adicional_campana_a = spark.createDataFrame(filas_nuevas)
df_fecha_no_aplica = df_fecha_no_aplica.union(df_adicional_campana_a)


df_fecha_no_aplica = df_fecha_no_aplica.withColumn('campana_siop_a',when(col('campana_siop_a').contains('MIGRA'),'MIGRACIONES')
                        .when(col('campana_siop_a')=='PORTABILIDAD_PERFILADA_1','PERFILADA')
                        .when(col('campana_siop_a')=='MOVIL_PORTA5','PERFILADA')
                        .when(col('campana_siop_a')=='PORTABILIDAD_PREPAGO_IVR' ,'IVR')
                        .when(col('campana_siop_a')=='PORTABILIDAD_RECUPERADOS','RECUPERADOS')
                        .when(col('campana_siop_a')=='SEGUNDAS_LINEAS','SEGUNDAS LINEAS')
                        .when(col('campana_siop_a')=='HOGAR FIBRA','HOGAR FIBRA')
                        .otherwise('NO__APLICA'))


df_resumen=df_resumen.join(df_fecha_no_aplica,['fecha','campana_siop_a'],'left')

df_resumen = df_resumen.withColumn(
    "dia_no_laborable",
    F.when(F.date_format("fecha", "E") == "Sun", F.lit(0))    # domingo
    .when(col('ref_fecha1').isNotNull(),F.lit(0))
    .otherwise(F.lit(1))                                     # demás días
).drop('ref_fecha1')


df_resumen=df_resumen.select('d_identidad', 'nombres_siop', 'agent', 'usuario_siop', 'supervisor_siop', 'estado', 'sub_estado', 'campana_siop', 'q_ventas_wdv', 'q_ventas_wdv_fibra', 'q_d_contacto_ccr', 'q_d_no_contacto_ccr', 'q_mr_contacto_ccr', 'q_mr_no_contacto_ccr', 'q_d_contacto_cloud', 'q_d_no_contacto_cloud', 'q_mr_contacto_cloud', 'q_mr_no_contacto_cloud', 'logged_in', 'on_queue', 'interacting', 'off_queue', 'estado_ref', 'idle', 'log_in', 'log_out', 'skills', 'ref_asistencia', 'entrada', 'salida', 'jornada', 'cargo', 'estado_siop', 'f_inicio', 'f_fin', 'grupo', 'antiguedad', 'subgerente', 'tipo_trabajo', 'dni_operaciones', 'cargo_operaciones', 'f_inicio_super', 'tipo_gestion', 'contrato', 'q_fibra_hogar', 'q_2da_linea', 'q_renovacion_equipo', 'q_nueva_linea', 'q_porta_pp_ss', 'q_porta_ss_ss', 'q_migracion', 'q_fibra_wdv', 'fecha', 'campana_siop_a', 'debe_laborar', 'laborados', 'faltas', 'inactivo', 'total', 'contratados_dia', 'activos_dia', 'conectados_dia', 'dias_venta', 't_acu', 'p_acu', 'dias_antiguedad', 'dias_antiguedad_super', 'antiguedad_super','dia_no_laborable', 'unico')


overwrite_table_SQL(df_resumen,'DB_a365','consolidado_out_dia_kevo','reporte_03')


AnalysisException: [UNRESOLVED_USING_COLUMN_FOR_JOIN] USING column `campana_siop_a` cannot be resolved on the left side of the join. The left-side columns: [`Interacting`, `campana_siop`, `cargo`, `cargo_operaciones`, `contrato`, `d_identidad`, `dni_operaciones`, `entrada`, `estado`, `estado_siop`, `f_inicio`, `f_inicio_super`, `fecha`, `idle`, `jornada`, `logged_in`, `nombres_siop`, `off_queue`, `on_queue`, `q_2da_linea`, `q_fibra_hogar`, `q_migracion`, `q_mr_contacto_ccr`, `q_mr_no_contacto_ccr`, `q_nueva_linea`, `q_porta_pp_ss`, `q_porta_ss_ss`, `q_renovacion_equipo`, `q_ventas`, `salida`, `sub_estado`, `supervisor_siop`, `t_operativo`, `tipo_gestion`, `tipo_trabajo`, `unico`, `usuario_siop`]. SQLSTATE: 42703

In [ ]:
query1=f"""
select
nombre_skill as campana_siop_a
,fecha
,1 as ref_fecha1
from DB_tiempo.dbo.fecha_no_aplica
where nombre_skill<>'MIGRACIONES_2'
"""
df_fecha_no_aplica=obtener_tabla_sql(spark,'DB_tiempo',query1)


In [7]:
query1=f"""
select
nombre_skill as campana_siop_a
,fecha
,1 as ref_fecha1
from DB_tiempo.dbo.fecha_no_aplica
where nombre_skill<>'MIGRACIONES_2'
"""
df_fecha_no_aplica_1=obtener_tabla_sql(spark,'DB_tiempo',query1)
print(df_fecha_no_aplica_1.columns)

['campana_siop_a', 'fecha', 'ref_fecha1']


In [ ]:
df_extra.show()

In [ ]:
=VAR TablaBase =
    all(
consolidado_out_dia_kevo[d_identidad];
	consolidado_out_dia_kevo[supervisor];
	consolidado_out_dia_kevo[nombre_completo];
	consolidado_out_dia_kevo[estado_siop];
	consolidado_out_dia_kevo[antiguedad];
	consolidado_out_dia_kevo[usuario]		
    )
VAR maxPos = MAXX(TablaBase ;[ranking])
VAR P75 = maxPos * 0.75
VAR P50 = maxPos * 0.5
VAR P25 = maxPos * 0.25
RETURN
IF (
    HASONEVALUE ( consolidado_out_dia_kevo[dni_] );
    SWITCH (
        TRUE();
        [ranking] <= P25; "Q1";
        [ranking] <= P50; "Q2";
        [ranking] <= P75; "Q3";       
         "Q4"
    )
)


In [ ]:
df_resumen=df_resumen.select('usuario_siop',
 'unico')



In [ ]:

def tabla_tarifa_conversion_costo_ingreso(spark):
    subir_hab_in(spark)

    query1 = f"""
    select * from DB_venta.dbo.tarifa_producto
    """
    df_tarifa = obtener_tabla_sql(spark,"DB_CCR",query1)

    from pyspark.sql import functions as F

    df_tarifa = df_tarifa.withColumn("tipo_operacion",
                                        F.concat(F.lit("tar_"),
                                                F.regexp_replace(F.lower(F.col("tipo_operacion")), " ","_")))

    df_pivot_tarifa = (
        df_tarifa.groupBy(col("campana_reporte_gc").alias('campana_siop_a'))  # columnas que se mantienen
        .pivot("tipo_operacion")                   # columna que se vuelve encabezado
        .agg(F.first("tarifa"))                    # valor que va en las celdas
    )

    query1=f"""
    select tipo_operacion, porc_conversion AS porc_conver_01 from DB_venta.dbo.porc_conversion_producto
    where fecha='2025-07-01'
    """
    df_conversion_01 = obtener_tabla_sql(spark,'DB_tiempo',query1)

    df_conversion_01 = df_conversion_01.withColumn("tipo_operacion",
                                        F.concat(F.lit("conv_"),
                                                F.regexp_replace(F.lower(F.col("tipo_operacion")), " ","_")))

    df_pivot_conversion_01 = df_conversion_01.groupBy().pivot("tipo_operacion").agg(F.first("porc_conver_01"))

    query1=f"""
    select tipo_operacion, porc_conversion AS porc_conver_edu from DB_venta.dbo.porc_conversion_producto_edu
    """
    df_conversion_edu = obtener_tabla_sql(spark,'DB_tiempo',query1)
    df_conversion_edu = df_conversion_edu.withColumn("tipo_operacion",
                                        F.concat(F.lit("conv_edu"),
                                                F.regexp_replace(F.lower(F.col("tipo_operacion")), " ","_")))

    df_pivot_conversion_edu = df_conversion_edu.groupBy().pivot("tipo_operacion").agg(F.first("porc_conver_edu"))

    query1=f"""
    select tipo_operacion, porc_conversion AS porc_conver_paya from DB_venta.dbo.porc_conversion_producto_asistencia
    """
    df_conversion_paya = obtener_tabla_sql(spark,'DB_tiempo',query1)
    df_conversion_paya = df_conversion_paya.withColumn("tipo_operacion",
                                        F.concat(F.lit("conv_paya"),
                                                F.regexp_replace(F.lower(F.col("tipo_operacion")), " ","_")))

    df_pivot_conversion_paya = df_conversion_paya.groupBy().pivot("tipo_operacion").agg(F.first("porc_conver_paya"))

    from functools import reduce

    dfs = [df_pivot_tarifa,df_pivot_conversion_01, df_pivot_conversion_edu, df_pivot_conversion_paya]

    df_tarifa_conversion = reduce(
        lambda df1, df2: df1.crossJoin(df2),
        dfs
    )

    query1=f"""
    select * from DB_venta.dbo.tabla_costo_TC_ingreso_costo
    """
    df_costo_ingreso = obtener_tabla_sql(spark,'DB_tiempo',query1)

    return   df_tarifa_conversion.join(df_costo_ingreso,["campana_siop_a"],"left")

def obtener_ventas(spark,fecha_fin):
    subir_ventas_outbound_funnel_back(spark)
    subir_ventas_fibra_wdv(spark)
    query1=f"""
    select tipo_operacion, porc_conversion AS porc_conversion_01 from DB_venta.dbo.porc_conversion_producto
    where fecha='2025-07-01'
    """
    df_conversion_01 = obtener_tabla_sql(spark,'DB_tiempo',query1)

    query1=f"""
    select tipo_operacion, porc_conversion AS porc_conversion_edu from DB_venta.dbo.porc_conversion_producto_edu
    """
    df_conversion_edu = obtener_tabla_sql(spark,'DB_tiempo',query1)

    query1=f"""
    select tipo_operacion, porc_conversion AS porc_conversion_paya from DB_venta.dbo.porc_conversion_producto_asistencia
    """
    df_conversion_paya = obtener_tabla_sql(spark,'DB_tiempo',query1)

    query1=f"""
    select campana_reporte_gc as campana_siop_a, tipo_operacion, tarifa from DB_venta.dbo.tarifa_producto 
    """
    df_tarifa = obtener_tabla_sql(spark,'DB_tiempo',query1).persist()

    query1=f"""
    select distinct fecha,d_identidad,campana_siop_a from DB_A365.dbo.consolidado01_dota
    """
    df_dota_dia=obtener_tabla_sql(spark,'DB_tiempo',query1).persist()

    query1=f"""
    select
    ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS indice_001
    ,fecha_registro as fecha
    , tipo_operacion
    , num_doc_usuario_reg_linea as d_identidad
    ,campania_usuario_reg_linea
    , q_ventas
    , q_ult_3_meses as dentro
    , q_ventas-q_ult_3_meses as fuera
    from DB_venta.dbo.resumen_fb_01
    where fecha_registro between DATEADD(MONTH, DATEDIFF(MONTH, 0, '{fecha_fin}'), 0) and '{fecha_fin}'
    """
    df_wdv=obtener_tabla_sql(spark,'DB_tiempo',query1)

    df_wdv = df_wdv.withColumn("d_identidad", regexp_replace(col("d_identidad"), r'^0+', ''))

    df_wdv=df_wdv.join(df_dota_dia,['fecha','d_identidad'],'left')

    df_wdv = df_wdv.withColumn('campana_siop_a',when(col('campana_siop_a').isNull(),
                                                    when(col('campania_usuario_reg_linea').contains('MIGRA'),'MIGRACIONES')
                                                    .when(col('campania_usuario_reg_linea').contains('PERFILADA'),'PERFILADA')
                                                    .when(col('campania_usuario_reg_linea')=='PORTA5','PERFILADA')
                                                    .when(col('campania_usuario_reg_linea')=='PREPAGO IVR','IVR')
                                                    .when(col('campania_usuario_reg_linea')=='EXCLUSIVA RECUPERADOS','RECUPERADOS')
                                                    .when(col('campania_usuario_reg_linea').contains('SEGUNDAS LINEAS'),'SEGUNDAS LINEAS')
                                                    .otherwise('NO_APLICA'))
                                                .otherwise(col('campana_siop_a')))

    window = Window.partitionBy("d_identidad").orderBy("fecha").rowsBetween(Window.unboundedPreceding, 0)
    df_wdv=  df_wdv.withColumn("campana_siop_a", F.last("campana_siop_a", True).over(window)) 
    window = Window.partitionBy("d_identidad").orderBy(col("fecha").desc()).rowsBetween(Window.unboundedPreceding, 0)
    df_wdv=  df_wdv.withColumn("campana_siop_a", F.last("campana_siop_a", True).over(window)) 
                
    df_wdv=df_wdv.join(df_conversion_01,['tipo_operacion'],'left')
    df_wdv=df_wdv.join(df_conversion_edu,['tipo_operacion'],'left')
    df_wdv=df_wdv.join(df_tarifa,['tipo_operacion','campana_siop_a'],'left')

    df_wdv = df_wdv.withColumn("q_ventas", col("q_ventas").cast(IntegerType())) \
                .withColumn("porc_conversion_01", col("porc_conversion_01").cast(DoubleType())) \
                .withColumn("porc_conversion_edu", col("porc_conversion_edu").cast(DoubleType())) \
                .withColumn("tarifa", col("tarifa").cast(IntegerType()))

    df_wdv = df_wdv.withColumn(
        "por_conversion_dia_semana",
        F.when(F.date_format("fecha", "E") == "Sat", F.lit(0.5))  # sábado
        .when(F.date_format("fecha", "E") == "Sun", F.lit(0))    # domingo
        .otherwise(F.lit(1))                                     # demás días
    )
    df_wdv = df_wdv.filter(col('campana_siop_a').isNotNull())

    df_wdv = df_wdv.withColumn("q_ventas_netas_conversion",col("q_ventas") * col("porc_conversion_01") * col("por_conversion_dia_semana"))
    df_wdv = df_wdv.withColumn("factura_en_base_q_ventas_netas_proyectadas",col("q_ventas") * col("porc_conversion_01") * col("tarifa") * col("por_conversion_dia_semana"))
    df_wdv = df_wdv.withColumn("factura_en_base_q_ventas_netas_proyectadas_edu",col("q_ventas") * col("porc_conversion_edu") * col("tarifa") * col("por_conversion_dia_semana"))
    df_wdv = df_wdv.withColumn("q_ventas_netas_conversion_edu",col("q_ventas") * col("porc_conversion_edu") * col("por_conversion_dia_semana"))

    df_wdv = df_wdv.withColumn(
        'q_ventas_foco',
        when(
            col('campana_siop_a').contains('MIGRACIONES') & (col('tipo_operacion') == 'MIGRACION'),
            col('q_ventas')
        ).when(
            (col('campana_siop_a').isin('PERFILADA','RECUPERADOS','IVR','SEGUNDAS LINEAS')) & col('tipo_operacion').contains('PORTA'),
            col('q_ventas')
        ).when(
            (col('campana_siop_a')=='SEGUNDAS LINEAS') & (col('tipo_operacion')=='2DA LINEA'),
            col('q_ventas')
        )
        .otherwise(0)
    )
    df_wdv = df_wdv.withColumn(
        'q_ventas_netas_conversion_foco',
        when(
            col('campana_siop_a').contains('MIGRACIONES') & (col('tipo_operacion') == 'MIGRACION'),
            col('q_ventas_netas_conversion')
        ).when(
            (col('campana_siop_a').isin('PERFILADA','RECUPERADOS','IVR','SEGUNDAS LINEAS')) & col('tipo_operacion').contains('PORTA'),
            col('q_ventas_netas_conversion')
        ).when(
            (col('campana_siop_a')=='SEGUNDAS LINEAS') & (col('tipo_operacion')=='2DA LINEA'),
            col('q_ventas_netas_conversion')
        )
        .otherwise(0)
    )
    df_wdv = df_wdv.withColumn(
        'q_ventas_netas_conversion_edu_foco',
        when(
            col('campana_siop_a').contains('MIGRACIONES') & (col('tipo_operacion') == 'MIGRACION'),
            col('q_ventas_netas_conversion_edu')
        ).when(
            (col('campana_siop_a').isin('PERFILADA','RECUPERADOS','IVR','SEGUNDAS LINEAS')) & col('tipo_operacion').contains('PORTA'),
            col('q_ventas_netas_conversion_edu')
        ).when(
            (col('campana_siop_a')=='SEGUNDAS LINEAS') & (col('tipo_operacion')=='2DA LINEA'),
            col('q_ventas_netas_conversion_edu')
        )
        .otherwise(0)
    )
    df_wdv = df_wdv.withColumn(
        'factura_en_base_q_ventas_netas_proyectadas_foco',
        when(
            col('campana_siop_a').contains('MIGRACIONES') & (col('tipo_operacion') == 'MIGRACION'),
            col('factura_en_base_q_ventas_netas_proyectadas')
        ).when(
            (col('campana_siop_a').isin('PERFILADA','RECUPERADOS','IVR','SEGUNDAS LINEAS')) & col('tipo_operacion').contains('PORTA'),
            col('factura_en_base_q_ventas_netas_proyectadas')
        ).when(
            (col('campana_siop_a')=='SEGUNDAS LINEAS') & (col('tipo_operacion')=='2DA LINEA'),
            col('factura_en_base_q_ventas_netas_proyectadas')
        )
        .otherwise(0)
    )
    df_wdv = df_wdv.withColumn(
        'factura_en_base_q_ventas_netas_proyectadas_foco_edu',
        when(
            col('campana_siop_a').contains('MIGRACIONES') & (col('tipo_operacion') == 'MIGRACION'),
            col('factura_en_base_q_ventas_netas_proyectadas_edu')
        ).when(
            (col('campana_siop_a').isin('PERFILADA','RECUPERADOS','IVR','SEGUNDAS LINEAS')) & col('tipo_operacion').contains('PORTA'),
            col('factura_en_base_q_ventas_netas_proyectadas_edu')
        ).when(
            (col('campana_siop_a')=='SEGUNDAS LINEAS') & (col('tipo_operacion')=='2DA LINEA'),
            col('factura_en_base_q_ventas_netas_proyectadas_edu')
        )
        .otherwise(0)
    )

    df_wdv = df_wdv.withColumn('q_migracion',when(col('tipo_operacion') == 'MIGRACION',col('q_ventas')))
    df_wdv = df_wdv.withColumn('q_nueva_linea',when(col('tipo_operacion') == 'NUEVA LINEA',col('q_ventas')))
    df_wdv = df_wdv.withColumn('q_hogar_fibra',when(col('tipo_operacion') == 'HOGAR FIBRA',col('q_ventas')))
    df_wdv = df_wdv.withColumn('q_2da_linea',when(col('tipo_operacion') == '2DA LINEA',col('q_ventas')))
    df_wdv = df_wdv.withColumn('q_portas',when((col('tipo_operacion') == 'PORTA PP-SS') |
                                            (col('tipo_operacion') == 'PORTA SS-SS'),col('q_ventas')))

    df_wdv = df_wdv.withColumn('q_migracion_conv_01',when(col('tipo_operacion') == 'MIGRACION',col('q_ventas_netas_conversion')))
    df_wdv = df_wdv.withColumn('q_nueva_linea_conv_01',when(col('tipo_operacion') == 'NUEVA LINEA',col('q_ventas_netas_conversion')))
    df_wdv = df_wdv.withColumn('q_hogar_fibra_conv_01',when(col('tipo_operacion') == 'HOGAR FIBRA',col('q_ventas_netas_conversion')))
    df_wdv = df_wdv.withColumn('q_2da_linea_conv_01',when(col('tipo_operacion') == '2DA LINEA',col('q_ventas_netas_conversion')))
    df_wdv = df_wdv.withColumn('q_portas_conv_01',when((col('tipo_operacion') == 'PORTA PP-SS') |
                                            (col('tipo_operacion') == 'PORTA SS-SS'),col('q_ventas_netas_conversion')))

    df_wdv = df_wdv.withColumn('q_migracion_conv_edu',when(col('tipo_operacion') == 'MIGRACION',col('q_ventas_netas_conversion_edu')))
    df_wdv = df_wdv.withColumn('q_nueva_linea_conv_edu',when(col('tipo_operacion') == 'NUEVA LINEA',col('q_ventas_netas_conversion_edu')))
    df_wdv = df_wdv.withColumn('q_hogar_fibra_conv_edu',when(col('tipo_operacion') == 'HOGAR FIBRA',col('q_ventas_netas_conversion_edu')))
    df_wdv = df_wdv.withColumn('q_2da_linea_conv_edu',when(col('tipo_operacion') == '2DA LINEA',col('q_ventas_netas_conversion_edu')))
    df_wdv = df_wdv.withColumn('q_portas_conv_edu',when((col('tipo_operacion') == 'PORTA PP-SS') |
                                            (col('tipo_operacion') == 'PORTA SS-SS'),col('q_ventas_netas_conversion_edu')))


    df_wdv = df_wdv.withColumn('factura_migracion_proyectadas',when(col('tipo_operacion') == 'MIGRACION',col('factura_en_base_q_ventas_netas_proyectadas')))
    df_wdv = df_wdv.withColumn('factura_nueva_linea_proyectadas',when(col('tipo_operacion') == 'NUEVA LINEA',col('factura_en_base_q_ventas_netas_proyectadas')))
    df_wdv = df_wdv.withColumn('factura_hogar_fibra_proyectadas',when(col('tipo_operacion') == 'HOGAR FIBRA',col('factura_en_base_q_ventas_netas_proyectadas')))
    df_wdv = df_wdv.withColumn('factura_2da_linea_proyectadas',when(col('tipo_operacion') == '2DA LINEA',col('factura_en_base_q_ventas_netas_proyectadas')))
    df_wdv = df_wdv.withColumn('factura_portas_proyectadas',when((col('tipo_operacion') == 'PORTA PP-SS') |
                                            (col('tipo_operacion') == 'PORTA SS-SS'),col('factura_en_base_q_ventas_netas_proyectadas')))


    agg_expr1 = [
        'fecha',
        'd_identidad',
        ]
    agg_expr2 = [
        F.first(F.col('campana_siop_a')).alias('campana_siop_wdv'),
        F.sum("q_ventas").alias("q_ventas"),
        F.sum("dentro").alias("dentro"),
        F.sum("fuera").alias("fuera"),
        F.sum("q_ventas_foco").alias("q_ventas_foco"),
        F.sum("q_migracion").alias("q_migracion"),
        F.sum("q_nueva_linea").alias("q_nueva_linea"),
        F.sum("q_hogar_fibra").alias("q_hogar_fibra"),
        F.sum("q_2da_linea").alias("q_2da_linea"),
        F.sum("q_portas").alias("q_portas"),
        
        F.sum("q_ventas_netas_conversion").alias("q_ventas_netas_conversion"),
        F.sum("q_ventas_netas_conversion_foco").alias("q_ventas_netas_conversion_foco"),
        F.sum("q_migracion_conv_01").alias("q_migracion_conv_01"),
        F.sum("q_nueva_linea_conv_01").alias("q_nueva_linea_conv_01"),
        F.sum("q_hogar_fibra_conv_01").alias("q_hogar_fibra_conv_01"),
        F.sum("q_2da_linea_conv_01").alias("q_2da_linea_conv_01"),
        F.sum("q_portas_conv_01").alias("q_portas_conv_01"),
        
        F.sum("q_ventas_netas_conversion_edu").alias("q_ventas_netas_conversion_edu"),
        F.sum("q_ventas_netas_conversion_edu_foco").alias("q_ventas_netas_conversion_edu_foco"),
        F.sum("q_migracion_conv_edu").alias("q_migracion_conv_edu"),
        F.sum("q_nueva_linea_conv_edu").alias("q_nueva_linea_conv_edu"),
        F.sum("q_hogar_fibra_conv_edu").alias("q_hogar_fibra_conv_edu"),
        F.sum("q_2da_linea_conv_edu").alias("q_2da_linea_conv_edu"),
        F.sum("q_portas_conv_edu").alias("q_portas_conv_edu"),
        
        F.sum("factura_en_base_q_ventas_netas_proyectadas").alias("factura_en_base_q_ventas_netas_proyectadas"),
        F.sum("factura_en_base_q_ventas_netas_proyectadas_foco").alias("factura_en_base_q_ventas_netas_proyectadas_foco"),
        F.sum("factura_migracion_proyectadas").alias("factura_migracion_proyectadas"),
        F.sum("factura_nueva_linea_proyectadas").alias("factura_nueva_linea_proyectadas"),
        F.sum("factura_hogar_fibra_proyectadas").alias("factura_hogar_fibra_proyectadas"),
        F.sum("factura_2da_linea_proyectadas").alias("factura_2da_linea_proyectadas"),
        F.sum("factura_portas_proyectadas").alias("factura_portas_proyectadas"),
        
    ]

    df_wdv = df_wdv.repartition('fecha','d_identidad')
    df_wdv_resumen = df_wdv.groupBy(agg_expr1).agg(*agg_expr2)

# sldkfnsdklfndsfndslkfds

    query1=f"""
        select
        num_doc_usuario_reg_linea as d_identidad
        ,nro_negocio as orden
        ,1 as ref
        ,campania_usuario_reg_linea
        from DB_venta.dbo.reporte_fb_01
    """
    df_wdv_dni=obtener_tabla_sql(spark,'DB_tiempo',query1)

    df_wdv_dni = df_wdv_dni.withColumn('campana_siop_a_01',when(col('campania_usuario_reg_linea').contains('MIGRA'),'MIGRACIONES')
                                                .when(col('campania_usuario_reg_linea').contains('PERFILADA'),'PERFILADA')
                                                .when(col('campania_usuario_reg_linea')=='PORTA5','PERFILADA')
                                                .when(col('campania_usuario_reg_linea')=='PREPAGO IVR','IVR')
                                                .when(col('campania_usuario_reg_linea')=='EXCLUSIVA RECUPERADOS','RECUPERADOS')
                                                .when(col('campania_usuario_reg_linea').contains('SEGUNDAS LINEAS'),'SEGUNDAS LINEAS')
                                                .otherwise('NO_APLICA')).drop('campania_usuario_reg_linea')

    query1=f"""
        select
        dni as d_identidad
        ,nro_negocio as orden
        ,2 as ref
        ,punto_venta
        from DB_venta.dbo.reporte_hab_01
        where dni is not null
        and punto_venta in(4822,5385,4824,4821,4820,5055,5226,3778,5015)
    """
    
    df_paya_dni=obtener_tabla_sql(spark,'DB_tiempo',query1)

    df_paya_dni = df_paya_dni.withColumn('campana_siop_a_01',when(col('punto_venta').isin(5055,5226),'PERFILADA')
                                                    .when(col('punto_venta').isin(4820),'RECUPERADOS')
                                                    .when(col('punto_venta').isin(4821),'IVR')
                                                    .when(col('punto_venta').isin(4824),'SEGUNDAS LINEAS')
                                                    .when(col('punto_venta').isin(4822,5385),'MIGRACIONES')
                                                    .otherwise('NO_APLICA')).drop('punto_venta')

    query1=f"""
    select fecha_de_creacion as fecha,* from DB_venta.dbo.tb_venta_outbound_funnel_back
    """
    df_info_back=obtener_tabla_sql(spark,'DB_tiempo',query1)

    df_dni=df_wdv_dni.union(df_paya_dni)

    window_spec_01 = Window.partitionBy('orden').orderBy(col("ref").asc())
    df_dni = df_dni.withColumn("unico", row_number().over(window_spec_01))
    df_dni= df_dni.filter(col("unico") == 1).drop('unico','ref')
    df_info_back=df_info_back.join(df_dni,['orden'],'left')
    df_info_back=df_info_back.join(df_dota_dia,['fecha', 'd_identidad'],'left')

    df_info_back = df_info_back.withColumn('campana_siop_a',when(col('campana_siop_a').isNull(),col('campana_siop_a_01'))
                                                    .otherwise(col('campana_siop_a'))).drop('campana_siop_a_01')

    window = Window.partitionBy("d_identidad").orderBy("fecha").rowsBetween(Window.unboundedPreceding, 0)
    df_info_back=  df_info_back.withColumn("campana_siop_a", F.last("campana_siop_a", True).over(window))\
        .withColumn("d_identidad", F.last("d_identidad", True).over(window)) 
    window = Window.partitionBy("d_identidad").orderBy(col("fecha").desc()).rowsBetween(Window.unboundedPreceding, 0)
    df_info_back=  df_info_back.withColumn("campana_siop_a", F.last("campana_siop_a", True).over(window)) \
        .withColumn("d_identidad", F.last("d_identidad", True).over(window)) 

    cols_to_lower = [
        'detalle_creacion', 'detalle_agenda', 'detalle_cierre', 'area', 
        'estado', 'subestado', 'usuario', 'tipo_de_entrega', 
        'estado_de_entrega', 'motivo_de_anulacion', 'estado_de_habilitacion', 
        'motivo_de_no_habilitacion'
    ]

    # aplica lower() a cada columna
    for c in cols_to_lower:
        df_info_back = df_info_back.withColumn(c, lower(col(c)))
        
    df_info_back = df_info_back.withColumn('marca',when(length(col('orden'))==9,'negocio').otherwise('nonegocio'))

    df_info_back = df_info_back.withColumn('tipo_producto',when(col('area').isin('activación de línea','portabilidad'),'voz').otherwise('no_aplica'))

    df_info_back = df_info_back.filter((col('marca')=='negocio')&(col('tipo_producto')=='voz'))
    df_info_back = df_info_back.repartition('fecha','d_identidad').persist()

    fecha_dt = datetime.strptime(fecha_fin, "%Y-%m-%d")
    fecha_inicio = fecha_dt.replace(day=1).strftime("%Y-%m-%d")
    # print(f'{fecha_inicio} inicio y {fecha_fin} fin ')
    
    df_info_back_creacion = df_info_back.filter(
        (F.col("detalle_creacion") == "aplica") &
        (F.col("fecha") <= fecha_fin) &
        (F.col("fecha") >= fecha_inicio)
    )

    agg_expr1 = [
        'fecha',
        'd_identidad',
        ]

    agg_expr2 = [
        F.first(F.col('campana_siop_a')).alias('campana_siop_info_back_creacion'),
        F.sum(F.col('cantidad_de_lineas')).alias('trz_brutas'),
        F.sum(F.when((F.col("tipo_de_entrega") == "express regular") , F.col("cantidad_de_lineas")).otherwise(0)).alias("trz_express_regular"),
        F.sum(F.when((F.col("tipo_de_entrega") == "express agendado") , F.col("cantidad_de_lineas")).otherwise(0)).alias("trz_express_agendado"),
        F.sum(F.when((F.col("tipo_de_entrega") == "retiro en tienda") , F.col("cantidad_de_lineas")).otherwise(0)).alias("trz_retiro_tienda"),
        F.sum(F.when((F.col("tipo_de_entrega") == "activacion") , F.col("cantidad_de_lineas")).otherwise(0)).alias("trz_activacion"),
        F.sum(F.when((F.col("tipo_de_entrega") == "brighstar") , F.col("cantidad_de_lineas")).otherwise(0)).alias("trz_brighstar"),
    ]

    df_info_back_creacion_resumen = df_info_back_creacion.groupBy(agg_expr1).agg(*agg_expr2)

    df_info_back_agenda = df_info_back.filter(
        (F.col("detalle_agenda") == "aplica") &
        (F.col("fecha_de_agenda") <= fecha_fin) &
        (F.col("fecha_de_agenda") >= fecha_inicio)
    )
    
    agg_expr1 = [
        col('fecha_de_agenda').alias('fecha'),
        'd_identidad',
        ]

    agg_expr2 = [
        F.first(F.col('campana_siop_a')).alias('campana_siop_info_back_agenda'),
        F.sum(F.col('cantidad_de_lineas')).alias('trz_programados'),
        F.sum(F.when(F.col("estado_de_entrega") == "reparto", F.col("cantidad_de_lineas")).otherwise(0)).alias("trz_reparto"),
        F.sum(F.when(F.col("estado_de_entrega") == "entregado", F.col("cantidad_de_lineas")).otherwise(0)).alias("trz_entregado"),
        F.sum(F.when(F.col("estado_de_entrega") == "anulado final", F.col("cantidad_de_lineas")).otherwise(0)).alias("trz_anulados"),
        F.sum(F.when((F.col("estado_de_entrega") == "entregado")&
                    (F.col("estado_de_habilitacion") == "habilitado"), F.col("cantidad_de_lineas")).otherwise(0)).alias("trz_habilitados"),
    ]

    df_info_back_agenda_resumen = df_info_back_agenda.groupBy(agg_expr1).agg(*agg_expr2)

    df_info_back_cierre = df_info_back.filter(
        (F.col("detalle_cierre") == "aplica") &
        (F.col("fecha_de_cierre") <= fecha_fin) &
        (F.col("fecha_de_cierre") >= fecha_inicio)
    )

    df_info_back=df_info_back.withColumn('hass',when((col("estado_de_entrega") == "entregado")&
                    (col("estado_de_habilitacion") == "habilitado"), col("cantidad_de_lineas")).otherwise(0))
    
    agg_expr1 = [
        col('fecha_de_cierre').alias('fecha'),
        'd_identidad',
        ]

    agg_expr2 = [
        F.first(F.col('campana_siop_a')).alias('campana_siop_info_back_cierre'),
        F.sum(F.when((F.col("estado_de_entrega") == "entregado")&
                    (F.col("estado_de_habilitacion") == "habilitado"), F.col("cantidad_de_lineas")).otherwise(0)).alias("trz_netas"),
    ]

    df_info_back_cierre_resumen = df_info_back_cierre.groupBy(agg_expr1).agg(*agg_expr2)


    # overwrite_table_SQL(df_info_back_resumen,'DB_temporal','djjjddl_dd','')
    query1=f"""
    select
    ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS indice_001
    ,fecha_gestion as fecha
    , dni as d_identidad
        ,case
        when producto='PORTABILIDAD' then 'PORTA SS-SS'
        else producto
    end as tipo_operacion
    ,punto_venta
    ,q_ventas as q_ventas_netas
    from DB_venta.dbo.resumen_hab_01
    where dni is not null
    and fecha_gestion between DATEADD(MONTH, DATEDIFF(MONTH, 0, '{fecha_fin}'), 0) and '{fecha_fin}'
    and punto_venta in(4822,5385,4824,4821,4820,5055,5226)
    """
    df_paya = obtener_tabla_sql(spark,'DB_tiempo',query1)     
    df_paya = df_paya.withColumn("d_identidad", regexp_replace(col("d_identidad"), r'^0+', ''))

    df_paya=df_paya.join(df_dota_dia,['fecha', 'd_identidad'],'outer')

    df_paya= df_paya.filter(col("tipo_operacion").isNotNull())

    window = Window.partitionBy("d_identidad").orderBy("fecha").rowsBetween(Window.unboundedPreceding, 0)
    df_paya=  df_paya.withColumn("campana_siop_a", F.last("campana_siop_a", True).over(window)) 
    window = Window.partitionBy("d_identidad").orderBy(col("fecha").desc()).rowsBetween(Window.unboundedPreceding, 0)
    df_paya=  df_paya.withColumn("campana_siop_a", F.last("campana_siop_a", True).over(window)) 
    df_paya= df_paya.filter(col("campana_siop_a").isNotNull())

    df_paya=df_paya.join(df_conversion_paya,['tipo_operacion'],'left')
    df_paya=df_paya.join(df_tarifa,['tipo_operacion','campana_siop_a'],'left')

    df_paya = df_paya.withColumn("q_ventas_netas", col("q_ventas_netas").cast(IntegerType())) \
                .withColumn("porc_conversion_paya", col("porc_conversion_paya").cast(DoubleType()))

    df_paya = df_paya.withColumn('campana_siop_a',when(col('campana_siop_a').isNull(),
                                                    when(col('punto_venta').isin(5055,5226),'PERFILADA')
                                                    .when(col('punto_venta').isin(4820),'RECUPERADOS')
                                                    .when(col('punto_venta').isin(4821),'IVR')
                                                    .when(col('punto_venta').isin(4824),'SEGUNDAS LINEAS')
                                                    .when(col('punto_venta').isin(4822,5385),'MIGRACIONES')
                                                    .otherwise('NO_APLICA'))
                                                .otherwise(col('campana_siop_a')))

    df_paya = df_paya.withColumn("factura_en_base_q_ventas_netas_paya",col("q_ventas_netas") * col("porc_conversion_paya") * col("tarifa") )

    df_paya = df_paya.withColumn(
        'q_ventas_paya_foco',
        when(
            col('campana_siop_a').contains('MIGRACIONES') & (col('tipo_operacion') == 'MIGRACION'),
            col('q_ventas_netas')
        ).when(
            (col('campana_siop_a').isin('PERFILADA','RECUPERADOS','IVR','SEGUNDAS LINEAS')) & col('tipo_operacion').contains('PORTA'),
            col('q_ventas_netas')
        ).when(
            (col('campana_siop_a')=='SEGUNDAS LINEAS') & (col('tipo_operacion')=='2DA LINEA'),
            col('q_ventas_netas')
        )
        .otherwise(0)
    )

    df_paya = df_paya.withColumn(
        'factura_en_base_q_ventas_netas_paya_foco',
        when(
            col('campana_siop_a').contains('MIGRACIONES') & (col('tipo_operacion') == 'MIGRACION'),
            col('factura_en_base_q_ventas_netas_paya')
        ).when(
            (col('campana_siop_a').isin('PERFILADA','RECUPERADOS','IVR','SEGUNDAS LINEAS')) & col('tipo_operacion').contains('PORTA'),
            col('factura_en_base_q_ventas_netas_paya')
        ).when(
            (col('campana_siop_a')=='SEGUNDAS LINEAS') & (col('tipo_operacion')=='2DA LINEA'),
            col('factura_en_base_q_ventas_netas_paya')
        )
        .otherwise(0)
    )

    df_paya = df_paya.withColumn('q_migracion_paya',when(col('tipo_operacion') == 'MIGRACION',col('q_ventas_netas')))
    df_paya = df_paya.withColumn('q_nueva_linea_paya',when(col('tipo_operacion') == 'NUEVA LINEA',col('q_ventas_netas')))
    df_paya = df_paya.withColumn('q_hogar_fibra_paya',when(col('tipo_operacion') == 'HOGAR FIBRA',col('q_ventas_netas')))
    df_paya = df_paya.withColumn('q_2da_linea_paya',when(col('tipo_operacion') == '2DA LINEA',col('q_ventas_netas')))
    df_paya = df_paya.withColumn('q_portas_paya',when((col('tipo_operacion') == 'PORTA PP-SS') |
                                            (col('tipo_operacion') == 'PORTA SS-SS'),col('q_ventas_netas')))

    df_paya = df_paya.withColumn('factura_migracion_paya',when(col('tipo_operacion') == 'MIGRACION',col('factura_en_base_q_ventas_netas_paya')))
    df_paya = df_paya.withColumn('factura_nueva_linea_paya',when(col('tipo_operacion') == 'NUEVA LINEA',col('factura_en_base_q_ventas_netas_paya')))
    df_paya = df_paya.withColumn('factura_hogar_fibra_paya',when(col('tipo_operacion') == 'HOGAR FIBRA',col('factura_en_base_q_ventas_netas_paya')))
    df_paya = df_paya.withColumn('factura_2da_linea_paya',when(col('tipo_operacion') == '2DA LINEA',col('factura_en_base_q_ventas_netas_paya')))
    df_paya = df_paya.withColumn('factura_portas_paya',when((col('tipo_operacion') == 'PORTA PP-SS') |
                                            (col('tipo_operacion') == 'PORTA SS-SS'),col('factura_en_base_q_ventas_netas_paya')))

    agg_expr1 = [
        'fecha',
        'd_identidad',
        ]
    agg_expr2 = [
        F.first(F.col('campana_siop_a')).alias('campana_siop_paya'),
        F.sum("q_ventas_netas").alias("q_ventas_netas"),
        F.sum("q_ventas_paya_foco").alias("q_ventas_paya_foco"),
        F.sum("q_migracion_paya").alias("q_migracion_paya"),
        F.sum("q_nueva_linea_paya").alias("q_nueva_linea_paya"),
        F.sum("q_hogar_fibra_paya").alias("q_hogar_fibra_paya"),
        F.sum("q_2da_linea_paya").alias("q_2da_linea_paya"),
        F.sum("q_portas_paya").alias("q_portas_paya"),
        
        F.sum("factura_en_base_q_ventas_netas_paya").alias("factura_en_base_q_ventas_netas_paya"),
        F.sum("factura_en_base_q_ventas_netas_paya_foco").alias("factura_en_base_q_ventas_netas_paya_foco"),
        F.sum("factura_migracion_paya").alias("factura_migracion_paya"),
        F.sum("factura_nueva_linea_paya").alias("factura_nueva_linea_paya"),
        F.sum("factura_hogar_fibra_paya").alias("factura_hogar_fibra_paya"),
        F.sum("factura_2da_linea_paya").alias("factura_2da_linea_paya"),
        F.sum("factura_portas_paya").alias("factura_portas_paya"),
        
    ]

    df_paya = df_paya.repartition('fecha','d_identidad')
    df_paya_resumen = df_paya.groupBy(agg_expr1).agg(*agg_expr2)

    query1=f"""
    select 
    distinct
    [FECHA HABILITACION] as fecha 
    ,[DNI ASESOR] as d_identidad
    ,sum(cast([PORTABILIDAD NETAS SIEBEL] as int)) over(partition by
    [FECHA HABILITACION],[DNI ASESOR]) as siebel
    ,sum(cast([PORTABILIDADES NETAS PAYA] as int)) over(partition by
    [FECHA HABILITACION],[DNI ASESOR]) as netas_paya
    from DB_venta.dbo.backoffice_hab_in
    where [FECHA HABILITACION] between DATEADD(MONTH, DATEDIFF(MONTH, 0, '{fecha_fin}'), 0) and '{fecha_fin}'
    """
    df_backoffice_hab_int=obtener_tabla_sql(spark,'DB_tiempo',query1)     
    df_backoffice_hab_int = df_backoffice_hab_int.withColumn("d_identidad", regexp_replace(col("d_identidad"), r'^0+', ''))

    df_backoffice_hab_int = df_backoffice_hab_int.withColumn("siebel", col("siebel").cast(IntegerType())) \
                .withColumn("netas_paya", col("netas_paya").cast(DoubleType()))

    df_backoffice_hab_int=df_backoffice_hab_int.join(df_dota_dia,['fecha', 'd_identidad'],'outer')

    window = Window.partitionBy("d_identidad").orderBy("fecha").rowsBetween(Window.unboundedPreceding, 0)
    df_backoffice_hab_int=  df_backoffice_hab_int.withColumn("campana_siop_a", F.last("campana_siop_a", True).over(window)) 
    window = Window.partitionBy("d_identidad").orderBy(col("fecha").desc()).rowsBetween(Window.unboundedPreceding, 0)
    df_backoffice_hab_int=  df_backoffice_hab_int.withColumn("campana_siop_a", F.last("campana_siop_a", True).over(window)) 

    query1=f"""
    select campana_siop_a,tc,ingreso_usd from DB_venta.dbo.tabla_costo_TC_ingreso_costo
    """
    df_ingreso = obtener_tabla_sql(spark,'DB_tiempo',query1)

    df_backoffice_hab_int=df_backoffice_hab_int.join(df_ingreso,['campana_siop_a'],'left')

    df_backoffice_hab_int = df_backoffice_hab_int.withColumn("ingreso_usd", col("ingreso_usd").cast(DoubleType())) 

    df_backoffice_hab_int=df_backoffice_hab_int.withColumn('ingreso_usd',col('siebel')*col('ingreso_usd'))

    agg_expr1 = [
        'fecha',
        'd_identidad',
        ]
    agg_expr2 = [
        F.sum("siebel").alias("siebel"),
        F.sum("netas_paya").alias("netas_paya"),
        F.sum("ingreso_usd").alias("ingreso_usd"),
    ]

    df_backoffice_hab_int = df_backoffice_hab_int.repartition('fecha','d_identidad')
    df_backoffice_hab_int_resumen = df_backoffice_hab_int.groupBy(agg_expr1).agg(*agg_expr2)


    query1=f"""
    select cast(fechainstalacion as date) as fecha, dni as d_identidad,producto as tipo_operacion from DB_venta.dbo.tb_venta_fibra_wdv
    where cast(fechainstalacion as date) between DATEADD(MONTH, DATEDIFF(MONTH, 0, '{fecha_fin}'), 0) and '{fecha_fin}'
    """
    df_wdv_fibra=obtener_tabla_sql(spark,'DB_venta',query1)     
    
    df_wdv_fibra = df_wdv_fibra.withColumn("d_identidad", regexp_replace(col("d_identidad"), r'^0+', ''))

    df_wdv_fibra=df_wdv_fibra.join(df_dota_dia,['fecha', 'd_identidad'],'outer')

    agg_expr1 = [
        'fecha',
        'd_identidad',
        ]
    agg_expr2 = [
        F.sum(F.when(F.col("tipo_operacion") == "Fibra", 1).otherwise(0)).alias("q_fibra_wdv"),
    ]

    df_wdv_fibra = df_wdv_fibra.repartition('fecha','d_identidad')
    df_wdv_fibra_resumen = df_wdv_fibra.groupBy(agg_expr1).agg(*agg_expr2)

    dfs1 = [df_wdv_resumen, df_paya_resumen,df_info_back_creacion_resumen,df_info_back_agenda_resumen,df_info_back_cierre_resumen, df_backoffice_hab_int_resumen,df_wdv_fibra_resumen]

    df_consolidado_ventas=  reduce(
        lambda df1, df2: df1.join(df2,['fecha','d_identidad'],"outer"),
        dfs1
    )

    valores_reemplazo = {
        'q_ventas':0, 'dentro':0, 'fuera':0, 'q_ventas_foco':0, 'q_migracion':0, 'q_nueva_linea':0, 'q_hogar_fibra':0, 'q_2da_linea':0, 'q_portas':0, 'q_ventas_netas_conversion':0, 'q_ventas_netas_conversion_foco':0, 'q_migracion_conv_01':0, 'q_nueva_linea_conv_01':0, 'q_hogar_fibra_conv_01':0, 'q_2da_linea_conv_01':0, 'q_portas_conv_01':0, 'q_ventas_netas_conversion_edu':0, 'q_ventas_netas_conversion_edu_foco':0, 'q_migracion_conv_edu':0, 'q_nueva_linea_conv_edu':0, 'q_hogar_fibra_conv_edu':0, 'q_2da_linea_conv_edu':0, 'q_portas_conv_edu':0, 'factura_en_base_q_ventas_netas_proyectadas':0, 'factura_en_base_q_ventas_netas_proyectadas_foco':0, 'factura_migracion_proyectadas':0, 'factura_nueva_linea_proyectadas':0, 'factura_hogar_fibra_proyectadas':0, 'factura_2da_linea_proyectadas':0, 'factura_portas_proyectadas':0, 'q_ventas_netas':0, 'q_ventas_paya_foco':0, 'q_migracion_paya':0, 'q_nueva_linea_paya':0, 'q_hogar_fibra_paya':0, 'q_2da_linea_paya':0, 'q_portas_paya':0, 'factura_en_base_q_ventas_netas_paya':0, 'factura_en_base_q_ventas_netas_paya_foco':0, 'factura_migracion_paya':0, 'factura_nueva_linea_paya':0, 'factura_hogar_fibra_paya':0, 'factura_2da_linea_paya':0, 'factura_portas_paya':0, 'siebel':0, 'netas_paya':0,'trz_brutas':0, 'trz_programados':0, 'trz_reparto':0, 'trz_entregado':0, 'trz_anulados':0, 'trz_habilitados':0, 'q_fibra_wdv':0,
        }
    return df_consolidado_ventas.fillna(valores_reemplazo)

    # overwrite_table_SQL(df_consolidado_ventas,'DB_a365','edu03_asistencia','edu03_asistencia')

def obtener_tiempos(spark,fecha_fin):
    query1 = f"""
        select 
        fecha
        ,usuario_genesys
        ,calling_list
        ,logged_in as t_total01
        ,idle as t_espera01
        ,off_queue as t_pausa01
        ,case 
        when logged_in-idle-off_queue <0 then 0
        else logged_in-idle-off_queue 
        end as t_operativo01
        from DB_tiempo.dbo.reporte_tiempo_01
        where calling_list in(
            'VAG_CL_OUT_ECO_A365_MIGRACIONES',
            'VAG_CL_OUT_ECO_A365_MOVIL_PORTA5',
            'VAG_CL_OUT_ECO_A365_PORTA_IVR',
            'VAG_CL_OUT_ECO_A365_PORTA_PERFILADA',
            'VAG_CL_OUT_ECO_A365_PORTA_RECUPERADOS',
            'VAG_CL_OUT_ECO_A365_SEGUNDAS_LINEAS',
            'VAG_CL_OUT_EMP_A365_HOGAR_FIBRA_CLIENTE',
            'VAG_CL_OUT_EMP_A365_HOGAR_FIBRA_NO_CLIENTE',
            'VAG_CL_OUT_ECO_A365_HOGAR_FIBRA_CLIENTE',
            'VAG_CL_OUT_ECO_A365_HOGAR_FIBRA_NOCLIENTE'
        )
        and fecha between DATEADD(MONTH, DATEDIFF(MONTH, 0, '{fecha_fin}'), 0) and '{fecha_fin}'
        and logged_in is not null
    """
    df_tiempo = obtener_tabla_sql(spark,"DB_CCR",query1)

    query1 = f"""
        select 
        distinct 
        cast(DNI as int) as d_identidad
        ,NOMBRES AS nombres_t01
        ,usuario_temp as usuario_genesys
        from DB_dota.dbo.tb_dota_diaria_siop
        where usuario_temp is not null
        and DNI is not null
    """
    df_dni = obtener_tabla_sql(spark,"DB_CCR",query1)

    df_tiempo=df_tiempo.join(df_dni,['usuario_genesys'],'left')
    agg_expr1 = [
        'fecha',
        'd_identidad',
        ]
    agg_expr2 = [
        F.sum("t_total01").alias("t_total01"),
        F.sum("t_espera01").alias("t_espera01"),
        F.sum("t_operativo01").alias("t_operativo01"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_MIGRACIONES", F.col("t_total01")).otherwise(0)).alias("t_total_migras1"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_MIGRACIONES", F.col("t_espera01")).otherwise(0)).alias("t_espera_migras1"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_MIGRACIONES", F.col("t_operativo01")).otherwise(0)).alias("t_operativo_migras1"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_MIGRACIONES2", F.col("t_total01")).otherwise(0)).alias("t_total_migras2"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_MIGRACIONES2", F.col("t_espera01")).otherwise(0)).alias("t_espera_migras2"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_MIGRACIONES2", F.col("t_operativo01")).otherwise(0)).alias("t_operativo_migras2"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_MIGRACIONES3", F.col("t_total01")).otherwise(0)).alias("t_total_migras3"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_MIGRACIONES3", F.col("t_espera01")).otherwise(0)).alias("t_espera_migras3"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_MIGRACIONES3", F.col("t_operativo01")).otherwise(0)).alias("t_operativo_migras3"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_MOVIL_PORTA5", F.col("t_total01")).otherwise(0)).alias("t_total_porta5"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_MOVIL_PORTA5", F.col("t_espera01")).otherwise(0)).alias("t_espera_porta5"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_MOVIL_PORTA5", F.col("t_operativo01")).otherwise(0)).alias("t_operativo_porta5"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_PORTA_IVR", F.col("t_total01")).otherwise(0)).alias("t_total_ivr"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_PORTA_IVR", F.col("t_espera01")).otherwise(0)).alias("t_espera_ivr"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_PORTA_IVR", F.col("t_operativo01")).otherwise(0)).alias("t_operativo_ivr"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_PORTA_PERFILADA", F.col("t_total01")).otherwise(0)).alias("t_total_perfilada"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_PORTA_PERFILADA", F.col("t_espera01")).otherwise(0)).alias("t_espera_perfilada"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_PORTA_PERFILADA", F.col("t_operativo01")).otherwise(0)).alias("t_operativo_perfilada"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_PORTA_RECUPERADOS", F.col("t_total01")).otherwise(0)).alias("t_total_recuperados"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_PORTA_RECUPERADOS", F.col("t_espera01")).otherwise(0)).alias("t_espera_recuperados"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_PORTA_RECUPERADOS", F.col("t_operativo01")).otherwise(0)).alias("t_operativo_recuperados"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_SEGUNDAS_LINEAS", F.col("t_total01")).otherwise(0)).alias("t_total_2da_lineas"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_SEGUNDAS_LINEAS", F.col("t_espera01")).otherwise(0)).alias("t_espera_2da_lineas"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_SEGUNDAS_LINEAS", F.col("t_operativo01")).otherwise(0)).alias("t_operativo_2da_lineas"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_EMP_A365_HOGAR_FIBRA_CLIENTE", F.col("t_total01")).otherwise(0)).alias("t_total_emp_fibra_cliente"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_EMP_A365_HOGAR_FIBRA_CLIENTE", F.col("t_espera01")).otherwise(0)).alias("t_espera_emp_fibra_cliente"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_EMP_A365_HOGAR_FIBRA_CLIENTE", F.col("t_operativo01")).otherwise(0)).alias("t_operativo_emp_fibra_cliente"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_EMP_A365_HOGAR_FIBRA_NO_CLIENTE", F.col("t_total01")).otherwise(0)).alias("t_total_emp_fibra_no_cliente"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_EMP_A365_HOGAR_FIBRA_NO_CLIENTE", F.col("t_espera01")).otherwise(0)).alias("t_espera_emp_fibra_no_cliente"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_EMP_A365_HOGAR_FIBRA_NO_CLIENTE", F.col("t_operativo01")).otherwise(0)).alias("t_operativo_emp_fibra_no_cliente"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_HOGAR_FIBRA_CLIENTE", F.col("t_total01")).otherwise(0)).alias("t_total_eco_fibra_cliente"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_HOGAR_FIBRA_CLIENTE", F.col("t_espera01")).otherwise(0)).alias("t_espera_eco_fibra_cliente"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_HOGAR_FIBRA_CLIENTE", F.col("t_operativo01")).otherwise(0)).alias("t_operativo_eco_fibra_cliente"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_HOGAR_FIBRA_NOCLIENTE", F.col("t_total01")).otherwise(0)).alias("t_total_eco_fibra_no_cliente"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_HOGAR_FIBRA_NOCLIENTE", F.col("t_espera01")).otherwise(0)).alias("t_espera_eco_fibra_no_cliente"),
        F.sum(F.when(F.col("calling_list") == "VAG_CL_OUT_ECO_A365_HOGAR_FIBRA_NOCLIENTE", F.col("t_operativo01")).otherwise(0)).alias("t_operativo_eco_fibra_no_cliente"),
        
    ]

    df_tiempo = df_tiempo.repartition('fecha','d_identidad')

    return df_tiempo.groupBy(agg_expr1).agg(*agg_expr2)

def transact_sql_delete_mes_actual_consolidado(fecha_fin):
    try:
        conexion = pyodbc.connect(conn_str)
        cursor = conexion.cursor()
        query = f"""
            delete from DB_A365.dbo.edu03_asistencia
            where fecha_ref_reporte =DATEADD(MONTH, DATEDIFF(MONTH, 0, '{fecha_fin}'), 0)
        """
        cursor.execute(query)
        conexion.commit()
        
        conexion.close()
    except Exception as e:
        (f"Error durante la inserción: {e}")
    finally:
        engine.dispose()

def obtener_contacto_agente(spark,fecha_fin):
    query1 = f"""
        select 
        distinct 
        fecha 
        ,dni as d_identidad
        ,estado_01
        ,SUM(unico_todo) over(partition by fecha 
        ,dni ) as q_reg
        from DB_A365.dbo.tb_wrapup_name
        where fecha='{fecha_fin}'
    """
    df_genesys_cloud = obtener_tabla_sql(spark,"DB_CCR",query1)
    
    agg_expr1 = [
        'fecha',
        'd_identidad',
        ]
    agg_expr2 = [
        F.sum(F.when((F.col("estado1") == "CONTACTO")&(F.col("tipo_discado") == "mes_actual"), F.col("q_discado")).otherwise(0)).alias("q_discado_contacto_actual"),
        F.sum(F.when((F.col("estado1") == "CONTACTO")&(F.col("tipo_discado") == "mes_anterior"), F.col("q_discado")).otherwise(0)).alias("q_discado_contacto_anterior"),
        F.sum(F.when((F.col("estado1") == "NO CONTACTO")&(F.col("tipo_discado") == "mes_actual"), F.col("q_discado")).otherwise(0)).alias("q_discado_no_contacto_actual"),
        F.sum(F.when((F.col("estado1") == "NO CONTACTO")&(F.col("tipo_discado") == "mes_anterior"), F.col("q_discado")).otherwise(0)).alias("q_discado_no_contacto_anterior"),
        F.sum(F.when((F.col("estado1") == "CONTACTO")&(F.col("tipo_discado") == "mes_actual"), F.col("q_mr_d")).otherwise(0)).alias("q_mr_d_contacto_actual"),
        F.sum(F.when((F.col("estado1") == "CONTACTO")&(F.col("tipo_discado") == "mes_anterior"), F.col("q_mr_d")).otherwise(0)).alias("q_mr_d_contacto_anterior"),
        F.sum(F.when((F.col("estado1") == "NO CONTACTO")&(F.col("tipo_discado") == "mes_actual"), F.col("q_mr_d")).otherwise(0)).alias("q_mr_d_no_contacto_actual"),
        F.sum(F.when((F.col("estado1") == "NO CONTACTO")&(F.col("tipo_discado") == "mes_anterior"), F.col("q_mr_d")).otherwise(0)).alias("q_mr_d_no_contacto_anterior"),
        # F.sum(F.when((F.col("estado1") == "CONTACTO")&(F.col("tipo_discado") == "mes_actual"), F.col("q_mr_m")).otherwise(0)).alias("q_mr_m_contacto_actual"),
        # F.sum(F.when((F.col("estado1") == "CONTACTO")&(F.col("tipo_discado") == "mes_anterior"), F.col("q_mr_m")).otherwise(0)).alias("q_mr_m_contacto_anterior"),
        # F.sum(F.when((F.col("estado1") == "NO CONTACTO")&(F.col("tipo_discado") == "mes_actual"), F.col("q_mr_m")).otherwise(0)).alias("q_mr_m_no_contacto_actual"),
        # F.sum(F.when((F.col("estado1") == "NO CONTACTO")&(F.col("tipo_discado") == "mes_anterior"), F.col("q_mr_m")).otherwise(0)).alias("q_mr_m_no_contacto_anterior"),
    ]

    df_ccr = df_ccr.repartition('fecha','d_identidad')

    return df_ccr.groupBy(agg_expr1).agg(*agg_expr2)


In [14]:
# def subir_ventas_outbound_funnel_back(spark):
from pyspark.sql import functions as F
    

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, trunc
from functools import reduce

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-12.8.1.jre8.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-12.8.1.jre8.jar') \
    .config('spark.executor.memory', '20g') \
    .config('spark.driver.memory', '20g') \
    .config('spark.sql.session.timeZone', 'UTC') \
    .config('spark.sql.legacy.timeParserPolicy', 'LEGACY') \
    .getOrCreate()

# fecha_fin='2025-07-31'
# fecha_paya='2025-07-31'
# fecha_backoffice='2025-07-31'


# fecha_fin='2025-08-31'
# fecha_paya='2025-08-31'
# fecha_backoffice='2025-08-31'


fecha_fin='2025-09-17'
fecha_paya='2025-09-10'
fecha_backoffice='2025-09-16'

transact_sql_delete_mes_actual_consolidado(fecha_fin)
consolidado_cargar_dota_reciente(spark,fecha_fin)

query1=f"""
select * from DB_A365.dbo.consolidado01_dota
"""
df_dota_a=obtener_tabla_sql(spark,'DB_tiempo',query1)
df_ventas_a=obtener_ventas(spark,fecha_fin)
# df_tarifa_conversion_costo_ingreso=tabla_tarifa_conversion_costo_ingreso(spark)

df_tiempos_a=obtener_tiempos(spark,fecha_fin)
df_contacto_a=obtener_contacto_agente(spark,fecha_fin)

# overwrite_table_SQL(df_contacto_a,'DB_temporal','dddd_Dadad_1','')

dfs3 = [df_dota_a, df_ventas_a, df_tiempos_a, df_contacto_a]

df_consolidado=  reduce(
    lambda df1, df2: df1.join(df2,['fecha','d_identidad'],"outer"),
    dfs3
)

df_consolidado=df_consolidado.withColumn('fecha_paya',lit(fecha_paya))
df_consolidado=df_consolidado.withColumn('fecha_backoffice',lit(fecha_backoffice))

df_consolidado=df_consolidado.withColumn('campana_siop_a',when(col('campana_siop_a').isNull(),'NO_APLICA').otherwise(col('campana_siop_a')))

query1=f"""
select
nombre_skill as campana_siop_a
,fecha
,1 as ref_fecha1
from DB_tiempo.dbo.fecha_no_aplica
"""
df_fecha_no_aplica=obtener_tabla_sql(spark,'DB_tiempo',query1)

df_fecha_no_aplica = df_fecha_no_aplica.withColumn('campana_siop_a',when(col('campana_siop_a').contains('MIGRA'),'MIGRACIONES')
                        .when(col('campana_siop_a')=='PORTABILIDAD_PERFILADA_1','PERFILADA')
                        .when(col('campana_siop_a')=='MOVIL_PORTA5','PERFILADA')
                        .when(col('campana_siop_a')=='PORTABILIDAD_PREPAGO_IVR' ,'IVR')
                        .when(col('campana_siop_a')=='PORTABILIDAD_RECUPERADOS','RECUPERADOS')
                        .when(col('campana_siop_a')=='SEGUNDAS_LINEAS','SEGUNDAS LINEAS')
                        .otherwise('NO__APLICA'))


df_consolidado=df_consolidado.join(df_fecha_no_aplica,['fecha','campana_siop_a'],'left')



df_consolidado = df_consolidado.withColumn(
    "dia_no_laborable",
    F.when(F.date_format("fecha", "E") == "Sun", F.lit(0))    # domingo
    .when(col('ref_fecha1').isNotNull(),F.lit(0))
    .otherwise(F.lit(1))                                     # demás días
).drop('ref_fecha1')

df_consolidado=df_consolidado.withColumn('campana_siop_a',when(col('tipo_gestion')=='OTROS',
            when(col('campana_siop_wdv').isNotNull(),col('campana_siop_wdv'))
            .when(col('campana_siop_info_back_creacion').isNotNull(),col('campana_siop_info_back_creacion'))
            .when(col('campana_siop_info_back_agenda').isNotNull(),col('campana_siop_info_back_agenda'))
            .when(col('campana_siop_info_back_cierre').isNotNull(),col('campana_siop_info_back_cierre'))
            .when(col('campana_siop_paya').isNotNull(),col('campana_siop_paya'))
            .otherwise('NO_APLICA'))
        .otherwise(col('campana_siop_a')))
                                                               
df_consolidado=df_consolidado.filter(col('campana_siop_a')!='NO_APLICA').drop('campana_siop_WDV','campana_siop_info_back','campana_siop_paya')


fecha_dt = datetime.strptime(fecha_fin, "%Y-%m-%d")
fecha_inicio = fecha_dt.replace(day=1).strftime("%Y-%m-%d")

query1 = f"""
select
distinct
nombre_skill
,n_sem as n_sem_carga 
,case
    when fecha_carga ='2024-11-02' then '2024-11-01'
    when fecha_carga ='2024-11-04' then '2024-11-01'
    when fecha_carga ='2024-12-02' then '2024-12-01'
    when fecha_carga ='2024-12-03' then '2024-12-01'
    when fecha_carga ='2024-12-04' then '2024-12-01'
    when fecha_carga ='2025-01-02' then '2025-01-01'
    when fecha_carga ='2025-02-03' then '2025-02-01'
    when fecha_carga ='2025-05-02' then '2025-05-01'
    when fecha_carga ='2025-06-02' then '2025-06-01'
    when fecha_carga ='2025-08-16' then '2025-08-15'
    when fecha_carga ='2025-08-18' then '2025-08-15'
    else fecha_carga
end as fecha
from DB_BaseSemanal.dbo.tb_listaBaseCargada
where fecha_carga between
DATEADD(DAY, 1 - DAY('{fecha_fin}'), '{fecha_fin}') 
and EOMONTH('{fecha_fin}')
and nombre_skill is not null
and nombre_skill<>'MOVIL_PORTA5'
"""
df_n_sem=obtener_tabla_sql(spark,'DB_BaseSemanal',query1)

fecha_inicio_1=datetime.strptime(fecha_inicio, "%Y-%m-%d")
fecha_fin_1=datetime.strptime(fecha_fin, "%Y-%m-%d")

df_fechas = (
    spark.range(1)
    .withColumn(
        "fecha",
        F.explode(
            F.sequence(
                F.to_date(F.lit(fecha_inicio)),  
                F.to_date(F.lit(fecha_fin)),     
                F.expr("interval 1 day")
            )
        )
    )
).drop('id')

df_nombre_skill=df_n_sem.select('nombre_skill').distinct()
df_nombre_skill_fecha=df_nombre_skill.crossJoin(df_fechas)
df_n_sem=df_n_sem.join(df_nombre_skill_fecha,['nombre_skill','fecha'],'right')

window = Window.partitionBy('nombre_skill').orderBy('fecha').rowsBetween(Window.unboundedPreceding, 0)
df_n_sem = df_n_sem.withColumn("n_sem_carga", F.last("n_sem_carga", True).over(window))

df_n_sem = df_n_sem.withColumn('campana_siop_a',when(col('nombre_skill').contains('MIGRA'),'MIGRACIONES_1')
                                                        .when(col('nombre_skill')=='PORTABILIDAD_PERFILADA_1','PERFILADA')
                                                        .when(col('nombre_skill')=='PORTABILIDAD_PREPAGO_IVR' ,'IVR')
                                                        .when(col('nombre_skill')=='PORTABILIDAD_RECUPERADOS','RECUPERADOS')
                                                        .when(col('nombre_skill')=='SEGUNDAS_LINEAS','SEGUNDAS LINEAS')
                                                        .otherwise('NO_APLICA')).drop('nombre_skill')

df_consolidado=df_consolidado.join(df_n_sem,['campana_siop_a','fecha'],'left')

from pyspark.sql.functions import col, dayofmonth, trunc, date_format, expr

df_consolidado = df_consolidado.withColumn("primer_dia_mes", trunc(col("fecha"), "month")) \
    .withColumn("dia_semana_inicio", date_format(col("primer_dia_mes"), "u").cast("int")) \
    .withColumn("dia_mes", dayofmonth(col("fecha"))) \
    .withColumn("n_sem", expr("int((dia_mes + dia_semana_inicio - 2) / 7) + 1")).drop('primer_dia_mes', 'dia_semana_inicio', 'dia_mes')

df_consolidado=df_consolidado.withColumn('n_sem_carga',when(col('n_sem_carga').isNull(),col('n_sem')).otherwise(col('n_sem_carga')))

df_consolidado = df_consolidado.withColumn(
    "trz_dia_venta",
    F.when(col('dia_no_laborable') == 0, F.lit(0)) 
    .when(F.date_format("fecha", "E") == "Sat", F.lit(0.8)) 
    .when(F.date_format("fecha", "E") == "Sun", F.lit(0)) 
    .otherwise(F.lit(1))
)

df_consolidado = df_consolidado.withColumn(
    "trz_dia_hab",
    F.when(col('dia_no_laborable') == 0, F.lit(0)) 
    .when(F.date_format("fecha", "E") == "Sat", F.lit(0))  
    .when(F.date_format("fecha", "E") == "Sun", F.lit(0))  
    .otherwise(F.lit(1))                                   
)

df_consolidado=df_consolidado.withColumn('fecha_ref_reporte',lit(fecha_inicio))

# window_spec = Window.partitionBy('fecha','d_identidad').orderBy(col("fecha").asc())
# df_consolidado = df_consolidado.withColumn("unico", row_number().over(window_spec))
# df_consolidado= df_consolidado.filter(col("unico") == 1).drop('unico')

df_consolidado = df_consolidado.withColumn(
    "tipo_de_cambio",
    F.when((F.col("fecha") >= "2025-07-01") & (F.col("fecha") <= "2025-07-31"), F.lit(3.652))
     .when((F.col("fecha") >= "2025-08-01") & (F.col("fecha") <= "2025-08-31"), F.lit(3.652))
     .when((F.col("fecha") >= "2025-09-01") & (F.col("fecha") <= "2025-09-30"), F.lit(3.652))
     .otherwise(F.lit(0))
)

df_consolidado = df_consolidado.withColumn('obs_01',when(col('calculo_FTE_ult')==1,'Dotacion actual')
                                        .otherwise('Dotacion anterior'))


query1 = f"""
select
distinct 
campana_siop_a
,dias_hab_mes
,sum(ventas_) over(partition by campana_siop_a) as ventas_
,sum(dias_hab_mes) over(partition by campana_siop_a) as dias_hab_mes
,sum(s1_dias) over(partition by campana_siop_a) as s1_dias
,sum(s2_dias) over(partition by campana_siop_a) as s2_dias
,sum(s3_dias) over(partition by campana_siop_a) as s3_dias
,sum(s4_dias) over(partition by campana_siop_a) as s4_dias
,sum(s5_dias) over(partition by campana_siop_a) as s5_dias
,1 as ref_unica
from DB_BaseSemanal.DBO.meta_out_mes 
where year(fecha_inicio_mes)=year('{fecha_fin}')
and month(fecha_inicio_mes)=month('{fecha_fin}')
"""
df_metas=obtener_tabla_sql(spark,'DB_BaseSemanal',query1)

window_spec = Window.partitionBy('campana_siop_a').orderBy(col("fecha").asc())
df_consolidado = df_consolidado.withColumn("ref_unica", row_number().over(window_spec))

df_consolidado=df_consolidado.join(df_metas,['campana_siop_a','ref_unica'],'left').drop('ref_unica')

window_spec = Window.partitionBy('fecha','campana_siop_a').orderBy(col("fecha").asc())
df_consolidado = df_consolidado.withColumn("unico_dia_hab", row_number().over(window_spec))

df_consolidado = df_consolidado.withColumn('unico_dia_transcurrido',when(col('unico_dia_hab')==1,col('dia_no_laborable'))
                                        .otherwise(0)).drop('unico_dia_hab')

# Export_list_base_sql(df_consolidado,'DB_A365','edu03_asistencia','consolidado_actualizado')


In [25]:
query1 = f"""
select
distinct 
campana_siop_a
,dias_hab_mes
,sum(ventas_) over(partition by campana_siop_a) as ventas_
,sum(dias_hab_mes) over(partition by campana_siop_a) as dias_hab_mes
,sum(s1_dias) over(partition by campana_siop_a) as s1_dias
,sum(s2_dias) over(partition by campana_siop_a) as s2_dias
,sum(s3_dias) over(partition by campana_siop_a) as s3_dias
,sum(s4_dias) over(partition by campana_siop_a) as s4_dias
,sum(s5_dias) over(partition by campana_siop_a) as s5_dias
,1 as ref_unica
from DB_BaseSemanal.DBO.meta_out_mes 
where year(fecha_inicio_mes)=year('{fecha_fin}')
and month(fecha_inicio_mes)=month('{fecha_fin}')
"""
df_metas=obtener_tabla_sql(spark,'DB_BaseSemanal',query1)

Py4JJavaError: An error occurred while calling o35413.jdbc.
: com.microsoft.sqlserver.jdbc.SQLServerException: The column 'dias_hab_mes' was specified multiple times for 'tmp'.
	at com.microsoft.sqlserver.jdbc.SQLServerException.makeFromDatabaseError(SQLServerException.java:278)
	at com.microsoft.sqlserver.jdbc.SQLServerStatement.getNextResult(SQLServerStatement.java:1787)
	at com.microsoft.sqlserver.jdbc.SQLServerPreparedStatement.doExecutePreparedStatement(SQLServerPreparedStatement.java:688)
	at com.microsoft.sqlserver.jdbc.SQLServerPreparedStatement$PrepStmtExecCmd.doExecute(SQLServerPreparedStatement.java:607)
	at com.microsoft.sqlserver.jdbc.TDSCommand.execute(IOBuffer.java:7745)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.executeCommand(SQLServerConnection.java:4706)
	at com.microsoft.sqlserver.jdbc.SQLServerStatement.executeCommand(SQLServerStatement.java:321)
	at com.microsoft.sqlserver.jdbc.SQLServerStatement.executeStatement(SQLServerStatement.java:253)
	at

In [ ]:
['campana_siop_a', 'fecha', 'd_identidad', 'duracion', 'estado_ref', 'ref_asistencia', 'debe_laborar', 'laborados', 'faltas', 'inactivo', 'total', 'contratados_dia', 'activos_dia', 'conectados_dia', 'dias_venta', 't_acu', 'p_acu', 'supervisor_siop', 'nombres_siop', 'campana_siop', 'entrada', 'salida', 'jornada', 'cargo', 'estado_siop', 'sub_estado', 'f_inicio', 'f_fin', 'grupo', 'subgerente', 'tipo_trabajo', 'usuario_siop', 'dni_operaciones', 'cargo_operaciones', 'f_inicio_super', 'contrato', 'estado', 'max_fecha', 'max_fecha_super', 'dias_antiguedad', 'dias_antiguedad_super', 'antiguedad', 'antiguedad_super', 'calculo_FTE_ult', 'supervisor_siop_ult', 'dni_operaciones_ult', 'cargo_operaciones_ult', 'cargo_ult', 'estado_siop_ult', 'campana_siop_a_ult', 'estado_ult', 'antiguedad_ult', 'antiguedad_super_ult', 'agente_unico_ult', 'tc', 'costo_soles', 'tipo_gestion', 'tipo_gestion_ult', 'q_ventas', 'dentro', 'fuera', 's1_foco', 'q_migracion', 'q_nueva_linea', 'q_hogar_fibra', 'q_2da_linea', 'q_portas', 'q_ventas_netas_conversion', 'q_ventas_netas_conversion_foco', 'q_migracion_conv_01', 'q_nueva_linea_conv_01', 'q_hogar_fibra_conv_01', 'q_2da_linea_conv_01', 'q_portas_conv_01', 'q_ventas_netas_conversion_edu', 'q_ventas_netas_conversion_edu_foco', 'q_migracion_conv_edu', 'q_nueva_linea_conv_edu', 'q_hogar_fibra_conv_edu', 'q_2da_linea_conv_edu', 'q_portas_conv_edu', 'factura_en_base_q_ventas_netas_proyectadas', 'factura_en_base_q_ventas_netas_proyectadas_foco', 'factura_migracion_proyectadas', 'factura_nueva_linea_proyectadas', 'factura_hogar_fibra_proyectadas', 'factura_2da_linea_proyectadas', 'factura_portas_proyectadas', 'q_ventas_netas', 'q_ventas_paya_foco', 'q_migracion_paya', 'q_nueva_linea_paya', 'q_hogar_fibra_paya', 'q_2da_linea_paya', 'q_portas_paya', 'factura_en_base_q_ventas_netas_paya', 'factura_en_base_q_ventas_netas_paya_foco', 'factura_migracion_paya', 'factura_nueva_linea_paya', 'factura_hogar_fibra_paya', 'factura_2da_linea_paya', 'factura_portas_paya', 'campana_siop_info_back_creacion', 'trz_brutas', 'trz_express_regular', 'trz_express_agendado', 'trz_retiro_tienda', 'trz_activacion', 'trz_brighstar', 'campana_siop_info_back_agenda', 'trz_programados', 'trz_reparto', 'trz_entregado', 'trz_anulados', 'trz_habilitados', 'campana_siop_info_back_cierre', 'trz_netas', 'siebel', 'netas_paya', 'ingreso_usd', 't_total01', 't_espera01', 't_operativo01', 't_total_migras1', 't_espera_migras1', 't_operativo_migras1', 't_total_migras2', 't_espera_migras2', 't_operativo_migras2', 't_total_migras3', 't_espera_migras3', 't_operativo_migras3', 't_total_porta5', 't_espera_porta5', 't_operativo_porta5', 't_total_ivr', 't_espera_ivr', 't_operativo_ivr', 't_total_perfilada', 't_espera_perfilada', 't_operativo_perfilada', 't_total_recuperados', 't_espera_recuperados', 't_operativo_recuperados', 't_total_2da_lineas', 't_espera_2da_lineas', 't_operativo_2da_lineas', 't_total_emp_fibra_cliente', 't_espera_emp_fibra_cliente', 't_operativo_emp_fibra_cliente', 't_total_emp_fibra_no_cliente', 't_espera_emp_fibra_no_cliente', 't_operativo_emp_fibra_no_cliente', 't_total_eco_fibra_cliente', 't_espera_eco_fibra_cliente', 't_operativo_eco_fibra_cliente', 't_total_eco_fibra_no_cliente', 't_espera_eco_fibra_no_cliente', 't_operativo_eco_fibra_no_cliente', 'q_discado_contacto_actual', 'q_discado_contacto_anterior', 'q_discado_no_contacto_actual', 'q_discado_no_contacto_anterior', 'q_mr_d_contacto_actual', 'q_mr_d_contacto_anterior', 'q_mr_d_no_contacto_actual', 'q_mr_d_no_contacto_anterior', 'fecha_paya', 'fecha_backoffice', 'dia_no_laborable', 'n_sem_carga', 'n_sem', 'trz_dia_venta', 'trz_dia_hab', 'fecha_ref_reporte', 'tipo_de_cambio', 'obs_01', 'dias_hab_mes', 'ventas_', 'unico_dia_transcurrido']s1_

In [22]:
df_consolidado.filter(col('ventas_').isNotNull()).show()

+---------------+----------+-----------+--------+----------+--------------+------------+---------+------+--------+-----+---------------+-----------+--------------+----------+-----+-----+--------------------+--------------------+--------------------+--------+--------+--------+------------+-----------+--------------------+----------+----------+---------------+---------------+------------+--------------+---------------+--------------------+--------------+----------+------+----------+---------------+---------------+---------------------+---------------+----------------+---------------+--------------------+-------------------+---------------------+------------+---------------+------------------+----------+---------------+--------------------+----------------+-----+-----------+------------+----------------+--------+------+-----+-------------+-----------+-------------+-------------+-----------+--------+-------------------------+------------------------------+-------------------+--------------

In [ ]:
df_nulos=df_consolidado_ventas.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in df_consolidado_ventas.columns
])
# overwrite_table_SQL(df_nulos,'DB_venta','temp_nulos','tabla temp_nulos')

print(df_nulos.show(truncate=False, vertical=False))

+-----+-----------+--------+------+-----+-------------+-----------+-------------+-------------+-----------+--------+-------------------------+------------------------------+-------------------+---------------------+---------------------+-------------------+----------------+-----------------------------+----------------------------------+--------------------+----------------------+----------------------+--------------------+-----------------+------------------------------------------+-----------------------------------------------+-----------------------------+-------------------------------+-------------------------------+-----------------------------+--------------------------+--------------+------------------+----------------+------------------+------------------+----------------+-------------+-----------------------------------+----------------------------------------+----------------------+------------------------+------------------------+----------------------+-------------------+

base


In [153]:
query1=f"""
select * from DB_A365.dbo.consolidado02_base
"""
df_resumen_01=obtener_tabla_sql(spark,'DB_tiempo',query1)
print(df_resumen_01.columns)

['campana_siop_a', 'fecha', 'd_identidad', 'duracion', 'estado_ref', 'ref_asistencia', 'debe_laborar', 'laborados', 'faltas', 'inactivo', 'total', 'contratados_dia', 'activos_dia', 'conectados_dia', 'dias_venta', 't_acu', 'p_acu', 'supervisor_siop', 'nombres_siop', 'campana_siop', 'entrada', 'salida', 'jornada', 'cargo', 'estado_siop', 'sub_estado', 'f_inicio', 'f_fin', 'grupo', 'subgerente', 'tipo_trabajo', 'usuario_siop', 'dni_operaciones', 'cargo_operaciones', 'contrato', 'estado', 'max_fecha', 'dias_antiguedad', 'antiguedad', 'calculo_FTE_ult', 'supervisor_siop_ult', 'dni_operaciones_ult', 'cargo_operaciones_ult', 'cargo_ult', 'estado_siop_ult', 'antiguedad_ult', 'agente_unico_ult', 'hogar_fibra', '2da_linea', 'renovacion_equipo', 'nueva_linea', 'porta_pp_ss', 'porta_ss_ss', 'migracion', 'dentro', 'fuera', 'q_ventas', 'hogar_fibra_paya', '2da_linea_paya', 'renovacion_equipo_paya', 'nueva_linea_paya', 'porta_pp_ss_paya', 'porta_ss_ss_paya', 'migracion_paya', 'q_ventas_netas', 'porta

In [ ]:
['campana_siop_a', 'fecha', 'd_identidad', 'duracion', 'estado_ref', 'ref_asistencia', 'debe_laborar', 'laborados', 'faltas', 'inactivo', 'total', 'contratados_dia', 'activos_dia', 'conectados_dia', 'dias_venta', 't_acu', 'p_acu', 'supervisor_siop', 'nombres_siop', 'campana_siop', 'entrada', 'salida', 'jornada', 'cargo', 'estado_siop', 'sub_estado', 'f_inicio', 'f_fin', 'grupo', 'subgerente', 'tipo_trabajo', 'usuario_siop', 'dni_operaciones', 'cargo_operaciones', 'contrato', 'estado', 'max_fecha', 'dias_antiguedad', 'antiguedad', 'calculo_FTE_ult', 'supervisor_siop_ult', 'dni_operaciones_ult', 'cargo_operaciones_ult', 'cargo_ult', 'estado_siop_ult', 'antiguedad_ult', 'agente_unico_ult', 'hogar_fibra', '2da_linea', 'renovacion_equipo', 'nueva_linea', 'porta_pp_ss', 'porta_ss_ss', 'migracion', 'dentro', 'fuera', 'q_ventas', 'hogar_fibra_paya', '2da_linea_paya', 'renovacion_equipo_paya', 'nueva_linea_paya', 'porta_pp_ss_paya', 'porta_ss_ss_paya', 'migracion_paya', 'q_ventas_netas', 'porta_siebel', 't_total01', 't_espera01', 't_pausa01', 't_operativo01', 'tar_2da_linea', 'tar_hogar_fibra', 'tar_migracion', 'tar_nueva_linea', 'tar_porta_pp-ss', 'tar_porta_ss-ss', 'conv_2da_linea', 'conv_hogar_fibra', 'conv_hogar_inalambrico', 'conv_migracion', 'conv_nueva_linea', 'conv_porta_pp-ss', 'conv_porta_ss-ss', 'conv_renovacion_de_equipo', 'conv_edu2da_linea', 'conv_eduhogar_fibra', 'conv_eduhogar_inalambrico', 'conv_edumigracion', 'conv_edunueva_linea', 'conv_eduporta_pp-ss', 'conv_eduporta_ss-ss', 'conv_edurenovacion_de_equipo', 'conv_paya2da_linea', 'conv_payahogar_fibra', 'conv_payahogar_inalambrico', 'conv_payamigracion', 'conv_payanueva_linea', 'conv_payaporta_pp-ss', 'conv_payaporta_ss-ss', 'conv_payarenovacion_de_equipo', 'costo_soles', 'tc', 'ingreso_usd', 'fecha_paya', 'fecha_backoffice', 'por_conversion_dia_semana']

46342942
74644701

ingreso el 26 ago

campana no aaplica

['campana_siop_a', 'fecha', 'd_identidad', 'duracion', 'estado_ref', 'ref_asistencia', 'debe_laborar', 'laborados', 'faltas', 'inactivo', 'total', 'contratados_dia', 'activos_dia', 'conectados_dia', 'dias_venta', 't_acu', 'p_acu', 'supervisor_siop', 'nombres_siop', 'campana_siop', 'entrada', 'salida', 'jornada', 'cargo', 'estado_siop', 'sub_estado', 'f_inicio', 'f_fin', 'grupo', 'subgerente', 'tipo_trabajo', 'usuario_siop', 'dni_operaciones', 'cargo_operaciones', 'contrato', 'estado', 'max_fecha', 'dias_antiguedad', 'antiguedad', 'calculo_FTE_ult', 'supervisor_siop_ult', 'dni_operaciones_ult', 'cargo_operaciones_ult', 'cargo_ult', 'estado_siop_ult', 'antiguedad_ult', 'agente_unico_ult', 'hogar_fibra', '2da_linea', 'renovacion_equipo', 'nueva_linea', 'porta_pp_ss', 'porta_ss_ss', 'migracion', 'dentro', 'fuera', 'q_ventas', 'hogar_fibra_paya', '2da_linea_paya', 'renovacion_equipo_paya', 'nueva_linea_paya', 'porta_pp_ss_paya', 'porta_ss_ss_paya', 'migracion_paya', 'q_ventas_netas', 'porta

+--------------+-----+-----------+--------+----------+--------------+------------+---------+------+--------+-----+---------------+-----------+--------------+----------+-----+-----+---------------+------------+------------+-------+------+-------+-----+-----------+----------+--------+-----+-----+----------+------------+------------+---------------+-----------------+--------+------+---------+---------------+----------+---------------+-------------------+-------------------+---------------------+---------+---------------+--------------+----------------+-----------+---------+-----------------+-----------+-----------+-----------+---------+------+-----+--------+----------------+--------------+----------------------+----------------+----------------+----------------+--------------+--------------+------------+---------+----------+---------+-------------+-------------+---------------+-------------+---------------+---------------+---------------+--------------+----------------+-------------------

In [92]:
query1=f"""
select * from DB_A365.dbo.consolidado01_dota
"""
df_aa=obtener_tabla_sql(spark,'DB_tiempo',query1)

df_nulos=df_aa.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in df_aa.columns
])
# overwrite_table_SQL(df_nulos,'DB_venta','temp_nulos','tabla temp_nulos')
print(df_nulos.show(truncate=False, vertical=False))




+--------------+-----------+-----+--------+----------+--------------+------------+---------+------+--------+-----+---------------+-----------+--------------+----------+-----+-----+---------------+------------+------------+-------+------+-------+-----+-----------+----------+--------+-----+-----+----------+------------+------------+---------------+-----------------+--------+------+---------+---------------+----------+---------------+-------------------+-------------------+---------------------+---------+---------------+--------------+----------------+---+-----------+
|campana_siop_a|d_identidad|fecha|duracion|estado_ref|ref_asistencia|debe_laborar|laborados|faltas|inactivo|total|contratados_dia|activos_dia|conectados_dia|dias_venta|t_acu|p_acu|supervisor_siop|nombres_siop|campana_siop|entrada|salida|jornada|cargo|estado_siop|sub_estado|f_inicio|f_fin|grupo|subgerente|tipo_trabajo|usuario_siop|dni_operaciones|cargo_operaciones|contrato|estado|max_fecha|dias_antiguedad|antiguedad|calculo_F

In [ ]:
print(df_dota_a.columns)
print(df_ventas_a.columns)
print(df_tiempos_a.columns)
print(df_tarifa_conversion_costo_ingreso.columns)



['d_identidad', 'fecha', 'duracion', 'estado_ref', 'ref_asistencia', 'debe_laborar', 'laborados', 'faltas', 'inactivo', 'total', 'contratados_dia', 'activos_dia', 'conectados_dia', 'dias_venta', 't_acu', 'p_acu', 'supervisor_siop', 'nombres_siop', 'campana_siop', 'entrada', 'salida', 'jornada', 'cargo', 'estado_siop', 'sub_estado', 'f_inicio', 'f_fin', 'grupo', 'subgerente', 'tipo_trabajo', 'usuario_siop', 'dni_operaciones', 'cargo_operaciones', 'contrato', 'estado', 'campana_siop_a', 'max_fecha', 'dias_antiguedad', 'antiguedad', 'calculo_FTE_ult', 'supervisor_siop_ult', 'dni_operaciones_ult', 'cargo_operaciones_ult', 'cargo_ult', 'estado_siop_ult', 'antiguedad_ult', 'agente_unico_ult']
['fecha', 'd_identidad', 'hogar_fibra', '2da_linea', 'renovacion_equipo', 'nueva_linea', 'porta_pp_ss', 'porta_ss_ss', 'migracion', 'dentro', 'fuera', 'q_ventas', 'hogar_fibra_paya', '2da_linea_paya', 'renovacion_equipo_paya', 'nueva_linea_paya', 'porta_pp_ss_paya', 'porta_ss_ss_paya', 'migracion_paya',

In [37]:
df_dnisdd.filter(col('usuario_genesys').isNull()).show()

+---------------+-----+---------+----------+---------+-------------+-----------+
|usuario_genesys|fecha|t_total01|t_espera01|t_pausa01|t_operativo01|d_identidad|
+---------------+-----+---------+----------+---------+-------------+-----------+
+---------------+-----+---------+----------+---------+-------------+-----------+



In [38]:
df_dnisdd.filter(col('d_identidad').isNull()).show()


+---------------+----------+---------+----------+---------+-------------+-----------+
|usuario_genesys|     fecha|t_total01|t_espera01|t_pausa01|t_operativo01|d_identidad|
+---------------+----------+---------+----------+---------+-------------+-----------+
|     aap_aalejo|2025-08-26|   51.767|       0.0|   51.767|          0.0|       NULL|
|  aap_bherencia|2025-08-26|   97.546|       0.0|   97.546|          0.0|       NULL|
|    aap_clagosa|2025-08-26|  839.646|       0.0|  839.646|          0.0|       NULL|
|   aap_ralvites|2025-08-26|   71.966|       0.0|   71.966|          0.0|       NULL|
|  aap_szaldivar|2025-08-26| 2718.483|       0.0| 2718.483|          0.0|       NULL|
|    aap_isinisi|2025-08-26|   58.028|       0.0|   58.028|          0.0|       NULL|
+---------------+----------+---------+----------+---------+-------------+-----------+



In [ ]:
valores_reemplazo = {
    'hogar_fibra':0,
    '2da_linea':0,
    'renovacion_equipo':0,
    'nueva_linea':0,
    'porta_pp_ss':0,
    'porta_ss_ss':0,
    'migracion':0,
    'dentro':0,
    'fuera':0,
    'q_ventas':0,
    'hogar_fibra_paya':0,
    '2da_linea_paya':0,
    'renovacion_equipo_paya':0,
    'nueva_linea_paya':0,
    'porta_pp_ss_paya':0,
    'porta_ss_ss_paya':0,
    'migracion_paya':0,
    'q_ventas_netas':0,
    'porta_siebel':0
    }
df_tot = df_tot.fillna(valores_reemplazo)

In [ ]:
valores_reemplazo = {
    'hogar_fibra': 0,
    '2da_linea': 0,
    'renovacion_equipo': 0,
    'nueva_linea': 0,
    'porta_pp_ss': 0,
    'porta_ss_ss': 0,
    'migracion': 0,
    'q_ventas': 0,
    'q_ventas_proyectadas': 0,
    'fuera': 0,
    'dentro': 0,
    'brutas_a_la_fecha': 0,
    'ingresos_actuales': 0,
    'ingresos_proyectados': 0,
    'brutas_a_la_fecha_recepcion': 0,
    'netas_desde_fecha_recepcion': 0,
    'ingresos_actuales_desde_fecha_recepcion': 0,
    'ingresos_proyectados_desde_fecha_recepcion': 0,
    't_total01': 0,
    't_espera01': 0,
    't_pausa01': 0,
    't_operativo01': 0,
    }
df_tot = df_tot.fillna(valores_reemplazo)
df_tot=df_tot.persist()


In [34]:

df_nulos=df_tiempos_a.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in df_tiempos_a.columns
])
# overwrite_table_SQL(df_nulos,'DB_venta','temp_nulos','tabla temp_nulos')
print(df_nulos.show(truncate=False, vertical=False))

+-----+---------+----------+---------+-------------+-----------+
|fecha|t_total01|t_espera01|t_pausa01|t_operativo01|d_identidad|
+-----+---------+----------+---------+-------------+-----------+
|0    |0        |0         |0        |0            |6          |
+-----+---------+----------+---------+-------------+-----------+

None


In [69]:
overwrite_table_SQL(df_paya,'DB_temporal','holorraresdfdsfds___bodorarr_q','')


In [ ]:
dddd

In [ ]:

window_spec = Window.partitionBy("indice_001")
df_wdv = df_wdv.withColumn(
    "q_ventas_proyectadas",
    sum("q_ventas_proyectadas").over(window_spec)
)
window_spec = Window.partitionBy("indice_001")
df_wdv = df_wdv.withColumn(
    "valor_proyectado_dolar",
    sum("valor_proyectado_dolar").over(window_spec)
)

window_spec = Window.partitionBy('indice_001').orderBy(col("indice_001").asc())
df_wdv = df_wdv.withColumn("unico", row_number().over(window_spec))
df_wdv = df_wdv.filter(col("unico")==1).drop('unico')



from pyspark.sql import functions as F

df_wdv = df_wdv.withColumn(
    "brutas_a_la_fecha",
    when(
        col('campana_siop_a').contains('MIGRACIONES') & (col('tipo_operacion') == 'MIGRACION'),
        col('q_ventas')
    ).when(
        (col('campana_siop_a').isin('PERFILADA','RECUPERADOS','IVR','SEGUNDAS LINEAS')) & col('tipo_operacion').contains('PORTA'),
        col('q_ventas')
    ).when(
        (col('campana_siop_a')=='SEGUNDAS LINEAS') & (col('tipo_operacion')=='2DA LINEA'),
        col('q_ventas')
    )
    .otherwise(0)
)

df_wdv = df_wdv.withColumn(
    'netas_a_la_fecha',
    when(
        col('campana_siop_a').contains('MIGRACIONES') & (col('tipo_operacion') == 'MIGRACION'),
        col('q_ventas_proyectadas')
    ).when(
        (col('campana_siop_a').isin('PERFILADA','RECUPERADOS','IVR','SEGUNDAS LINEAS')) & col('tipo_operacion').contains('PORTA'),
        col('q_ventas_proyectadas')
    ).when(
        (col('campana_siop_a')=='SEGUNDAS LINEAS') & (col('tipo_operacion')=='2DA LINEA'),
        col('q_ventas_proyectadas')
    )
    .otherwise(0)
)


agg_expr1 = [
     col('fecha_registro').alias('fecha'),
     col('num_doc_usuario_reg_linea').alias('d_identidad'),
    ]

agg_expr2 = [
    F.sum(F.when(F.col("tipo_operacion") == 'HOGAR FIBRA', F.col("q_ventas")).otherwise(0)).alias('hogar_fibra'),
    F.sum(F.when(F.col("tipo_operacion") == '2DA LINEA', F.col("q_ventas")).otherwise(0)).alias('2da_linea'),
    F.sum(F.when(F.col("tipo_operacion") == 'RENOVACION DE EQUIPO', F.col("q_ventas")).otherwise(0)).alias('renovacion_equipo'),
    F.sum(F.when(F.col("tipo_operacion") == 'NUEVA LINEA', F.col("q_ventas")).otherwise(0)).alias('nueva_linea'),
    F.sum(F.when(F.col("tipo_operacion") == 'PORTA PP-SS', F.col("q_ventas")).otherwise(0)).alias('porta_pp_ss'),
    F.sum(F.when(F.col("tipo_operacion") == 'PORTA SS-SS', F.col("q_ventas")).otherwise(0)).alias('porta_ss_ss'),
    F.sum(F.when(F.col("tipo_operacion") == 'MIGRACION', F.col("q_ventas")).otherwise(0)).alias('migracion'),
    F.sum("q_ventas").alias("q_ventas"),
    F.sum("q_ventas_proyectadas").alias("q_ventas_proyectadas"),
    F.sum("dentro").alias("dentro"),
    F.sum("fuera").alias("fuera"),
    F.sum("brutas_a_la_fecha").alias("brutas_a_la_fecha"),
    F.sum("netas_a_la_fecha").alias("netas_a_la_fecha"),
    F.sum("valor_proyectado_dolar").alias("ingresos_actuales"),
]


df_wdv = df_wdv.repartition('fecha_registro','num_doc_usuario_reg_linea')
df_wdv_resumen = df_wdv.groupBy(agg_expr1).agg(*agg_expr2)

df_wdv_resumen = df_wdv_resumen.withColumn("ingresos_proyectados",col("ingresos_actuales") * 9.6 * 25)

df_wdv_resumen = df_wdv_resumen.withColumn(
    'brutas_a_la_fecha_recepcion',when(col('fecha')>lit(fecha_paya),0)
    .otherwise(col('brutas_a_la_fecha'))
)

df_wdv_resumen = df_wdv_resumen.withColumn(
    'netas_desde_fecha_recepcion',when(col('fecha')<=lit(fecha_paya),0)
    .otherwise(col('netas_a_la_fecha'))
)
df_wdv_resumen = df_wdv_resumen.withColumn(
    'ingresos_actuales_desde_fecha_recepcion',when(col('fecha')<=lit(fecha_paya),0)
    .otherwise(col('ingresos_actuales'))
)
df_wdv_resumen = df_wdv_resumen.withColumn(
    'ingresos_proyectados_desde_fecha_recepcion',when(col('fecha')<=lit(fecha_paya),0)
    .otherwise(col('ingresos_proyectados'))
)


In [ ]:
['fecha', 'd_identidad', 'hogar_fibra', '2da_linea', 'renovacion_equipo', 'nueva_linea', 'porta_pp_ss', 'porta_ss_ss', 'migracion', 'q_ventas', 'q_ventas_proyectadas', 'dentro', 'fuera', 'brutas_a_la_fecha', 'netas_a_la_fecha', 'ingresos_actuales', 'ingresos_proyectados', 'brutas_a_la_fecha_recepcion', 'netas_desde_fecha_recepcion', 'ingresos_actuales_desde_fecha_recepcion', 'ingresos_proyectados_desde_fecha_recepcion']brutavruta

['fecha', 'd_identidad', 'hogar_fibra', '2da_linea', 'renovacion_equipo', 'nueva_linea', 'porta_pp_ss', 'porta_ss_ss', 'migracion', 'q_ventas', 'q_ventas_proyectadas', 'dentro', 'fuera', 'brutas_a_la_fecha', 'netas_a_la_fecha', 'ingresos_actuales', 'ingresos_proyectados', 'brutas_a_la_fecha_recepcion', 'netas_desde_fecha_recepcion', 'ingresos_actuales_desde_fecha_recepcion', 'ingresos_proyectados_desde_fecha_recepcion']


In [292]:
overwrite_table_SQL(df_wdv_resumen,'DB_temporal','dddd_borasdasdrar','')

In [ ]:
df_wdv_resumen.select(F.sum("otros").alias("suma_otros")).show()

In [ ]:

window_spec = Window.partitionBy('fecha','d_identidad').orderBy(F.col("fecha").asc())
df_tot = df_tot.withColumn("unico", row_number().over(window_spec))
# df_tot= df_tot.filter(col("unico") == 1).drop('unico')
df_tot= df_tot.filter(col("duracion").isNotNull())

In [252]:
valores_reemplazo = {
    'hogar_fibra': 0,
    '2da_linea': 0,
    'renovacion_equipo': 0,
    'nueva_linea': 0,
    'porta_pp_ss': 0,
    'porta_ss_ss': 0,
    'migracion': 0,
    'q_ventas': 0,
    'q_ventas_proyectadas': 0,
    'fuera': 0,
    'dentro': 0,
    'brutas_a_la_fecha': 0,
    'ingresos_actuales': 0,
    'ingresos_proyectados': 0,
    'brutas_a_la_fecha_recepcion': 0,
    'netas_desde_fecha_recepcion': 0,
    'ingresos_actuales_desde_fecha_recepcion': 0,
    'ingresos_proyectados_desde_fecha_recepcion': 0,
    't_total01': 0,
    't_espera01': 0,
    't_pausa01': 0,
    't_operativo01': 0,
    }
df_tot = df_tot.fillna(valores_reemplazo)
df_tot=df_tot.persist()


In [ ]:
ss

In [ ]:
paya
df_paya = df_paya.withColumn(
    'netas_a_la_fecha_recepcion',
    when(
        col('campana_siop_a').contains('MIGRACIONES') & (col('tipo_operacion') == 'MIGRACION'),
        col('q_ventas_netas')
    ).when(
        (col('campana_siop_a').isin('PERFILADA','RECUPERADOS','IVR','SEGUNDAS LINEAS')) & col('tipo_operacion').contains('PORTA'),
        col('q_ventas_netas')
    ).when(
        (col('campana_siop_a')=='SEGUNDAS LINEAS') & (col('tipo_operacion')=='2DA LINEA'),
        col('q_ventas_netas')
    )
    .otherwise(0)
)





window_spec = Window.partitionBy("indice_001")
df_paya = df_paya.withColumn(
    "ingresos_reales_paya",
    sum("ingresos_reales_paya").over(window_spec)
)

window_spec = Window.partitionBy('indice_001').orderBy(col("indice_001").asc())
df_paya = df_paya.withColumn("unico", row_number().over(window_spec))
df_paya = df_paya.filter(col("unico")==1)

agg_expr1 = [
     'fecha',
     'd_identidad',
    ]

agg_expr2 = [
    F.sum(F.when(F.col("tipo_operacion") == 'HOGAR FIBRA', F.col("q_ventas_netas")).otherwise(0)).alias('hogar_fibra_paya'),
    F.sum(F.when(F.col("tipo_operacion") == '2DA LINEA', F.col("q_ventas_netas")).otherwise(0)).alias('2da_linea_paya'),
    F.sum(F.when(F.col("tipo_operacion") == 'RENOVACION DE EQUIPO', F.col("q_ventas_netas")).otherwise(0)).alias('renovacion_equipo_paya'),
    F.sum(F.when(F.col("tipo_operacion") == 'NUEVA LINEA', F.col("q_ventas_netas")).otherwise(0)).alias('nueva_linea_paya'),
    F.sum(F.when(F.col("tipo_operacion") == 'PORTA PP-SS', F.col("q_ventas_netas")).otherwise(0)).alias('porta_pp_ss_paya'),
    F.sum(F.when(F.col("tipo_operacion") == 'PORTA SS-SS', F.col("q_ventas_netas")).otherwise(0)).alias('porta_ss_ss_paya'),
    F.sum(F.when(F.col("tipo_operacion") == 'MIGRACION', F.col("q_ventas_netas")).otherwise(0)).alias('migracion_paya'),
    F.sum("q_ventas_netas").alias("q_ventas_netas_paya"),
    F.sum("netas_a_la_fecha_recepcion").alias("netas_a_la_fecha_recepcion_paya"),
    F.sum("ingresos_reales_paya").alias("ingresos_reales_paya_dolar"),
]

df_paya = df_paya.repartition('fecha','d_identidad')
df_paya_resumen = df_paya.groupBy(agg_expr1).agg(*agg_expr2)

df_tot=df_tot.join(df_paya_resumen,['fecha','d_identidad'],'outer')


In [255]:

query1 = f"""
select 
distinct
[FECHA HABILITACION] AS fecha
,[DNI ASESOR] as d_identidad
,SUM(cast([PORTABILIDAD NETAS SIEBEL] as int)) over(partition by [FECHA HABILITACION]
,[DNI ASESOR]) as hab_int
from DB_venta.dbo.backoffice_hab_in
"""
df_back_hab_in = obtener_tabla_sql(spark,"DB_venta",query1)

df_tot=df_tot.join(df_back_hab_in,['fecha','d_identidad'],'left')


In [256]:
query1=f"""
select campana_siop_a,cast(costo_soles as real)costo_soles,tc from DB_venta.dbo.tabla_costo_TC_ingreso_costo

"""
df_tarifa_costo = obtener_tabla_sql(spark,'DB_tiempo',query1)

df_tot=df_tot.join(df_tarifa_costo,['campana_siop_a'],'left')
df_tot=df_tot.withColumn('costo_soles',col('dias_venta')*col('costo_soles'))


In [257]:
valores_reemplazo = {
    'hogar_fibra_paya': 0,
    '2da_linea_paya': 0,
    'renovacion_equipo_paya': 0,
    'nueva_linea_paya': 0,
    'porta_pp_ss_paya': 0,
    'porta_ss_ss_paya': 0,
    'migracion_paya': 0,
    'q_ventas_netas_paya': 0,
    'netas_a_la_fecha_recepcion_paya': 0,
    'ingresos_reales_paya_dolar': 0,
    'hab_int': 0,
    }
df_tot = df_tot.fillna(valores_reemplazo)

window_spec_fecha = Window.partitionBy("nombres_siop")

df_tot = df_tot.withColumn("max_fecha", F.max("fecha").over(window_spec_fecha))

In [259]:
overwrite_table_SQL(df_tot,'DB_a365','edu03_asistencia','')


Error al insertar JDBC: [COLUMN_NOT_DEFINED_IN_TABLE] "STRING" column `cargo_ult` is not defined in table `edu03_asistencia`, defined table columns are: `campana_siop_a`, `fecha`, `d_identidad`, `duracion`, `estado_ref`, `ref_asistencia`, `debe_laborar`, `laborados`, `faltas`, `inactivo`, `total`, `contratados_dia`, `activos_dia`, `conectados_dia`, `dias_venta`, `t_acu`, `p_acu`, `supervisor_siop`, `nombres_siop`, `campana_siop`, `entrada`, `salida`, `jornada`, `cargo`, `estado_siop`, `sub_estado`, `f_inicio`, `f_fin`, `grupo`, `subgerente`, `tipo_trabajo`, `usuario_siop`, `dni_operaciones`, `cargo_operaciones`, `contrato`, `estado`, `hogar_fibra`, `2da_linea`, `renovacion_equipo`, `nueva_linea`, `porta_pp_ss`, `porta_ss_ss`, `migracion`, `q_ventas`, `q_ventas_proyectadas`, `dentro`, `fuera`, `brutas_a_la_fecha`, `netas_a_la_fecha`, `ingresos_actuales`, `ingresos_proyectados`, `brutas_a_la_fecha_recepcion`, `netas_desde_fecha_recepcion`, `ingresos_actuales_desde_fecha_recepcion`, `ingr

In [260]:
df_tot = df_tot.withColumn('agente_unico_ult',when(col('fecha')==fecha_fin,1)
                                        .otherwise(0))

In [267]:
overwrite_table_SQL(df_tot,'DB_A365','edu03_asistencia','')


In [262]:
print(df_tot.columns)

['campana_siop_a', 'fecha', 'd_identidad', 'duracion', 'estado_ref', 'ref_asistencia', 'debe_laborar', 'laborados', 'faltas', 'inactivo', 'total', 'contratados_dia', 'activos_dia', 'conectados_dia', 'dias_venta', 't_acu', 'p_acu', 'supervisor_siop', 'nombres_siop', 'campana_siop', 'entrada', 'salida', 'jornada', 'cargo', 'estado_siop', 'sub_estado', 'f_inicio', 'f_fin', 'grupo', 'subgerente', 'tipo_trabajo', 'usuario_siop', 'dni_operaciones', 'cargo_operaciones', 'contrato', 'estado', 'hogar_fibra', '2da_linea', 'renovacion_equipo', 'nueva_linea', 'porta_pp_ss', 'porta_ss_ss', 'migracion', 'q_ventas', 'q_ventas_proyectadas', 'dentro', 'fuera', 'brutas_a_la_fecha', 'netas_a_la_fecha', 'ingresos_actuales', 'ingresos_proyectados', 'brutas_a_la_fecha_recepcion', 'netas_desde_fecha_recepcion', 'ingresos_actuales_desde_fecha_recepcion', 'ingresos_proyectados_desde_fecha_recepcion', 't_total01', 't_espera01', 't_pausa01', 't_operativo01', 'unico', 'hogar_fibra_paya', '2da_linea_paya', 'renova

In [264]:
agg_expr1 = [
     'fecha',col('campana_siop_a').alias('campana_siop'),'cargo','nombres_siop','supervisor_siop'
    ]

agg_expr2 = [
    F.sum("contratados_dia").alias("contratados"),
    F.sum("activos_dia").alias("activos"),
    F.sum("conectados_dia").alias("conectados"),
]   

df_tot = df_tot.repartition('fecha','campana_siop')
df_tot_resumen_01 = df_tot.groupBy(agg_expr1).agg(*agg_expr2)

In [265]:
overwrite_table_SQL(df_tot,'DB_A365','edu03_asistencia_unico','')


In [ ]:
window_spec_fecha = Window.partitionBy("nombres_siop")

df_tot = df_tot.withColumn("max_fecha", F.max("fecha").over(window_spec_fecha))
# df_tot= df_tot.filter(col("max_fecha") == f'{fecha_fin}')

In [ ]:
from pyspark.sql.functions import datediff

df_tot = df_tot.withColumn("dias_antiguedad", datediff(col("fecha"), col("f_inicio")))
df_tot = df_tot.withColumn('antiguedad',when(col('dias_antiguedad')<15,'OJT')
                                                            .when(col('dias_antiguedad')<30,'menor a 30 días')
                                                            .when(col('dias_antiguedad')<91,'1 a 3 meses')
                                                            .when(col('dias_antiguedad')<181,'3 a 6 meses')
                                                            .otherwise('mayor a 6 meses'))

In [ ]:




overwrite_table_SQL(df_tot_resumen_01,'DB_A365','edu03_asistencia_01_diaria','')


In [104]:
window_spec_fecha = Window.partitionBy("nombres_siop")

df_tot_1 = df_tot.withColumn("max_fecha", F.max("fecha").over(window_spec_fecha))
df_tot_1= df_tot_1.filter(col("max_fecha") == f'{fecha_fin}')

agg_expr1 = [
     col('campana_siop_a').alias('campana_siop'),'cargo','supervisor_siop','nombres_siop','f_inicio'
    ]

agg_expr2 = [
    F.first("max_fecha").alias("fecha"),
    F.sum("q_ventas").alias("ventas_brutas"),
    F.sum("q_ventas_paya").alias("habilitadas_interno"),
    F.sum("contratados_dia").alias("contratados"),
    F.sum("activos_dia").alias("activos"),
    F.sum("dias_venta").alias("dias_venta"),
    F.sum("debe_laborar").alias("debe_laborar"),
    F.sum("laborados").alias("laborados"),
    F.sum("t_total01").alias("hrs_acumuladas"),
]   


df_tot_1 = df_tot_1.repartition('campana_siop_a','cargo','supervisor_siop','nombres_siop')
df_tot_resumen_01_super = df_tot_1.groupBy(agg_expr1).agg(*agg_expr2)

from pyspark.sql.functions import datediff

df_tot_resumen_01_super = df_tot_resumen_01_super.withColumn("dias_antiguedad", datediff(col("fecha"), col("f_inicio")))
df_tot_resumen_01_super = df_tot_resumen_01_super.withColumn('antiguedad',when(col('dias_antiguedad')<15,'OJT')
                                                            .when(col('dias_antiguedad')<30,'menor a 30 días')
                                                            .when(col('dias_antiguedad')<91,'1 a 3 meses')
                                                            .when(col('dias_antiguedad')<181,'3 a 6 meses')
                                                            .otherwise('mayor a 6 meses'))

overwrite_table_SQL(df_tot_resumen_01_super,'DB_A365','edu03_asistencia_01_super','')


In [ ]:
import sys 
sys.path.append('C:/Users/A365/Documents/script_01/resumen')
from funciones import *

def edu01_tiempos(fecha_fin):
    from pyspark.sql import functions as F
    from pyspark.sql.functions import col, trunc
    spark = SparkSession.builder \
        .appName("SparkExample") \
        .master("local[*]") \
        .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-12.8.1.jre8.jar') \
        .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-12.8.1.jre8.jar') \
        .config('spark.executor.memory', '20g') \
        .config('spark.driver.memory', '20g') \
        .config('spark.sql.session.timeZone', 'UTC') \
        .config('spark.sql.legacy.timeParserPolicy', 'LEGACY') \
        .getOrCreate()

    query1 = f"""
        select 
        fecha
        ,nombre_skill
        ,calling_list
        ,supervisor_siop as supervisor
        ,usuario_genesys
        ,cast(logged_in as float) as t_total
        from DB_tiempo.dbo.reporte_informe_tiempo
        where calling_list in(
            'VAG_CL_OUT_ECO_A365_MIGRACIONES'
            ,'VAG_CL_OUT_ECO_A365_MIGRACIONES2'
            ,'VAG_CL_OUT_ECO_A365_PORTA_EXCLUSIVA'
            ,'VAG_CL_OUT_ECO_A365_PORTA_IVR'
            ,'VAG_CL_OUT_ECO_A365_PORTA_PERFILADA'
            ,'VAG_CL_OUT_ECO_A365_PORTA_RECUPERADOS'
            ,'VAG_CL_OUT_ECO_A365_MOVIL_PORTA5'
            ,'VAG_CL_OUT_ECO_A365_SEGUNDAS_LINEAS'
            ,'VAG_CL_OUT_EMP_A365_HOGAR_FIBRA_CLIENTE'
            ,'VAG_CL_OUT_EMP_A365_HOGAR_FIBRA_NO_CLIENTE'
            ,'VAG_CL_OUT_ECO_A365_HOGAR_FIBRA_CLIENTE'
            ,'VAG_CL_OUT_ECO_A365_HOGAR_FIBRA_NOCLIENTE'
            ,'VAG_CL_OUT_ECO_A365_HOGAR_FIBRA_CLIENTE2'
        )
        and fecha between DATEADD(MONTH, DATEDIFF(MONTH, 0, '{fecha_fin}'), 0) and '{fecha_fin}'
        and logged_in  is not null
    """
    df_agent_status = obtener_tabla_sql(spark,"DB_CCR",query1)

    window_spec_r = Window.partitionBy('nombre_skill','fecha','usuario_genesys').orderBy(F.col("fecha").asc())
    window_spec_dia_transcurrido = Window.partitionBy('nombre_skill','fecha').orderBy(F.col("fecha").asc())

    df_agent_status = df_agent_status.withColumns({
        "unico_r": F.row_number().over(window_spec_r),
        "dia_valido": F.row_number().over(window_spec_dia_transcurrido)
    })

    agg_expr1 = ['fecha','calling_list', 'nombre_skill', 'supervisor']

    agg_expr2 = [
        F.sum(F.when(F.col("dia_valido") == 1, 1).otherwise(0)).alias('dias_transcurrido'),
        F.sum(F.when(F.col("unico_r") == 1, 1).otherwise(0)).alias('q_persona'),
        F.sum("t_total").alias("t_total")
    ]
    df_agent_status = df_agent_status.repartition('fecha','calling_list', 'nombre_skill', 'supervisor')
    df_agent_status = df_agent_status.groupBy(agg_expr1).agg(*agg_expr2)
    df_agent_status=add_n_sem_carga(spark,df_agent_status,fecha_fin,'nombre_skill','fecha')

    df_agent_status = df_agent_status.withColumn("primer_dia_mes", trunc(col("fecha"), "month")) \
        .withColumn("dia_semana_inicio", date_format(col("primer_dia_mes"), "u").cast("int")) \
        .withColumn("dia_mes", dayofmonth(col("fecha"))) \
        .withColumn("n_sem", expr("int((dia_mes + dia_semana_inicio - 2) / 7) + 1"))

    df_agent_status=df_agent_status.withColumn('n_sem_carga',when(col('n_sem_carga').isNull(),col('n_sem')).otherwise(col('n_sem_carga')))

    df_agent_status = df_agent_status.withColumn('criterio', lit('tiempo'))

    df_agent_status=df_agent_status.select(
    'criterio','fecha', 'nombre_skill', 'calling_list', 'supervisor', 'dias_transcurrido', 'q_persona', 't_total', 'n_sem_carga', 'n_sem'
    )

    from pyspark.sql.functions import dayofweek

    df_agent_status = df_agent_status.filter(dayofweek("fecha") != 1)

    query1=f"""
    select * from DB_tiempo.dbo.fecha_no_aplica
    """
    df_fecha_no_aplica=obtener_tabla_sql(spark,'DB_tiempo',query1)

    df_agent_status=df_agent_status.join(df_fecha_no_aplica,['fecha','nombre_skill'],'leftanti')

    df_agent_status = df_agent_status.withColumn(
        'campana01',
        when(col('calling_list') == 'VAG_CL_OUT_ECO_A365_MOVIL_PORTA5', col('calling_list'))
        .when(col('nombre_skill').isin('a365_hogar_fibra_cliente'),'Hogar 3.0')
        .when(col('calling_list').contains('EMP'), regexp_replace(col('calling_list'), 'VAG_CL_OUT_', ''))
        .when(col('calling_list').contains('OUT_ECO_A365_'), regexp_replace(col('calling_list'), 'VAG_CL_OUT_ECO_', ''))
        .otherwise('otros')
    )

    df_agent_status=df_agent_status.withColumn(
        'campana02',
        when(col('nombre_skill').contains('emp_'),'Empresas')
        .when(col('nombre_skill').isin('PORTABILIDAD_PERFILADA_1','MOVIL_PORTA5'),'Perfilada 1')
        .when(col('nombre_skill').isin('a365_hogar_fibra_cliente'),'Hogar 3.0')
        .when(col('nombre_skill')=='PORTABILIDAD_PREPAGO_IVR','IVR')
        .when(col('nombre_skill')=='PORTABILIDAD_RECUPERADOS','Recuperados')
        .when(col('nombre_skill')=='SEGUNDAS_LINEAS','Segundas lineas')
        .when(col('nombre_skill')=='MIGRACIONES_1','Migraciones 1')
        .when(col('nombre_skill')=='MIGRACIONES_2','Migraciones 2')
        .otherwise('otros'))

    df_agent_status=df_agent_status.withColumn(
        'campana03',
        when(col('nombre_skill').contains('emp_'),'Empresas')
        .when(col('nombre_skill').isin('PORTABILIDAD_PERFILADA_1','MOVIL_PORTA5'),'Perfilada')
        .when(col('nombre_skill').isin('a365_hogar_fibra_cliente'),'Hogar 3.0')
        .when(col('nombre_skill')=='PORTABILIDAD_PREPAGO_IVR','IVR')
        .when(col('nombre_skill')=='PORTABILIDAD_RECUPERADOS','Recuperados')
        .when(col('nombre_skill')=='SEGUNDAS_LINEAS','Segundas Lineas')
        .when(col('nombre_skill').isin('MIGRACIONES_1','MIGRACIONES_2'),'Migraciones')
        .otherwise('otros'))

    df_agent_status=df_agent_status.withColumn(
        'campana_siop',
        when(col('nombre_skill').contains('emp_'),'Entel - Portabilidad Empresas')
        .when(col('nombre_skill').isin('PORTABILIDAD_PERFILADA_1','MOVIL_PORTA5'),'Entel - Portabilidad Perfilada')
        .when(col('nombre_skill').isin('a365_hogar_fibra_cliente'),'Hogar 3.0')
        .when(col('nombre_skill')=='PORTABILIDAD_PREPAGO_IVR','Entel - Portabilidad Prepago Ivr')
        .when(col('nombre_skill')=='PORTABILIDAD_RECUPERADOS','Entel - Portabilidad Recuperados')
        .when(col('nombre_skill')=='SEGUNDAS_LINEAS','Entel - Segundas LIneas')
        .when(col('nombre_skill').isin('MIGRACIONES_1','MIGRACIONES_2'),'Entel - Migraciones')
        .otherwise('otros'))


    window_spec = Window.partitionBy('fecha','campana02').orderBy(F.col("fecha").asc())
    df_agent_status = df_agent_status.withColumn("dias_transcurridos_campana2", row_number().over(window_spec))

    df_agent_status=df_agent_status.withColumn(
        'dias_transcurridos_campana2',
        when(col('dias_transcurridos_campana2')==1,1)
        .otherwise(0))

    window_spec = Window.partitionBy('fecha','campana03').orderBy(F.col("fecha").asc())
    df_agent_status = df_agent_status.withColumn("dias_transcurridos_campana3", row_number().over(window_spec))

    df_agent_status=df_agent_status.withColumn(
        'dias_transcurridos_campana3',
        when(col('dias_transcurridos_campana3')==1,1)
        .otherwise(0))

    window_spec = Window.partitionBy('fecha','campana_siop').orderBy(F.col("fecha").asc())
    df_agent_status = df_agent_status.withColumn("dias_transcurridos_campana_siop", row_number().over(window_spec))

    df_agent_status=df_agent_status.withColumn(
        'dias_transcurridos_campana_siop',
        when(col('dias_transcurridos_campana_siop')==1,1)
        .otherwise(0))

    overwrite_table_SQL(df_agent_status,"DB_a365",'edu01_tiempos','tiempo')

   
    
    
    
    


In [ ]:
# def edu02_ventas(fecha_fin):
from pyspark.sql import functions as F
from pyspark.sql.functions import col, trunc
spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-12.8.1.jre8.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-12.8.1.jre8.jar') \
    .config('spark.executor.memory', '20g') \
    .config('spark.driver.memory', '20g') \
    .config('spark.sql.session.timeZone', 'UTC') \
    .config('spark.sql.legacy.timeParserPolicy', 'LEGACY') \
    .getOrCreate()

query1=f"""
select
ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS indice_001,
* from DB_venta.dbo.resumen_fb_01
where fecha_registro between DATEADD(MONTH, DATEDIFF(MONTH, 0, '{fecha_fin}'), 0) and '{fecha_fin}'
"""
df_wdv=obtener_tabla_sql(spark,'DB_tiempo',query1)

query1=f"""
select tipo_operacion, porc_conversion from DB_venta.dbo.porc_conversion_producto
where fecha='2025-07-01'
"""
df_conversion = obtener_tabla_sql(spark,'DB_tiempo',query1)

query1=f"""
select tipo_operacion, porc_conversion AS porc_conversion_edu from DB_venta.dbo.porc_conversion_producto_edu
"""
df_conversion_edu = obtener_tabla_sql(spark,'DB_tiempo',query1)

query1=f"""
select campana_reporte_gc as campana_siop_a, tipo_operacion, tarifa from DB_venta.dbo.tarifa_producto 
"""
df_tarifa = obtener_tabla_sql(spark,'DB_tiempo',query1)

df_wdv = df_wdv.withColumn('campana_siop_a',when(col('campania_usuario_reg_linea').contains('MIGRA'),'MIGRACIONES')
                        .when(col('campania_usuario_reg_linea')=='PERFILADA 1','PERFILADA')
                        .when(col('campania_usuario_reg_linea')=='PORTA5','PERFILADA')
                        .when(col('campania_usuario_reg_linea')=='PREPAGO IVR' ,'IVR')
                        .when(col('campania_usuario_reg_linea')=='EXCLUSIVA RECUPERADOS','RECUPERADOS')
                        .when(col('campania_usuario_reg_linea')=='SEGUNDAS LINEAS 1','SEGUNDAS LINEAS')
                        .otherwise('NO_APLICA'))

df_wdv=df_wdv.join(df_conversion,['tipo_operacion'],'left')
df_wdv=df_wdv.join(df_conversion_edu,['tipo_operacion'],'left')
df_wdv=df_wdv.join(df_tarifa,['campana_reporte_gc', 'tipo_operacion'],'left')

df_wdv = df_wdv.withColumn("q_ventas", col("q_ventas").cast(IntegerType())) \
            .withColumn("porc_conversion", col("porc_conversion").cast(DoubleType())) \
            .withColumn("porc_conversion_edu", col("porc_conversion").cast(DoubleType())) \
            .withColumn("tarifa", col("tarifa").cast(IntegerType()))

df_wdv = df_wdv.withColumn(
    "por_conversion_dia_semana",
    F.when(F.date_format("fecha_registro", "E") == "Sat", F.lit(0.5))  # sábado
    .when(F.date_format("fecha_registro", "E") == "Sun", F.lit(0))    # domingo
    .otherwise(F.lit(1))                                     # demás días
)

df_wdv = df_wdv.withColumn("valor_proyectado",col("q_ventas") * col("porc_conversion") * col("tarifa") * col("por_conversion_dia_semana"))

window_spec = Window.partitionBy("indice_001")
df_wdv = df_wdv.withColumn(
    "valor_proyectado_dolar",
    sum("valor_proyectado").over(window_spec)
)

# overwrite_table_SQL(df_wdv,"DB_a365",'edu02_wdv_PRUEBA','venta_bruta')

window_spec = Window.partitionBy('indice_001').orderBy(col("indice_001").asc())
df_wdv = df_wdv.withColumn("unico", row_number().over(window_spec))
df_wdv = df_wdv.filter(col("unico")==1)

df_wdv=df_wdv.select(
    'campana_reporte_gc', 'tipo_operacion', 'indice_001', 'fecha_registro', 'hora', 'n_sem_carga', 'n_sem', 'tipo_reparto', 'nombre_skill', 'campania_usuario_reg_linea', 'campana', 'nombre_skill_1', 'nombre_lista', 'fecha_carga', 'nom_agente', 'nombre_usuario_reg_linea', 'num_doc_usuario_reg_linea', 'supervisor_web_ventas', 'supervisor', 'prioridad_1', 'prioridad_2', 'usuario_siop', 'comuna', 'region', 'compania', 'bloque', 'tipo_producto', 'q_ventas', 'q_mes_actual', 'q_ult_3_meses', 'q_ult_12_meses', 'valor_proyectado_dolar'
)

overwrite_table_SQL(df_wdv,"DB_a365",'edu02_wdv','venta_bruta')
# Export_list_base_sql(df_agent_status,"DB_a365",'edu01_wdv','venta_bruta')


In [46]:
fecha_fin='2025-08-26'



# edu01_tiempos(fecha_fin)
# edu02_ventas(fecha_fin)

In [4]:
overwrite_table_SQL(df_agent_status,"DB_a365",'edu01_tiempos','tiempo')


NameError: name 'df_agent_status' is not defined

In [15]:
import sys 
sys.path.append('C:/Users/A365/Documents/script_01/resumen')
from funciones import *

In [4]:
campaigns[14]

('CMP_CL_OUT_ECO_A365_MIGRACIONES',
 'TB_outbound_ccrmovil',
 'DB_MIGRACIONES',
 'MIGRACIONES_2')

In [ ]:
fecha_a='2025-08-01'


### Para productividad

In [ ]:
    fecha_fin='2025-08-19'


In [21]:

query1 = f"""
select * from DB_tiempo.dbo.reporte_tiempo_01
WHERE calling_list in(
'VAG_CL_OUT_ECO_A365_MIGRACIONES'
,'VAG_CL_OUT_ECO_A365_MIGRACIONES2'
,'VAG_CL_OUT_ECO_A365_PORTA_EXCLUSIVA'
,'VAG_CL_OUT_ECO_A365_PORTA_IVR'
,'VAG_CL_OUT_ECO_A365_PORTA_PERFILADA'
,'VAG_CL_OUT_ECO_A365_PORTA_RECUPERADOS'
,'VAG_CL_OUT_ECO_A365_MOVIL_PORTA5'
,'VAG_CL_OUT_ECO_A365_SEGUNDAS_LINEAS'
)
"""
df_info_t = obtener_tabla_sql(spark,"DB_CCR",query1)
print(df_info_t.columns)

['fecha', 'calling_list', 'usuario_genesys', 'agent_name', 'logged_in', 'idle', 'off_queue', 'total_talk', 'total_acw', 'interacting', 'on_queue', 'not_responding', 'available', 'busy', 'away', 'break', 'meal', 'system_away', 'meeting', 'training', 'communicating', 'total_acd', 'log_in', 'log_out', 'origen', 'ref1', 'dni_siop', 'cargo_actual', 'nombre_completo', 'campana_siop', 'supervisor_siop', 'from_1', 'antiguedad', 'nombre_skill', 'gestion', 'n_sem', 'tipo_dia']


In [ ]:
['fecha', 'calling_list', 'usuario_genesys', 'agent_name', 'logged_in', 'idle', 'off_queue', 'total_talk', 'total_acw', 'interacting', 'on_queue', 'not_responding', 'available', 'busy', 'away', 'break', 'meal', 'system_away', 'meeting', 'training', 'communicating', 'total_acd', 'log_in', 'log_out', 'origen', 'ref1', 'dni_siop', 'cargo_actual', 'nombre_completo', 'campana_siop', 'supervisor_siop', 'from_1', 'antiguedad', 'nombre_skill', 'gestion', 'n_sem', 'tipo_dia']

In [ ]:

query1=f"""
select 
fecha
,case
when nombre_skill ='PORTABILIDAD_RECUPERADOS' then 'sk_CL_OUT_ECO_A365_PORTA_RECUPERADOS'
when nombre_skill ='PORTABILIDAD_PERFILADA_1' then 'sk_CL_OUT_ECO_A365_PORTA_PERFILADA'
when nombre_skill ='PORTABILIDAD_PREPAGO_IVR' then 'SK_CL_OUT_ECO_A365_PORTA_IVR'
end as Skills
from DB_tiempo.dbo.fecha_no_aplica
where nombre_skill in(
'PORTABILIDAD_RECUPERADOS'
,'PORTABILIDAD_PERFILADA_1'
,'PORTABILIDAD_PREPAGO_IVR'
)
"""
df_fecha_no_aplica=obtener_tabla_sql(spark,'DB_tiempo',query1)

query1=f"""
select
fecha_registro as fecha
,num_doc_usuario_reg_linea as dni
,q_ventas
from DB_venta.dbo.resumen_fb_01
where fecha_registro between DATEADD(MONTH, DATEDIFF(MONTH, 0, '{fecha_fin}'), 0) and '{fecha_fin}'
and campania_usuario_reg_linea in(
'PERFILADA 2'
,'EXCLUSIVA RECUPERADOS'
,'PORTA5'
,'PREPAGO IVR'
,'PERFILADA 1'
)
"""
df_fb1=obtener_tabla_sql(spark,'DB_tiempo',query1)

query1=f"""
select 
DNI as dni
,NOMBRES as Agente
,case
when CAMPAÑA ='ENTEL - PORTABILIDAD EXCLUSIVA' then 'ENTEL - PORTABILIDAD PERFILADA'
when CAMPAÑA ='ENTEL - PORTABILIDAD NO PERFILADA' then 'ENTEL - PORTABILIDAD PERFILADA'
when CAMPAÑA ='ENTEL - PORTABILIDAD PERFILADA 2' then 'ENTEL - PORTABILIDAD PERFILADA'
else CAMPAÑA 
end as Campaña
,SUPERVISOR as Supervisor 
,usuario_temp as usuario_siop
from DB_dota.dbo.tb_dota_diaria_siop
where fecha='{fecha_fin}'
and CAMPAÑA in(
'ENTEL - PORTABILIDAD EXCLUSIVA'
,'ENTEL - PORTABILIDAD NO PERFILADA'
,'ENTEL - PORTABILIDAD PERFILADA 2'
,'ENTEL - PORTABILIDAD PERFILADA'
,'ENTEL - PORTABILIDAD RECUPERADOS'
,'ENTEL - PORTABILIDAD PREPAGO IVR'
)
and ESTADO='ACTIVO'

"""
df_dota=obtener_tabla_sql(spark,'DB_tiempo',query1)

agg_expr1 = [
    'fecha', 'dni'
]

agg_expr2 = [
    F.sum("q_ventas").alias("q_ventas")
]
df_resumen_fb1 = df_fb1.groupBy(agg_expr1).agg(*agg_expr2)
df_agent_status = df_agent_status.join(df_fecha_no_aplica,['fecha', 'Skills'],'leftanti')

update_t_total(df_agent_status,df_dota,fecha_fin)
add_operaciones(df_agent_status,df_dota,fecha_fin)
add_venta(df_resumen_fb1,df_dota,fecha_fin)


In [ ]:
# add_cabanillas_01(fecha_fin)
# add_cabanillas_02(fecha_fin)

In [104]:
fecha_fin='2025-09-04'

In [ ]:
def update_t_total(df,df_dota,fecha_fin): 
    df = df.withColumn("nombre_fecha", F.date_format("fecha", "dd_MMM").alias("nombre_fecha"))

    fecha_dt = datetime.strptime(fecha_fin, "%Y-%m-%d").date()

    anio = fecha_dt.year
    mes = fecha_dt.month

    start_date = date(anio, mes, 1)

    ultimo_dia = calendar.monthrange(anio, mes)[1]  
    end_date = date(anio, mes, ultimo_dia)

    dias_mes = [
        (start_date + timedelta(days=i)).strftime("%d_%b").lower()
        for i in range((end_date - start_date).days + 1)
    ]

    df = df.withColumn("nombre_fecha", F.date_format("fecha", "dd_MMM").cast("string").alias("nombre_fecha"))
    df = df.withColumn("nombre_fecha", F.lower(F.col("nombre_fecha")))

    df_pivot = (
        df.groupBy("usuario_siop")
        .pivot("nombre_fecha", dias_mes)
        .agg(F.first("t_total"))
    )
    df_pivot = df_dota.join(df_pivot,['usuario_siop'],'inner')

    overwrite_table_SQL(df_pivot,"DB_a365",'reporte_diario_t_total_setiembre','t_total')

def add_operaciones(df,df_dota,fecha_fin):
    df = df.withColumn("nombre_fecha", F.date_format("fecha", "dd_MMM").alias("nombre_fecha"))

    fecha_dt = datetime.strptime(fecha_fin, "%Y-%m-%d").date()

    anio = fecha_dt.year
    mes = fecha_dt.month

    start_date = date(anio, mes, 1)

    ultimo_dia = calendar.monthrange(anio, mes)[1]  
    end_date = date(anio, mes, ultimo_dia)

    dias_mes = [
        (start_date + timedelta(days=i)).strftime("%d_%b").lower()
        for i in range((end_date - start_date).days + 1)
    ]

    df = df.withColumn("nombre_fecha", F.date_format("fecha", "dd_MMM").cast("string").alias("nombre_fecha"))
    df = df.withColumn("nombre_fecha", F.lower(F.col("nombre_fecha")))

    df_pivot = (
        df.groupBy("usuario_siop")
        .pivot("nombre_fecha", dias_mes)
        .agg(F.first("t_operativo"))
    )
    df_pivot = df_dota.join(df_pivot,['usuario_siop'],'inner')


    overwrite_table_SQL(df_pivot,"DB_a365",'reporte_diario_t_operativo_setiembre','t_operativo')

def add_venta(df,df_dota,fecha_fin):
    df = df.withColumn("nombre_fecha", F.date_format("fecha", "dd_MMM").alias("nombre_fecha"))

    fecha_dt = datetime.strptime(fecha_fin, "%Y-%m-%d").date()

    anio = fecha_dt.year
    mes = fecha_dt.month

    start_date = date(anio, mes, 1)

    ultimo_dia = calendar.monthrange(anio, mes)[1]  
    end_date = date(anio, mes, ultimo_dia)

    dias_mes = [
        (start_date + timedelta(days=i)).strftime("%d_%b").lower()
        for i in range((end_date - start_date).days + 1)
    ]

    df = df.withColumn("nombre_fecha", F.date_format("fecha", "dd_MMM").cast("string").alias("nombre_fecha"))
    df = df.withColumn("nombre_fecha", F.lower(F.col("nombre_fecha")))

    df_pivot = (
        df.groupBy("dni")
        .pivot("nombre_fecha", dias_mes)
        .agg(F.first("q_ventas"))
    )
    df_pivot = df_dota.join(df_pivot,['dni'],'inner')


    overwrite_table_SQL(df_pivot,"DB_a365",'reporte_diario_ventas_setiembre','q_ventas')

def add_cabanillas_01(fecha_fin):
    from pyspark.sql import functions as F
    from pyspark.sql.functions import col, trunc
    from pyspark.sql import functions as F
    from pyspark.sql.types import DateType
    from datetime import datetime, date, timedelta
    import calendar
    
    spark = SparkSession.builder \
        .appName("SparkExample") \
        .master("local[*]") \
        .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-12.8.1.jre8.jar') \
        .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-12.8.1.jre8.jar') \
        .config('spark.executor.memory', '20g') \
        .config('spark.driver.memory', '20g') \
        .config('spark.sql.session.timeZone', 'UTC') \
        .config('spark.sql.legacy.timeParserPolicy', 'LEGACY') \
        .getOrCreate()

    query1 = f"""
    select 
    [Interval Start] as fecha
    ,user_genesys as usuario_siop
    ,cast([Logged In]as float)/3600 AS t_total
    ,(cast([Logged In]as float)-cast( Idle as float)- cast([Off Queue]as float))/3600 AS t_operativo
    ,Skills
    from DB_Tiempo.dbo.tb_agent_status
    where [Interval Start]>= DATEADD(MONTH, DATEDIFF(MONTH, 0, '{fecha_fin}'), 0) 
    and Skills in(
    'sk_CL_OUT_ECO_A365_PORTA_PERFILADA'
    ,'sk_CL_OUT_ECO_A365_PORTA_RECUPERADOS'
    ,'SK_CL_OUT_ECO_A365_PORTA_IVR'
    ,'SK_CL_OUT_ECO_A365_MOVIL_PORTA5'
    )
    """
    df_agent_status = obtener_tabla_sql(spark,"DB_CCR",query1)

    query1=f"""
    select 
    fecha
    ,case
    when nombre_skill ='PORTABILIDAD_RECUPERADOS' then 'sk_CL_OUT_ECO_A365_PORTA_RECUPERADOS'
    when nombre_skill ='PORTABILIDAD_PERFILADA_1' then 'sk_CL_OUT_ECO_A365_PORTA_PERFILADA'
    when nombre_skill ='PORTABILIDAD_PREPAGO_IVR' then 'SK_CL_OUT_ECO_A365_PORTA_IVR'
    end as Skills
    from DB_tiempo.dbo.fecha_no_aplica
    where nombre_skill in(
    'PORTABILIDAD_RECUPERADOS'
    ,'PORTABILIDAD_PERFILADA_1'
    ,'PORTABILIDAD_PREPAGO_IVR'
    )
    """
    df_fecha_no_aplica=obtener_tabla_sql(spark,'DB_tiempo',query1)

    query1=f"""
    select
    fecha_registro as fecha
    ,num_doc_usuario_reg_linea as dni
    ,q_ventas
    from DB_venta.dbo.resumen_fb_01
    where fecha_registro between DATEADD(MONTH, DATEDIFF(MONTH, 0, '{fecha_fin}'), 0) and '{fecha_fin}'
    and campania_usuario_reg_linea in(
    'PERFILADA 2'
    ,'EXCLUSIVA RECUPERADOS'
    ,'PORTA5'
    ,'PREPAGO IVR'
    ,'PERFILADA 1'
    )
    """
    df_fb1=obtener_tabla_sql(spark,'DB_tiempo',query1)

    query1=f"""
    select 
    DNI as dni
    ,NOMBRES as nombre_siop
    ,case
    when CAMPAÑA ='ENTEL - PORTABILIDAD EXCLUSIVA' then 'ENTEL - PORTABILIDAD PERFILADA'
    when CAMPAÑA ='ENTEL - PORTABILIDAD NO PERFILADA' then 'ENTEL - PORTABILIDAD PERFILADA'
    when CAMPAÑA ='ENTEL - PORTABILIDAD PERFILADA 2' then 'ENTEL - PORTABILIDAD PERFILADA'
    else CAMPAÑA 
    end as Campaña
    ,SUPERVISOR as Supervisor 
    ,usuario_temp as usuario_siop
    from DB_dota.dbo.tb_dota_diaria_siop
    where fecha='{fecha_fin}'
    and CAMPAÑA in(
    'ENTEL - PORTABILIDAD EXCLUSIVA'
    ,'ENTEL - PORTABILIDAD NO PERFILADA'
    ,'ENTEL - PORTABILIDAD PERFILADA 2'
    ,'ENTEL - PORTABILIDAD PERFILADA'
    ,'ENTEL - PORTABILIDAD RECUPERADOS'
    ,'ENTEL - PORTABILIDAD PREPAGO IVR'
    ,'ENTEL - SEGUNDAS LINEAS'
    ,'ENTEL - MIGRACIONES'
    )
    and ESTADO='ACTIVO'

    """
    df_dota=obtener_tabla_sql(spark,'DB_tiempo',query1)

    agg_expr1 = [
        'fecha', 'dni'
    ]

    agg_expr2 = [
        F.sum("q_ventas").alias("q_ventas")
    ]
    df_resumen_fb1 = df_fb1.groupBy(agg_expr1).agg(*agg_expr2)
    df_agent_status = df_agent_status.join(df_fecha_no_aplica,['fecha', 'Skills'],'leftanti')

    update_t_total(df_agent_status,df_dota,fecha_fin)
    add_operaciones(df_agent_status,df_dota,fecha_fin)
    add_venta(df_resumen_fb1,df_dota,fecha_fin)

def add_cabanillas_02(fecha_fin):
    from pyspark.sql import functions as F
    from pyspark.sql.functions import col, trunc
    from pyspark.sql import functions as F
    from pyspark.sql.types import DateType
    from datetime import datetime, date, timedelta
    import calendar

    spark = SparkSession.builder \
        .appName("SparkExample") \
        .master("local[*]") \
        .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-12.8.1.jre8.jar') \
        .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-12.8.1.jre8.jar') \
        .config('spark.executor.memory', '20g') \
        .config('spark.driver.memory', '20g') \
        .config('spark.sql.session.timeZone', 'UTC') \
        .config('spark.sql.legacy.timeParserPolicy', 'LEGACY') \
        .getOrCreate()

    query1 = f"""
    select 
    [Interval Start] as fecha
    ,user_genesys as usuario_siop
    ,cast([Logged In]as float)/3600 AS t_total
    ,(cast([Logged In]as float)-cast( Idle as float)- cast([Off Queue]as float))/3600 AS t_operativo
    ,Skills
    from DB_Tiempo.dbo.tb_agent_status
    where [Interval Start]  between DATEADD(MONTH, DATEDIFF(MONTH, 0, '{fecha_fin}'), 0) and '{fecha_fin}'
    and Skills in(
    'sk_CL_OUT_ECO_A365_PORTA_EXCLUSIVA'
    ,'sk_CL_OUT_ECO_A365_PORTA_IVR'
    ,'sk_CL_OUT_ECO_A365_PORTA_PERFILADA'
    ,'sk_CL_OUT_ECO_A365_PORTA_RECUPERADOS'
    ,'sk_CL_OUT_ECO_A365_MOVIL_PORTA5'
    )
    """
    df_agent_status = obtener_tabla_sql(spark,"DB_CCR",query1)

    query1=f"""
    select 
    fecha
    ,case
    when nombre_skill ='PORTABILIDAD_RECUPERADOS' then 'sk_CL_OUT_ECO_A365_PORTA_RECUPERADOS'
    when nombre_skill ='PORTABILIDAD_PERFILADA_1' then 'sk_CL_OUT_ECO_A365_PORTA_PERFILADA'
    when nombre_skill ='MIGRACIONES_1' then 'sk_CL_OUT_ECO_A365_MIGRACIONES'
    when nombre_skill ='MIGRACIONES_2' then 'sk_CL_OUT_ECO_A365_MIGRACIONES2'
    when nombre_skill ='MOVIL_PORTA5' then 'sk_CL_OUT_ECO_A365_MOVIL_PORTA5'
    when nombre_skill ='PORTABILIDAD_PREPAGO_IVR' then 'sk_CL_OUT_ECO_A365_PORTA_IVR'
    when nombre_skill ='SEGUNDAS_LINEAS' then 'sk_CL_OUT_ECO_A365_SEGUNDAS_LINEAS'
    end as Skills
    from DB_tiempo.dbo.fecha_no_aplica
    where nombre_skill in(
    'PORTABILIDAD_RECUPERADOS'
    ,'PORTABILIDAD_PERFILADA_1'
    ,'MOVIL_PORTA5'
    ,'PORTABILIDAD_PERFILADA_2'
    ,'PORTABILIDAD_PREPAGO_IVR'
    ,'PORTABILIDAD_RECUPERADOS'
    )
    """
    df_fecha_no_aplica=obtener_tabla_sql(spark,'DB_tiempo',query1)

    query1=f"""
    select
    fecha_registro as fecha
    ,usuario_siop
    ,q_ventas
    from DB_venta.dbo.resumen_fb_01
    where fecha_registro between DATEADD(MONTH, DATEDIFF(MONTH, 0, '{fecha_fin}'), 0) and '{fecha_fin}'
    """
    df_fb1=obtener_tabla_sql(spark,'DB_tiempo',query1)

    query1=f"""
    select 
    DNI as dni
    ,NOMBRES as Agente
    ,case
    when CAMPAÑA ='ENTEL - PORTABILIDAD EXCLUSIVA' then 'ENTEL - PORTABILIDAD PERFILADA'
    when CAMPAÑA ='ENTEL - PORTABILIDAD NO PERFILADA' then 'ENTEL - PORTABILIDAD PERFILADA'
    when CAMPAÑA ='ENTEL - PORTABILIDAD PERFILADA 2' then 'ENTEL - PORTABILIDAD PERFILADA'
    else CAMPAÑA 
    end as Campaña
    ,SUPERVISOR as Supervisor 
    ,usuario_temp as usuario_siop
    from DB_dota.dbo.tb_dota_diaria_siop
    where fecha='{fecha_fin}'
    and CAMPAÑA in(
    'ENTEL - PORTABILIDAD EXCLUSIVA'
    ,'ENTEL - PORTABILIDAD NO PERFILADA'
    ,'ENTEL - PORTABILIDAD PERFILADA 2'
    ,'ENTEL - PORTABILIDAD PERFILADA'
    ,'ENTEL - PORTABILIDAD RECUPERADOS'
    ,'ENTEL - PORTABILIDAD PREPAGO IVR'
    )
    and ESTADO='ACTIVO'

    """
    df_dota=obtener_tabla_sql(spark,'DB_tiempo',query1)

    agg_expr1 = [
        'fecha', 'usuario_siop'
    ]

    agg_expr2 = [
        F.sum("q_ventas").alias("q_ventas")
    ]
    df_resumen_fb1 = df_fb1.groupBy(agg_expr1).agg(*agg_expr2)
    df_agent_status = df_agent_status.join(df_fecha_no_aplica,['fecha', 'Skills'],'leftanti')

    df_01=df_resumen_fb1.join(df_agent_status,['fecha', 'usuario_siop'],'outer')
    df_dota=df_dota.join(df_01,['usuario_siop'],'left')

    agg_expr1 = [
        'dni','Supervisor', 'Agente'
    ]

    agg_expr2 = [
        F.sum("q_ventas").alias("q_ventas"),
        F.sum("t_total").alias("t_total"),
        F.sum("t_operativo").alias("t_operativo")
    ]
    df_resumen = df_dota.groupBy(agg_expr1).agg(*agg_expr2)

    df_resumen = df_resumen.filter(
        ~(
            (col('t_total').isNull()) &
            (col('t_operativo').isNull()) &
            (col('q_ventas').isNull())
        )
    )
    overwrite_table_SQL(df_resumen,'DB_a365','reporte_diario_agentes_entel_ef_setiembre','')

add_cabanillas_01(fecha_fin)
add_cabanillas_02(fecha_fin)

t_total
t_operativo
q_ventas



In [3]:
import sys 
sys.path.append('C:/Users/A365/Documents/script_01/resumen')
from funciones import *


In [ ]:

reporte02_cabanillas(fecha_fin,1,'1990-01-01')


In [268]:
excel.Visible = True  


In [ ]:
import win32com.client as win32

excel = win32.gencache.EnsureDispatch('Excel.Application')
excel.Visible = False  
excel.DisplayAlerts = False  
excel.ScreenUpdating = False
excel.EnableEvents = False  
wb = excel.Workbooks.Open(archivo_excel)

wb = excel.Workbooks.Open(archivo_excel)

hojas_a_borrar = [
    "01_TIEMPO_CONEXION",df_pivot_t_total ,pegar en la fila 4 con cabeceras
    "01_TIEMPO_HABLADO",df_pivot_t_operativo,pegar en la fila 4 con cabeceras
    "01_VENTAS_BRUTAS",df_pivot_q_ventas,pegar en la fila 4 con cabeceras
]


for hoja_nombre in hojas_a_borrar:
    try:
        ws = wb.Sheets(hoja_nombre)
        ws.Range("4:100000").ClearContents() 
    except:
        pass


hojas_a_borrar = [
    "02_TABLA_AGENTES_ENTEL_EF"df_tabla_agente_entel_ef,pegar en la fila 4 con cabeceras
]


for hoja_nombre in hojas_a_borrar:
    try:
        ws = wb.Sheets(hoja_nombre)
        ws.Range("5:100000").ClearContents() 
    except:
        pass

# Guardar y cerrar
wb.Save()
wb.Close()
excel.Visible = True  
excel.DisplayAlerts = True  
excel.ScreenUpdating = True
excel.EnableEvents = True  

excel.Quit()


In [243]:
import win32com.client as win32

# Abrir Excel en modo "silencioso"
excel = win32.gencache.EnsureDispatch('Excel.Application')
# excel.Visible = False  
# excel.DisplayAlerts = False  
# excel.ScreenUpdating = False  
# excel.EnableEvents = False  

# Abrir el archivo
wb = excel.Workbooks.Open(archivo_excel)

# --- DataFrames que vas a pegar ---
df_map = {
    "01_TIEMPO_CONEXION": (df_pivot_t_total, 4),
    "01_TIEMPO_HABLADO": (df_pivot_t_operativo, 4),
    "01_VENTAS_BRUTAS": (df_pivot_q_ventas, 4),
    "02_TABLA_AGENTES_ENTEL_EF": (df_tabla_agente_entel_ef, 4)
}


In [244]:
ws = wb.Sheets('01_TIEMPO_CONEXION')


In [ ]:

# --- Procesar cada hoja ---
for hoja_nombre, (df, fila_inicio) in df_map.items():
    try:
        ws = wb.Sheets(hoja_nombre)

        # Limpiar valores previos (mantiene formatos)
        ws.Range(f"{fila_inicio}:{fila_inicio+100000}").ClearContents()
    
        # Convertir DataFrame en listas
        headers = df.columns.tolist()
        data = df.values.tolist()

        # Pegar cabeceras
        ws.Cells(fila_inicio, 1).Resize(1, len(headers)).Value = headers

        # Pegar datos (debajo de las cabeceras)
        if data:
            print(data)
            ws.Cells(fila_inicio + 1, 1).Resize(len(data), len(headers)).Value = data

    except Exception as e:
        print(f"No se pudo procesar hoja {hoja_nombre}: {e}")


# Guardar y cerrar
wb.Save()
wb.Close()


excel.Quit()
excel.Visible = True  
excel.DisplayAlerts = True  
excel.ScreenUpdating = True  
excel.EnableEvents = True  


In [218]:
excel.Visible = True  
excel.DisplayAlerts = True  
excel.ScreenUpdating = True  
excel.EnableEvents = True  


In [1]:
import sys 
sys.path.append('C:/Users/A365/Documents/script_01/resumen')
from funciones import *